In [ ]:
import os, pathlib as _pl

# Navigate to root-obj-perf/ regardless of where the Jupyter kernel starts
_cwd = _pl.Path.cwd()
if _cwd.name == "notebooks" and _cwd.parent.name == "root-obj-perf":
    os.chdir("..")
elif _cwd.name != "root-obj-perf" and (_cwd / "root-obj-perf").exists():
    os.chdir("root-obj-perf")
print(f"Working directory: {os.getcwd()}")


import json
import uproot
import awkward as ak
import vector
import numpy as np
import matplotlib.pyplot as plt
import warnings

# plotting params
plt.rcParams.update(
    {
        "figure.figsize": (10, 6),
        "axes.grid": True,
        "grid.alpha": 0.6,
        "grid.linestyle": "--",
        "font.size": 14,
        "figure.dpi": 200,
        "figure.facecolor": "none",
        "axes.facecolor": "white",
    }
)

# Suppress a harmless warning from the vector library with awkward arrays
warnings.filterwarnings("ignore", message="Passing an awkward array to a ufunc")

# Register the vector library with awkward array
ak.behavior.update(vector.backends.awkward.behavior)

# --- CONFIGURATION ---
with open("hh-bbbb-obj-config.json", "r") as config_file:
    config = json.load(config_file)

with open("config_part.json", "r") as config_file:
    CONFIG_PART = json.load(config_file)

## Load and test model

In [ ]:
import os
import json
import torch
import wandb
from model.parT import ParticleTransformer
from wandb_utils import extract_wandb_run_id

from data_pipeline.datasets import L1JetDataset
from data_pipeline.splitting import stratified_split
from data_pipeline.combined_loader import CombinedJetDataLoader
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm


torch.backends.cuda.matmul.allow_tf32 = True
print("TF32 matmul enabled for faster training on compatible GPUs.")

with open("config_part.json", "r") as f:
    config_part = json.load(f)

print(f"CWD: {os.getcwd()}")


device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Using device: {device}")

# run_path = "adityatandon29/L1-BTagging-ParT/4w2yuysk"
# run_id = extract_wandb_run_id(run_path)
# model_ckpt = torch.load(
#     "/Users/adityatandon/Documents/College/Physics/Year 4/Neural SBI/root-obj-perf/final_model.pth",
#     weights_only=True,
#     map_location=device,
# )
# model.load_state_dict(model_ckpt)


wandb.init(project="part-btag-analysis")
artifact_path = f"{config_part['wandb']['entity']}/{config_part['wandb']['project']}/{config_part['wandb']['artifact_name']}:{config_part['wandb']['ckpt_type']}"
artifact = wandb.use_artifact(
    artifact_path,
    type="model",
)
artifact_dir = artifact.download()
wandb.finish()
print(f"Model artifact downloaded from W&B: {artifact_dir}")
if os.path.exists(artifact_dir):
    print("Loading checkpoint")
    checkpoint_dir = os.path.join(artifact_dir, os.listdir(artifact_dir)[0])
    checkpoint = torch.load(checkpoint_dir, map_location=device, weights_only=False)
    config_part = checkpoint["config"]

    print(f"Exp name: {config_part['exp_name']}")
    try:
        print(f"Data path: {config_part['data_path']}")
        torch.manual_seed(42)

        dataset = L1JetDataset(config_part["data_path"])
        num_classes = config_part["model"]["num_classes"]

        # Perform stratified split
        train_dataset, val_dataset, train_indices, val_indices, stratify_labels = (
            stratified_split(
                dataset=dataset,
                val_split=config_part["training"]["val_split"],
                num_classes=num_classes,
                random_state=42,
                verbose=True,
            )
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=config_part["training"]["batch_size"],
            shuffle=True,
            num_workers=config_part["training"]["num_workers"],
            pin_memory=True,
            persistent_workers=True,
        )
        val_loader = DataLoader(
            val_dataset,
            batch_size=config_part["training"]["batch_size"],
            shuffle=False,
            num_workers=config_part["training"]["num_workers"],
            pin_memory=True,
            persistent_workers=True,
        )
        print("Data loaders prepared.")

    except KeyError:
        print("\nData path not found in checkpoint config")
        print("Using the new config structure.")
        dataset_used = config_part["training"]["data"]["use_dataset"]
        config_part["data_path"] = config_part["training"]["data"][
            f"{dataset_used}_data_path"
        ]
        config_part["training"]["val_split"] = config_part["training"]["data"][
            "val_split"
        ]
        config_part["training"]["val_split"] = config_part["training"]["data"][
            "val_split"
        ]
        config_part["training"]["num_workers"] = config_part["training"]["data"][
            "num_workers"
        ]
        print(f"Data path: {config_part['data_path']}")
        pt_regression = config_part.get("model", {}).get("pt_regression", False)
        combined_loader = CombinedJetDataLoader(
            # pf_data_path=config_part["training"]["data"]["pf_data_path"],
            pf_data_path=CONFIG_PART["training"]["data"]["pf_data_path"],
            puppi_data_path=config_part["training"]["data"]["puppi_data_path"],
            val_split=config_part["training"]["data"]["val_split"],
            batch_size=1024,
            match_mode=config_part["training"]["data"]["match_mode"],
            num_workers=4,
            random_state=42,
            # n_bins_pt=20,
            # n_bins_eta=20,
            # reweight_max_pt=500.0,
            # reweight_clip_max=100,
            # verbose=True,
            # smooth_sigma=0.0,
            # pt_min=40.0,
            # pt_max=70.0,
            pt_regression=pt_regression,
            # eta_min=-1.0,
            # eta_max=1.0,
        )
        if config_part["training"]["data"]["use_dataset"] == "pf":
            print("\nUsing PF dataset for training.")
            (
                train_loader,
                train_indices,
                val_loader,
                val_indices,
                train_labels,
                val_labels,
            ) = combined_loader.get_pf_loaders(shuffle=True)
        elif config_part["training"]["data"]["use_dataset"] == "puppi":
            print("\nUsing PUPPI dataset for training.")
            (
                train_loader,
                train_indices,
                val_loader,
                val_indices,
                train_labels,
                val_labels,
            ) = combined_loader.get_puppi_loaders(shuffle=True)
        print(
            f"Data loaders prepared with {len(train_loader.dataset)} training samples and {len(val_loader.dataset)} validation samples."
        )

    print(f"\nEpoch: {checkpoint['epoch']}")
    print(f"Val loss: {checkpoint['val_loss']}")
    print(f"Val_auc: {checkpoint['val_auc']}")
    pt_regression = config_part.get("model", {}).get("pt_regression", False)
    quantile_regression = config_part.get("model", {}).get("quantile_regression", False)
    model = ParticleTransformer(
        input_dim=config_part["input_dim"],
        embed_dim=config_part["model"]["embed_dim"],
        num_heads=config_part["model"]["num_heads"],
        num_layers=config_part["model"]["num_layers"],
        num_cls_layers=config_part["model"]["num_cls_layers"],
        dropout=config_part["model"]["dropout"],
        num_classes=config_part["model"]["num_classes"],
        pt_regression=pt_regression,
        quantile_regression=quantile_regression,
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(
        f"Checkpoint loaded (pt_regression={pt_regression}, quantile_regression={quantile_regression})"
    )

# Store as module-level for later cells
# CONFIG_PART = config_part

In [ ]:
# Run inference on validation set
model.eval()
all_outputs = []
all_labels = []
all_constituents = []  # Store full constituent data for jet reconstruction
all_reg_pt = []  # Regression head outputs (if pt_regression)
all_quantiles = []  # Quantile regression head outputs (if quantile_regression)
all_jet_pt_val = []  # Reco jet pT from dataset
all_gen_pt_val = []  # Gen pT from dataset
all_weights_val = []  # Kinematic reweighting weights from dataset (for training)
all_qcd_weights_val = []  # QCD cross-section weights (for testing metrics)

with torch.no_grad():
    for (
        X_batch,
        y_batch,
        mask_batch,
        weights_batch,
        jet_pt_batch,
        _,
        gen_pt_batch,
        qcd_weights_batch,
    ) in tqdm(val_loader):
        all_constituents.append(X_batch.cpu())
        X_batch = X_batch.to(device)
        mask_batch = mask_batch.to(device)
        y_batch = y_batch.to(device)
        weights_batch = weights_batch.to(device)

        out = model(X_batch, particle_mask=mask_batch)
        cls_logits = out["classification"]
        outputs = torch.nn.functional.sigmoid(cls_logits).squeeze()

        all_outputs.append(outputs.cpu())
        all_labels.append(y_batch.cpu())
        all_jet_pt_val.append(jet_pt_batch.squeeze().cpu())
        all_gen_pt_val.append(gen_pt_batch.squeeze().cpu())
        all_weights_val.append(weights_batch.squeeze().cpu())
        all_qcd_weights_val.append(qcd_weights_batch.squeeze().cpu())

        if "pt" in out:
            all_reg_pt.append(out["pt"].squeeze().cpu())
        if "quantiles" in out:
            all_quantiles.append(out["quantiles"].cpu())

all_outputs = torch.cat(all_outputs).numpy()
all_labels = torch.cat(all_labels).numpy()
all_constituents = torch.cat(all_constituents)
all_jet_pt_val = torch.cat(all_jet_pt_val).numpy()
all_gen_pt_val = torch.cat(all_gen_pt_val).numpy()
all_weights_val = torch.cat(all_weights_val).numpy()
all_qcd_weights_val = torch.cat(all_qcd_weights_val).numpy()

# Normalise raw cross-section weights from .npz (Convention C: raw σ_bin)
# to luminosity-scaled expected counts using evaluation.luminosity helpers
# Only use these scaled weights for significance calculations, not for ROC/AUC which should reflect the original dataset composition
from evaluation.luminosity import load_physics_config, scale_qcd_weights_raw

_physics = load_physics_config()
_sigma_to_ngen = {
    bin_cfg["weight"]: bin_cfg["n_gen"] for bin_cfg in config["QCD_background"].values()
}
if np.allclose(all_qcd_weights_val, 1.0):
    print(
        "WARNING: dataset has no qcd_weights key — using uniform 1.0 weights.\n"
        "ROC/AUC QCD weighting will be incorrect. Regenerate dataset with make_particle_dataset.py."
    )
has_regression = len(all_reg_pt) > 0
if has_regression:
    all_reg_pt = torch.cat(all_reg_pt).numpy()
    print(f"Regression outputs collected: {all_reg_pt.shape}")

has_quantile = len(all_quantiles) > 0
if has_quantile:
    all_quantiles = torch.cat(all_quantiles).numpy()
    print(f"Quantile regression outputs collected: {all_quantiles.shape}")
    print(
        f"  q16 range: {all_quantiles[:, 0].min():.4f} – {all_quantiles[:, 0].max():.4f}"
    )
    print(
        f"  q84 range: {all_quantiles[:, 1].min():.4f} – {all_quantiles[:, 1].max():.4f}"
    )
else:
    print("No quantile regression head in this model.")


# Reconstruct jet 4-vectors using vector library (proper 4-vector addition)
# Constituent format: [mass, pt, eta, phi, ...]
const_mass = all_constituents[:, :, 0].numpy()
const_pt = all_constituents[:, :, 1].numpy()
const_eta = all_constituents[:, :, 2].numpy()
const_phi = all_constituents[:, :, 3].numpy()

# Create constituent 4-vectors and sum to get jet 4-vectors
const_vectors = vector.array(
    {
        "pt": const_pt,
        "eta": const_eta,
        "phi": const_phi,
        "mass": const_mass,
    }
)
# Sum over constituents (axis=1) to get jet 4-vector
jet_vectors = const_vectors.sum(axis=1)

# Extract jet kinematics
val_jet_pt = jet_vectors.pt
val_jet_eta = jet_vectors.eta
val_jet_phi = jet_vectors.phi
val_jet_mass = jet_vectors.mass

print(f"\nJet kinematics reconstructed using vector library:")
print(f"  pT range: {val_jet_pt.min():.2f} - {val_jet_pt.max():.2f} GeV")
print(f"  eta range: {val_jet_eta.min():.2f} - {val_jet_eta.max():.2f}")

# Print regression info if available
if has_regression:
    sig_mask = all_labels.squeeze() == 1
    print(f"\nRegression summary (signal jets only):")
    print(
        f"  Predicted correction range: {all_reg_pt[sig_mask].min():.3f} – {all_reg_pt[sig_mask].max():.3f}"
    )
    print(
        f"  True correction (gen_pt/jet_pt) range: {(all_gen_pt_val[sig_mask] / (all_jet_pt_val[sig_mask] + 1e-9)).min():.3f} – {(all_gen_pt_val[sig_mask] / (all_jet_pt_val[sig_mask] + 1e-9)).max():.3f}"
    )

In [ ]:
print(f"Total validation samples: {len(all_outputs)}")
print(f"Num positive samples: {np.sum(all_labels)}")
print(f"Num negative samples: {len(all_outputs) - np.sum(all_labels)}")
print(f"% of positive samples: {100 * np.sum(all_labels) / len(all_outputs):.2f}%")

In [ ]:
# ============================================================
# REGRESSION HEAD ANALYSIS
# Uses model, val_loader, and regression outputs from cells above.
# ============================================================
signal_mask = all_labels.squeeze() == 1
jet_pt = all_jet_pt_val
gen_pt = all_gen_pt_val

print(f"Total validation jets: {len(all_labels)}")
print(f"  Signal jets: {signal_mask.sum()}")
print(f"  Background jets: {(~signal_mask).sum()}")
if has_regression:
    reg_pt = all_reg_pt
    print(f"\nRegression output (signal only):")
    print(
        f"  reg_pt  range: {reg_pt[signal_mask].min():.2f} – {reg_pt[signal_mask].max():.2f}"
    )
    print(
        f"  jet_pt  range: {jet_pt[signal_mask].min():.2f} – {jet_pt[signal_mask].max():.2f}"
    )
    print(
        f"  gen_pt  range: {gen_pt[signal_mask].min():.2f} – {gen_pt[signal_mask].max():.2f}"
    )
    correction = gen_pt[signal_mask] / (jet_pt[signal_mask] + 1e-9)
    print(
        f"  Correction factor (gen_pt / jet_pt) range: {correction.min():.3f} – {correction.max():.3f}"
    )
else:
    print("\nNo regression head in this model.")

# --- Plots ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. jet_pt distribution (signal vs background)
ax = axes[0, 0]
ax.hist(
    jet_pt[signal_mask],
    bins=60,
    range=(0, 500),
    histtype="step",
    density=True,
    label="Signal",
)
ax.hist(
    jet_pt[~signal_mask],
    bins=60,
    range=(0, 500),
    histtype="step",
    density=True,
    label="Background",
)
ax.set_xlabel("Reco jet $p_T$ [GeV]")
ax.set_ylabel("Density")
ax.set_title("Reco jet $p_T$ distribution")
ax.legend()

# 2. gen_pt distribution (signal only — bkg has gen_pt=0)
ax = axes[0, 1]
ax.hist(
    gen_pt[signal_mask],
    bins=60,
    range=(0, 500),
    histtype="step",
    density=True,
    color="C0",
)
ax.set_xlabel("Gen $p_T$ [GeV]")
ax.set_ylabel("Density")
ax.set_title("Gen $p_T$ (signal jets only)")

# 3. Correction factor = gen_pt / jet_pt (signal only)
ax = axes[0, 2]
correction = gen_pt[signal_mask] / (jet_pt[signal_mask] + 1e-9)
ax.hist(correction, bins=60, range=(0, 3), histtype="step", density=True, color="C2")
ax.axvline(1.0, color="gray", linestyle="--", alpha=0.7, label="No correction")
ax.set_xlabel("gen $p_T$ / reco $p_T$")
ax.set_ylabel("Density")
ax.set_title("Target correction factor (signal)")
ax.legend()

if has_regression:
    # 4. Regression output distribution
    ax = axes[1, 0]
    ax.hist(reg_pt[signal_mask], bins=60, histtype="step", density=True, label="Signal")
    ax.hist(
        reg_pt[~signal_mask], bins=60, histtype="step", density=True, label="Background"
    )
    ax.set_xlabel("Regression output (predicted correction)")
    ax.set_ylabel("Density")
    ax.set_title("Regression head output")
    ax.legend()

    # 5. Predicted correction vs true correction (signal only, scatter)
    ax = axes[1, 1]
    true_corr = gen_pt[signal_mask] / (jet_pt[signal_mask] + 1e-9)
    pred_corr = reg_pt[signal_mask]
    ax.scatter(true_corr, pred_corr, s=1, alpha=0.3)
    lim = max(true_corr.max(), pred_corr.max()) * 1.05
    ax.plot([0, lim], [0, lim], "r--", alpha=0.5, label="Ideal")
    ax.set_xlim(0, lim)
    ax.set_xlabel("True correction (gen $p_T$ / reco $p_T$)")
    ax.set_ylabel("Predicted correction")
    ax.set_title("Predicted vs true correction (signal)")
    ax.legend()

    # 6. Corrected jet pT vs gen pT (signal)
    ax = axes[1, 2]
    corrected_pt = jet_pt[signal_mask] * pred_corr
    ax.scatter(gen_pt[signal_mask], corrected_pt, s=1, alpha=0.3, label="Corrected")
    ax.scatter(
        gen_pt[signal_mask],
        jet_pt[signal_mask],
        s=1,
        alpha=0.1,
        color="gray",
        label="Uncorrected",
    )
    lim = 500
    ax.plot([0, lim], [0, lim], "r--", alpha=0.5)
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.set_xlabel("Gen $p_T$ [GeV]")
    ax.set_ylabel("Jet $p_T$ [GeV]")
    ax.set_title("Corrected vs uncorrected $p_T$ (signal)")
    ax.legend()
else:
    for i in range(3):
        axes[1, i].text(
            0.5,
            0.5,
            "No regression head",
            ha="center",
            va="center",
            transform=axes[1, i].transAxes,
        )
        axes[1, i].set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# training data
all_constituents_train = []
all_labels_train = []
all_masks_train = []
all_weights_train = []

for X_batch, y_batch, mask_batch, weights_batch, _, _, _, _ in tqdm(train_loader):
    all_constituents_train.append(X_batch.cpu())
    all_labels_train.append(y_batch.squeeze(-1).cpu())
    all_masks_train.append(mask_batch.cpu())
    all_weights_train.append(weights_batch.cpu())

all_constituents_train = torch.concat(all_constituents_train)
all_labels_train = torch.concat(all_labels_train)
all_masks_train = torch.concat(all_masks_train)
all_weights_train = torch.concat(all_weights_train)

const_mass_train = all_constituents_train[:, :, 0].numpy()
const_pt_train = all_constituents_train[:, :, 1].numpy()
const_eta_train = all_constituents_train[:, :, 2].numpy()
const_phi_train = all_constituents_train[:, :, 3].numpy()

# Create constituent 4-vectors and sum to get jet 4-vectors
const_vectors_train = vector.array(
    {
        "pt": const_pt_train,
        "eta": const_eta_train,
        "phi": const_phi_train,
        "mass": const_mass_train,
    }
)
# Sum over constituents (axis=1) to get jet 4-vector
jet_vectors_train = const_vectors_train.sum(axis=1)

# Extract jet kinematics
train_jet_pt = jet_vectors_train.pt
train_jet_eta = jet_vectors_train.eta
train_jet_phi = jet_vectors_train.phi
train_jet_mass = jet_vectors_train.mass

In [ ]:
# train signal/background vs val signal/background plots
fig, ax = plt.subplots()
ax.hist(
    train_jet_pt[all_labels_train == 1],
    bins=50,
    range=(0, 500),
    density=True,
    alpha=0.5,
    label="Train Signal",
)
ax.hist(
    train_jet_pt[all_labels_train == 0],
    bins=50,
    range=(0, 500),
    density=True,
    alpha=0.5,
    label="Train Background",
)
ax.hist(
    val_jet_pt[all_labels.reshape(all_labels.shape[0]) == 1],
    bins=50,
    range=(0, 500),
    density=True,
    histtype="step",
    label="Val Signal",
)
ax.hist(
    val_jet_pt[all_labels.reshape(all_labels.shape[0]) == 0],
    bins=50,
    range=(0, 500),
    density=True,
    histtype="step",
    label="Val Background",
)
ax.set_xlabel("Jet $p_T$ [GeV]")
ax.set_ylabel("Normalized Entries")
ax.set_title("Jet $p_T$ Distribution: Train vs Val")
ax.legend()
plt.show()

# reweighted training data plot
fig, ax = plt.subplots()
ax.hist(
    train_jet_pt[all_labels_train == 1],
    bins=50,
    range=(0, 500),
    weights=all_weights_train[all_labels_train == 1],
    density=True,
    alpha=0.5,
    label="Reweighted Train Signal",
)
ax.hist(
    train_jet_pt[all_labels_train == 0],
    bins=50,
    range=(0, 500),
    weights=all_weights_train[all_labels_train == 0],
    density=True,
    alpha=0.5,
    label="Reweighted Train Background",
)
ax.set_xlabel("Jet $p_T$ [GeV]")
ax.set_ylabel("Normalized Entries")
ax.set_title("Reweighted Jet $p_T$ Distribution: Train Set")
ax.legend()
plt.show()


fig, ax = plt.subplots()
ax.hist(
    train_jet_eta[all_labels_train == 1],
    bins=50,
    range=(-3, 3),
    weights=all_weights_train[all_labels_train == 1],
    density=True,
    alpha=0.5,
    label="Reweighted Train Signal",
)
ax.hist(
    train_jet_eta[all_labels_train == 0],
    bins=50,
    range=(-3, 3),
    weights=all_weights_train[all_labels_train == 0],
    density=True,
    alpha=0.5,
    label="Reweighted Train Background",
)
ax.set_xlabel("Jet $\\eta$ [GeV]")
ax.set_ylabel("Normalized Entries")
ax.set_title("Reweighted Jet $\\eta$ Distribution: Train Set")
ax.legend()
plt.show()

n_const_train = all_masks_train.sum(dim=1).cpu().numpy()
n_const_val = (all_constituents[:, :, 1] > 0).sum(dim=1).cpu().numpy()

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Training data
h0 = ax[0].hist2d(train_jet_eta, n_const_train, bins=50, cmap="viridis", cmin=1)
ax[0].set_xlabel(r"Jet $\eta$")
ax[0].set_ylabel("Number of Constituents")
ax[0].set_title("Train Set: $N_{const}$ vs $\eta$")
plt.colorbar(h0[3], ax=ax[0], label="Counts")

# Validation data
h1 = ax[1].hist2d(val_jet_eta, n_const_val, bins=50, cmap="viridis", cmin=1)
ax[1].set_xlabel(r"Jet $\eta$")
ax[1].set_ylabel("Number of Constituents")
ax[1].set_title("Validation Set: $N_{const}$ vs $\eta$")
plt.colorbar(h1[3], ax=ax[1], label="Counts")

plt.tight_layout()
plt.show()

In [ ]:
import evaluation.roc as roc_utils
from evaluation.luminosity import scale_qcd_weights_raw
import evaluation.luminosity as luminosity_utils
import importlib
import json, gc, os

# Ensure updated roc helper is available even if module was loaded before edits.
if "sig_weights" not in roc_utils.roc_from_scores.__code__.co_varnames:
    roc_utils = importlib.reload(roc_utils)
roc_from_scores = roc_utils.roc_from_scores
get_working_points = roc_utils.get_working_points

# Ensure build_eval_weights is available even if luminosity module was loaded before updates.
if not hasattr(luminosity_utils, "build_eval_weights"):
    luminosity_utils = importlib.reload(luminosity_utils)
build_eval_weights = luminosity_utils.build_eval_weights

# ============================================================
# ROC ANALYSIS AND COMPARISON WITH OTHER TAGGERS (CACHE-BASED)
# ============================================================

with open("hh-bbbb-obj-config.json", "r") as f:
    config = json.load(f)

# --- Config flags ---
eval_weight_mode = "qcd_only"  # "unweighted" | "qcd_only" | "full_physics"
ref_tagger_mode = "btag"  # "btag" (QCD b-jets = signal) | "trigger" (all QCD = bkg)
cache_pt_cut_gev = 25.0  # Set to None to disable cached-jet pT filtering

# Resolve cache path for both common notebook CWDs (repo root or notebooks/)
_cache_candidates = [
    "data/reference_tagger_cache.npz",
    "../data/reference_tagger_cache.npz",
]
CACHE_PATH = next(
    (p for p in _cache_candidates if os.path.exists(p)), _cache_candidates[-1]
)
print(f"Using reference tagger cache: {CACHE_PATH}")
if cache_pt_cut_gev is None:
    print("Cached-jet pT cut: disabled")
else:
    print(f"Cached-jet pT cut: pT >= {cache_pt_cut_gev:.1f} GeV")

# --- Load cache ---
cache = np.load(CACHE_PATH, allow_pickle=False)

_sigma_to_ngen = dict(
    zip(
        cache["meta_sigmas"].astype(float),
        cache["meta_ngens"].astype(int),
    )
)
_phys = config["physics"]
_luminosity_fb = _phys["luminosity_fb"]
_signal_xsec_pb = _phys["signal_xsec_pb"]
_n_gen_signal = _phys["n_gen_signal"]

# --- Tagger label -> cache prefix mapping ---
tagger_cache_map = {
    "Offline PNet": "Offline_PNet",
    "L1NG": "L1NG",
    "L1Ext": "L1Ext",
}

# Note: Offline UParT is not in cache by default (cache has PNet, L1NG, L1Ext).
ref_roc_results = {}
for label, prefix in tagger_cache_map.items():
    required_keys = [
        f"{prefix}_sig_scores",
        f"{prefix}_sig_pt",
        f"{prefix}_qcd_scores",
        f"{prefix}_qcd_labels",
        f"{prefix}_qcd_weights",
        f"{prefix}_qcd_pt",
    ]
    missing = [k for k in required_keys if k not in cache]
    if missing:
        raise KeyError(f"Missing cache arrays for {label}: {missing}")

    # Signal: gen-matched b-jets from HH ntuples
    sig_scores = cache[f"{prefix}_sig_scores"]
    sig_pt = cache[f"{prefix}_sig_pt"]

    # QCD jets
    qcd_scores = cache[f"{prefix}_qcd_scores"]
    qcd_labels = cache[f"{prefix}_qcd_labels"]
    qcd_weights_raw = cache[f"{prefix}_qcd_weights"]
    qcd_pt = cache[f"{prefix}_qcd_pt"]

    # Optional tunable pT cut on cached jets (applied to signal and QCD).
    n_sig_before = len(sig_scores)
    n_qcd_before = len(qcd_scores)
    if cache_pt_cut_gev is not None:
        sig_keep = sig_pt >= cache_pt_cut_gev
        qcd_keep = qcd_pt >= cache_pt_cut_gev

        sig_scores = sig_scores[sig_keep]
        qcd_scores = qcd_scores[qcd_keep]
        qcd_labels = qcd_labels[qcd_keep]
        qcd_weights_raw = qcd_weights_raw[qcd_keep]
        qcd_pt = qcd_pt[qcd_keep]

    if ref_tagger_mode == "btag":
        # b-tagging mode: positive class = HH b-jets + QCD b-jets; negative = QCD light jets.
        qcd_b_mask = qcd_labels == 1
        qcd_l_mask = ~qcd_b_mask

        hh_sig_scores = sig_scores
        qcd_b_sig_scores = qcd_scores[qcd_b_mask]
        qcd_b_sig_w_raw = qcd_weights_raw[qcd_b_mask]

        bkg_scores = qcd_scores[qcd_l_mask]
        bkg_w_raw = qcd_weights_raw[qcd_l_mask]

        sig_scores_combined = np.concatenate([hh_sig_scores, qcd_b_sig_scores])

        n_hh_sig = len(hh_sig_scores)
        n_qcd_b_sig = len(qcd_b_sig_scores)
        n_sig = len(sig_scores_combined)
        n_bkg = len(bkg_scores)
        if n_sig == 0 or n_bkg == 0:
            raise ValueError(
                f"After pT/mode selection, {label} has n_sig={n_sig}, n_bkg={n_bkg}. "
                "Lower cache_pt_cut_gev or change ref_tagger_mode."
            )

        if eval_weight_mode == "unweighted":
            sig_w = np.ones(n_sig, dtype=np.float64)
            bkg_w = np.ones(n_bkg, dtype=np.float64)
        elif eval_weight_mode == "qcd_only":
            # Keep HH jets at 1.0 while preserving QCD composition for both classes.
            qcd_b_sig_w = scale_qcd_weights_raw(
                qcd_b_sig_w_raw, _sigma_to_ngen, _luminosity_fb
            )
            bkg_w = scale_qcd_weights_raw(bkg_w_raw, _sigma_to_ngen, _luminosity_fb)
            sig_w = np.concatenate(
                [
                    np.ones(n_hh_sig, dtype=np.float64),
                    qcd_b_sig_w,
                ]
            )
        elif eval_weight_mode == "full_physics":
            # Mixed positive class convention: HH jets use signal physics weight,
            # QCD b-jets keep QCD per-event physics weights.
            hh_sig_w = luminosity_utils.signal_weight(
                n_hh_sig,
                luminosity_fb=_luminosity_fb,
                signal_xsec_pb=_signal_xsec_pb,
                n_gen_signal=_n_gen_signal,
            )
            qcd_b_sig_w = scale_qcd_weights_raw(
                qcd_b_sig_w_raw, _sigma_to_ngen, _luminosity_fb
            )
            bkg_w = scale_qcd_weights_raw(bkg_w_raw, _sigma_to_ngen, _luminosity_fb)
            sig_w = np.concatenate([hh_sig_w, qcd_b_sig_w])
        else:
            raise ValueError(
                f"Unknown eval_weight_mode {eval_weight_mode!r}. "
                "Choose from 'unweighted', 'qcd_only', 'full_physics'."
            )

        mode_summary = f"HH+QCDb signal: {n_hh_sig:,}+{n_qcd_b_sig:,}"
    else:
        # trigger mode: signal = HH only; background = all QCD.
        sig_scores_combined = sig_scores
        bkg_scores = qcd_scores
        bkg_w_raw = qcd_weights_raw

        n_sig = len(sig_scores_combined)
        n_bkg = len(bkg_scores)
        if n_sig == 0 or n_bkg == 0:
            raise ValueError(
                f"After pT/mode selection, {label} has n_sig={n_sig}, n_bkg={n_bkg}. "
                "Lower cache_pt_cut_gev or change ref_tagger_mode."
            )

        eval_w = build_eval_weights(
            bkg_w_raw,
            _sigma_to_ngen,
            n_sig,
            mode=eval_weight_mode,
            luminosity_fb=_luminosity_fb,
            signal_xsec_pb=_signal_xsec_pb,
            n_gen_signal=_n_gen_signal,
        )
        sig_w = eval_w[:n_sig]
        bkg_w = eval_w[n_sig:]
        mode_summary = "HH-only signal"

    roc_data = roc_from_scores(
        sig_scores_combined,
        bkg_scores,
        sig_weights=sig_w,
        bkg_weights=bkg_w,
    )
    ref_roc_results[label] = roc_data
    print(
        f"  {label}: AUC = {roc_data[2]:.4f}  |  n_sig = {n_sig:,}  |  n_bkg = {n_bkg:,} "
        f"| pT-cut kept sig {len(sig_scores):,}/{n_sig_before:,}, qcd {len(qcd_scores):,}/{n_qcd_before:,} "
        f"| {mode_summary}"
    )

del cache
gc.collect()

# --- Compute ROC for the Trained ParT model ---
# Apply kinematic cuts
from data_pipeline.root_loading import apply_custom_cuts
from sklearn.metrics import roc_curve, auc


val_jet_vectors = vector.array(
    {
        "pt": val_jet_pt,
        "eta": val_jet_eta,
        "phi": val_jet_phi,
        "mass": val_jet_mass,
    }
)

val_cuts_mask = apply_custom_cuts(
    val_jet_vectors, config, "l1barrelextpf", kinematic_only=True, return_jets=False
)
print(f"Jets after kinematic cuts: {np.sum(val_cuts_mask)}/{len(val_cuts_mask)}")

all_labels = all_labels.reshape(-1)
all_labels_after_cuts = all_labels[val_cuts_mask]
all_outputs_after_cuts = all_outputs[val_cuts_mask]
# Use QCD cross-section weights for metrics (not kinematic training weights)
all_weights_after_cuts = all_qcd_weights_val[val_cuts_mask]
# Keep kinematic training weights separately for reweighting visualisation plots
all_kinematic_weights_after_cuts = all_weights_val[val_cuts_mask]

# Compute ROC curve for trained ParT (using QCD weights)
trained_scores = np.asarray(all_outputs_after_cuts).reshape(-1)
trained_labels = np.asarray(all_labels_after_cuts).astype(int).reshape(-1)
trained_qcd_w_raw = np.asarray(all_weights_after_cuts).reshape(-1)

if len(trained_qcd_w_raw) != len(trained_labels):
    raise ValueError(
        f"Length mismatch for trained arrays: len(labels)={len(trained_labels)}, "
        f"len(qcd_weights_raw)={len(trained_qcd_w_raw)}"
    )

roc_weights = np.ones(len(trained_labels), dtype=np.float64)
bkg_mask = trained_labels == 0
sig_mask = ~bkg_mask

if eval_weight_mode == "qcd_only":
    print("Using QCD-only weighting for ROC calculation (Cell 10 style).")
    roc_weights[bkg_mask] = scale_qcd_weights_raw(
        trained_qcd_w_raw[bkg_mask], _sigma_to_ngen, _luminosity_fb
    )
elif eval_weight_mode == "full_physics":
    print("Using full-physics weighting for ROC calculation (Cell 10 style).")
    roc_weights[bkg_mask] = scale_qcd_weights_raw(
        trained_qcd_w_raw[bkg_mask], _sigma_to_ngen, _luminosity_fb
    )
    roc_weights[sig_mask] = luminosity_utils.signal_weight(
        int(np.sum(sig_mask)),
        luminosity_fb=_luminosity_fb,
        signal_xsec_pb=_signal_xsec_pb,
        n_gen_signal=_n_gen_signal,
    )
elif eval_weight_mode == "unweighted":
    print("Using uniform weights for ROC calculation.")
else:
    raise ValueError(
        f"Unknown eval_weight_mode {eval_weight_mode!r}. "
        "Choose from 'unweighted', 'qcd_only', 'full_physics'."
    )

# Keep memory lighter
roc_weights = np.asarray(roc_weights, dtype=np.float32)

# Recompute trained ParT ROC with selected weight mode
fpr_part, tpr_part, thresholds_part = roc_curve(
    trained_labels, trained_scores, sample_weight=roc_weights
)
roc_auc_part = auc(fpr_part, tpr_part)

# --- Export for downstream compatibility ---
offline_roc = ref_roc_results["Offline PNet"]
l1ng_roc = ref_roc_results["L1NG"]
l1ext_roc = ref_roc_results["L1Ext"]

pnet_wps = get_working_points("Offline PNet", offline_roc)
l1ng_wps = get_working_points("L1NG", l1ng_roc)
l1ext_wps = get_working_points("L1ExtJet", l1ext_roc)
part_wps = get_working_points("Trained ParT", (fpr_part, tpr_part, roc_auc_part, thresholds_part))

# Avoid stale values from older notebook runs when UParT was computed here.
if "offline_roc_upart" in globals():
    del offline_roc_upart
if "upart_wps" in globals():
    del upart_wps

print("\nWorking points computed.")
print(
    "Note: Offline UParT ROC/WPs are not produced in this cache cell unless UParT is added to TAGGER_REGISTRY."
)

In [ ]:
import evaluation.roc as roc_utils
import evaluation.luminosity as luminosity_utils
from evaluation.luminosity import scale_qcd_weights_raw
import importlib
import json
import os
import numpy as np
import matplotlib.pyplot as plt

roc_from_scores = roc_utils.roc_from_scores
build_eval_weights = luminosity_utils.build_eval_weights

# Compare all evaluation weighting modes for one reference mode at a time.
modes_to_compare = ["unweighted", "qcd_only", "full_physics"]
ref_mode_for_compare = ref_tagger_mode if "ref_tagger_mode" in globals() else "btag"
pt_cut_for_compare = cache_pt_cut_gev if "cache_pt_cut_gev" in globals() else 25.0

# Resolve cache path and load metadata.
_cache_candidates = [
    "data/reference_tagger_cache.npz",
    "../data/reference_tagger_cache.npz",
]
CACHE_PATH = next(
    (p for p in _cache_candidates if os.path.exists(p)), _cache_candidates[-1]
)
cache = np.load(CACHE_PATH, allow_pickle=False)

with open("hh-bbbb-obj-config.json", "r") as f:
    _cfg = json.load(f)

_sigma_to_ngen = dict(
    zip(cache["meta_sigmas"].astype(float), cache["meta_ngens"].astype(int))
)
_phys = _cfg["physics"]
_luminosity_fb = _phys["luminosity_fb"]
_signal_xsec_pb = _phys["signal_xsec_pb"]
_n_gen_signal = _phys["n_gen_signal"]

tagger_cache_map = {
    "Offline PNet": "Offline_PNet",
    "L1NG": "L1NG",
    "L1Ext": "L1Ext",
}


def _roc_for_mode(prefix, eval_mode, ref_mode, pt_cut_gev):
    sig_scores = cache[f"{prefix}_sig_scores"]
    sig_pt = cache[f"{prefix}_sig_pt"]
    qcd_scores = cache[f"{prefix}_qcd_scores"]
    qcd_labels = cache[f"{prefix}_qcd_labels"]
    qcd_weights_raw = cache[f"{prefix}_qcd_weights"]
    qcd_pt = cache[f"{prefix}_qcd_pt"]

    if pt_cut_gev is not None:
        sig_keep = sig_pt >= pt_cut_gev
        qcd_keep = qcd_pt >= pt_cut_gev
        sig_scores = sig_scores[sig_keep]
        qcd_scores = qcd_scores[qcd_keep]
        qcd_labels = qcd_labels[qcd_keep]
        qcd_weights_raw = qcd_weights_raw[qcd_keep]

    debug = {}

    if ref_mode == "btag":
        qcd_b_mask = qcd_labels == 1
        qcd_l_mask = ~qcd_b_mask

        hh_sig_scores = sig_scores
        qcd_b_sig_scores = qcd_scores[qcd_b_mask]
        qcd_b_sig_w_raw = qcd_weights_raw[qcd_b_mask]

        bkg_scores = qcd_scores[qcd_l_mask]
        bkg_w_raw = qcd_weights_raw[qcd_l_mask]

        sig_scores_combined = np.concatenate([hh_sig_scores, qcd_b_sig_scores])
        n_hh_sig = len(hh_sig_scores)
        n_qcd_b_sig = len(qcd_b_sig_scores)
        n_sig = len(sig_scores_combined)
        n_bkg = len(bkg_scores)

        if n_sig == 0 or n_bkg == 0:
            raise ValueError(
                f"Invalid sample sizes after cuts in btag mode: n_sig={n_sig}, n_bkg={n_bkg}"
            )

        if eval_mode == "unweighted":
            hh_sig_w = np.ones(n_hh_sig, dtype=np.float64)
            qcd_b_sig_w = np.ones(n_qcd_b_sig, dtype=np.float64)
            sig_w = np.concatenate([hh_sig_w, qcd_b_sig_w])
            bkg_w = np.ones(n_bkg, dtype=np.float64)
        elif eval_mode == "qcd_only":
            hh_sig_w = np.ones(n_hh_sig, dtype=np.float64)
            qcd_b_sig_w = scale_qcd_weights_raw(
                qcd_b_sig_w_raw, _sigma_to_ngen, _luminosity_fb
            )
            sig_w = np.concatenate([hh_sig_w, qcd_b_sig_w])
            bkg_w = scale_qcd_weights_raw(bkg_w_raw, _sigma_to_ngen, _luminosity_fb)
        elif eval_mode == "full_physics":
            hh_sig_w = luminosity_utils.signal_weight(
                n_hh_sig,
                luminosity_fb=_luminosity_fb,
                signal_xsec_pb=_signal_xsec_pb,
                n_gen_signal=_n_gen_signal,
            )
            qcd_b_sig_w = scale_qcd_weights_raw(
                qcd_b_sig_w_raw, _sigma_to_ngen, _luminosity_fb
            )
            sig_w = np.concatenate([hh_sig_w, qcd_b_sig_w])
            bkg_w = scale_qcd_weights_raw(bkg_w_raw, _sigma_to_ngen, _luminosity_fb)
        else:
            raise ValueError(f"Unknown eval_mode: {eval_mode}")

        debug = {
            "n_hh_sig": n_hh_sig,
            "n_qcd_b_sig": n_qcd_b_sig,
            "sum_hh_sig_w": float(np.sum(hh_sig_w)),
            "sum_qcd_b_sig_w": float(np.sum(qcd_b_sig_w)),
            "sum_sig_w": float(np.sum(sig_w)),
            "sum_bkg_w": float(np.sum(bkg_w)),
            "hh_sig_w_example": float(hh_sig_w[0]) if n_hh_sig > 0 else np.nan,
        }

    else:
        sig_scores_combined = sig_scores
        bkg_scores = qcd_scores
        bkg_w_raw = qcd_weights_raw

        n_sig = len(sig_scores_combined)
        n_bkg = len(bkg_scores)
        if n_sig == 0 or n_bkg == 0:
            raise ValueError(
                f"Invalid sample sizes after cuts in trigger mode: n_sig={n_sig}, n_bkg={n_bkg}"
            )

        eval_w = build_eval_weights(
            bkg_w_raw,
            _sigma_to_ngen,
            n_sig,
            mode=eval_mode,
            luminosity_fb=_luminosity_fb,
            signal_xsec_pb=_signal_xsec_pb,
            n_gen_signal=_n_gen_signal,
        )
        sig_w = eval_w[:n_sig]
        bkg_w = eval_w[n_sig:]

        debug = {
            "sum_sig_w": float(np.sum(sig_w)),
            "sum_bkg_w": float(np.sum(bkg_w)),
            "sig_w_example": float(sig_w[0]) if n_sig > 0 else np.nan,
        }

    roc_data = roc_from_scores(
        sig_scores_combined,
        bkg_scores,
        sig_weights=sig_w,
        bkg_weights=bkg_w,
    )
    return roc_data, sig_w, n_sig, n_bkg, debug


def _roc_for_trained_part(eval_mode):
    if (
        "all_outputs_after_cuts" not in globals()
        or "all_labels_after_cuts" not in globals()
        or "all_weights_after_cuts" not in globals()
    ):
        raise RuntimeError(
            "Trained ParT arrays not found. Run earlier validation/inference cells first."
        )

    scores_all = np.asarray(all_outputs_after_cuts).reshape(-1)
    labels_all = np.asarray(all_labels_after_cuts).astype(int).reshape(-1)
    qcd_w_raw_all = np.asarray(all_weights_after_cuts).reshape(-1)

    sig_scores = scores_all[labels_all == 1]
    bkg_scores = scores_all[labels_all == 0]
    bkg_w_raw = qcd_w_raw_all[labels_all == 0]

    n_sig = len(sig_scores)
    n_bkg = len(bkg_scores)
    if n_sig == 0 or n_bkg == 0:
        raise ValueError(
            f"Trained ParT has invalid sample sizes: n_sig={n_sig}, n_bkg={n_bkg}"
        )

    if eval_mode == "unweighted":
        sig_w = np.ones(n_sig, dtype=np.float64)
        bkg_w = np.ones(n_bkg, dtype=np.float64)
    elif eval_mode == "qcd_only":
        sig_w = np.ones(n_sig, dtype=np.float64)
        bkg_w = scale_qcd_weights_raw(bkg_w_raw, _sigma_to_ngen, _luminosity_fb)
    elif eval_mode == "full_physics":
        sig_w = luminosity_utils.signal_weight(
            n_sig,
            luminosity_fb=_luminosity_fb,
            signal_xsec_pb=_signal_xsec_pb,
            n_gen_signal=_n_gen_signal,
        )
        bkg_w = scale_qcd_weights_raw(bkg_w_raw, _sigma_to_ngen, _luminosity_fb)
    else:
        raise ValueError(f"Unknown eval_mode: {eval_mode}")

    debug = {
        "sum_sig_w": float(np.sum(sig_w)),
        "sum_bkg_w": float(np.sum(bkg_w)),
        "sig_w_example": float(sig_w[0]) if n_sig > 0 else np.nan,
    }

    roc_data = roc_from_scores(
        sig_scores,
        bkg_scores,
        sig_weights=sig_w,
        bkg_weights=bkg_w,
    )
    return roc_data, sig_w, n_sig, n_bkg, debug


all_compare_labels = list(tagger_cache_map.keys()) + ["Trained ParT"]
mode_roc_by_tagger = {label: {} for label in all_compare_labels}
mode_debug_by_tagger = {label: {} for label in all_compare_labels}

fig, axes = plt.subplots(
    1, len(all_compare_labels), figsize=(7 * len(all_compare_labels), 6), sharey=True
)
if len(all_compare_labels) == 1:
    axes = [axes]

mode_style = {
    "unweighted": {"linestyle": "-", "linewidth": 2.0},
    "qcd_only": {"linestyle": "--", "linewidth": 2.0},
    "full_physics": {"linestyle": ":", "linewidth": 2.4},
}

for ax, label in zip(axes, all_compare_labels):
    if label == "Trained ParT":
        print(f"\n{label} (trigger-style labels from validation set)")
    else:
        print(f"\n{label} ({ref_mode_for_compare} mode)")

    sig_w_qcd_only = None
    sig_w_full_physics = None

    for eval_mode in modes_to_compare:
        if label == "Trained ParT":
            roc_data, sig_w_mode, n_sig, n_bkg, debug = _roc_for_trained_part(eval_mode)
        else:
            prefix = tagger_cache_map[label]
            roc_data, sig_w_mode, n_sig, n_bkg, debug = _roc_for_mode(
                prefix,
                eval_mode,
                ref_mode_for_compare,
                pt_cut_for_compare,
            )

        mode_roc_by_tagger[label][eval_mode] = roc_data
        mode_debug_by_tagger[label][eval_mode] = debug

        if eval_mode == "qcd_only":
            sig_w_qcd_only = sig_w_mode
        elif eval_mode == "full_physics":
            sig_w_full_physics = sig_w_mode

        fpr_m, tpr_m, auc_m, _ = roc_data
        style = mode_style.get(eval_mode, {})
        ax.plot(
            fpr_m,
            tpr_m,
            label=f"{eval_mode} ROC (AUC={auc_m:.6f})",
            **style,
        )

        print(f"  {eval_mode:>12}: AUC={auc_m:.6f} | n_sig={n_sig:,} | n_bkg={n_bkg:,}")

        if label != "Trained ParT" and ref_mode_for_compare == "btag":
            hh_sum = debug["sum_hh_sig_w"]
            qcd_b_sum = debug["sum_qcd_b_sig_w"]
            sig_sum = debug["sum_sig_w"]
            hh_frac = hh_sum / (sig_sum + 1e-30)
            print(
                f"                 sig-weight split: HH={hh_sum:.3e} ({hh_frac:.3%}), "
                f"QCD-b={qcd_b_sum:.3e} ({1.0 - hh_frac:.3%})"
            )
            print(
                f"                 HH per-jet weight example: {debug['hh_sig_w_example']:.6e}"
            )

    if sig_w_qcd_only is not None and sig_w_full_physics is not None:
        wdiff = np.max(np.abs(sig_w_qcd_only - sig_w_full_physics))
        print(f"  max |sig_w(qcd_only)-sig_w(full_physics)| = {wdiff:.6e}")

    ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
    ax.set_title(label)
    ax.set_xlabel("FPR")
    ax.grid(alpha=0.3)
    ax.legend(loc="lower right", fontsize=8)

axes[0].set_ylabel("TPR")
pt_cut_text = "None" if pt_cut_for_compare is None else f"{pt_cut_for_compare:.1f} GeV"
fig.suptitle(
    f"ROC by evaluation mode | cache ref_mode={ref_mode_for_compare}, pT_cut={pt_cut_text}",
    fontsize=14,
)
plt.tight_layout()

if "save_fig_local" in globals():
    save_fig_local(fig, f"roc_compare_eval_modes_with_part_{ref_mode_for_compare}")
else:
    plt.show()

# Exposes results for downstream cells if needed.
roc_mode_compare_results = mode_roc_by_tagger
roc_mode_compare_debug = mode_debug_by_tagger

del cache

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

import evaluation.luminosity as luminosity_utils
from evaluation.luminosity import scale_qcd_weights_raw
import importlib

if not hasattr(luminosity_utils, "build_eval_weights"):
    luminosity_utils = importlib.reload(luminosity_utils)

# ------------------------------------------------------------------
# PR + Significance comparison across weighting modes, one tagger at a time
# Includes Offline PNet, L1NG, L1Ext (cache) + Trained ParT (validation set)
# ------------------------------------------------------------------

modes_to_compare = ["unweighted", "qcd_only", "full_physics"]
# For physical interpretation, use HH-only positives vs all QCD negatives.
ref_mode_for_prsig = "trigger"  # "trigger" recommended; set "btag" only if intentional.
pt_cut_for_prsig = cache_pt_cut_gev if "cache_pt_cut_gev" in globals() else 25.0

with open("hh-bbbb-obj-config.json", "r") as f:
    _cfg = json.load(f)

_sigma_to_ngen = {b["weight"]: b["n_gen"] for b in _cfg["QCD_background"].values()}
_phys = _cfg["physics"]
_luminosity_fb = _phys["luminosity_fb"]
_signal_xsec_pb = _phys["signal_xsec_pb"]
_n_gen_signal = _phys["n_gen_signal"]

_cache_candidates = [
    "data/reference_tagger_cache.npz",
    "../data/reference_tagger_cache.npz",
]
CACHE_PATH = next(
    (p for p in _cache_candidates if os.path.exists(p)), _cache_candidates[-1]
)
cache = np.load(CACHE_PATH, allow_pickle=False)

tagger_cache_map = {
    "Offline PNet": "Offline_PNet",
    "L1NG": "L1NG",
    "L1Ext": "L1Ext",
}


def _weights_for_mode_trigger_style(n_sig, bkg_w_raw, mode):
    if mode == "unweighted":
        sig_w = np.ones(n_sig, dtype=np.float64)
        bkg_w = np.ones(len(bkg_w_raw), dtype=np.float64)
    elif mode == "qcd_only":
        sig_w = np.ones(n_sig, dtype=np.float64)
        bkg_w = scale_qcd_weights_raw(bkg_w_raw, _sigma_to_ngen, _luminosity_fb)
    elif mode == "full_physics":
        sig_w = luminosity_utils.signal_weight(
            n_sig,
            luminosity_fb=_luminosity_fb,
            signal_xsec_pb=_signal_xsec_pb,
            n_gen_signal=_n_gen_signal,
        )
        bkg_w = scale_qcd_weights_raw(bkg_w_raw, _sigma_to_ngen, _luminosity_fb)
    else:
        raise ValueError(f"Unknown mode {mode!r}")
    return sig_w, bkg_w


def _cache_scores_weights(prefix, mode, ref_mode, pt_cut_gev):
    sig_scores = cache[f"{prefix}_sig_scores"]
    sig_pt = cache[f"{prefix}_sig_pt"]
    qcd_scores = cache[f"{prefix}_qcd_scores"]
    qcd_labels = cache[f"{prefix}_qcd_labels"]
    qcd_weights_raw = cache[f"{prefix}_qcd_weights"]
    qcd_pt = cache[f"{prefix}_qcd_pt"]

    if pt_cut_gev is not None:
        sig_keep = sig_pt >= pt_cut_gev
        qcd_keep = qcd_pt >= pt_cut_gev
        sig_scores = sig_scores[sig_keep]
        qcd_scores = qcd_scores[qcd_keep]
        qcd_labels = qcd_labels[qcd_keep]
        qcd_weights_raw = qcd_weights_raw[qcd_keep]

    if ref_mode == "btag":
        qcd_b_mask = qcd_labels == 1
        qcd_l_mask = ~qcd_b_mask

        hh_sig_scores = sig_scores
        qcd_b_sig_scores = qcd_scores[qcd_b_mask]
        qcd_b_sig_w_raw = qcd_weights_raw[qcd_b_mask]

        bkg_scores = qcd_scores[qcd_l_mask]
        bkg_w_raw = qcd_weights_raw[qcd_l_mask]

        n_hh_sig = len(hh_sig_scores)

        if mode == "unweighted":
            hh_sig_w = np.ones(n_hh_sig, dtype=np.float64)
            qcd_b_sig_w = np.ones(len(qcd_b_sig_scores), dtype=np.float64)
            sig_w = np.concatenate([hh_sig_w, qcd_b_sig_w])
            bkg_w = np.ones(len(bkg_scores), dtype=np.float64)
        elif mode == "qcd_only":
            hh_sig_w = np.ones(n_hh_sig, dtype=np.float64)
            qcd_b_sig_w = scale_qcd_weights_raw(
                qcd_b_sig_w_raw, _sigma_to_ngen, _luminosity_fb
            )
            sig_w = np.concatenate([hh_sig_w, qcd_b_sig_w])
            bkg_w = scale_qcd_weights_raw(bkg_w_raw, _sigma_to_ngen, _luminosity_fb)
        elif mode == "full_physics":
            hh_sig_w = luminosity_utils.signal_weight(
                n_hh_sig,
                luminosity_fb=_luminosity_fb,
                signal_xsec_pb=_signal_xsec_pb,
                n_gen_signal=_n_gen_signal,
            )
            qcd_b_sig_w = scale_qcd_weights_raw(
                qcd_b_sig_w_raw, _sigma_to_ngen, _luminosity_fb
            )
            sig_w = np.concatenate([hh_sig_w, qcd_b_sig_w])
            bkg_w = scale_qcd_weights_raw(bkg_w_raw, _sigma_to_ngen, _luminosity_fb)
        else:
            raise ValueError(f"Unknown mode {mode!r}")

        sig_scores_combined = np.concatenate([hh_sig_scores, qcd_b_sig_scores])
        return sig_scores_combined, bkg_scores, sig_w, bkg_w

    # trigger mode: HH-only positives, all QCD negatives
    bkg_scores = qcd_scores
    bkg_w_raw = qcd_weights_raw
    sig_w, bkg_w = _weights_for_mode_trigger_style(len(sig_scores), bkg_w_raw, mode)
    return sig_scores, bkg_scores, sig_w, bkg_w


def _class_normalize_weights(sig_w, bkg_w, target_sum=1.0):
    sig_norm = sig_w / (np.sum(sig_w) + 1e-30) * target_sum
    bkg_norm = bkg_w / (np.sum(bkg_w) + 1e-30) * target_sum
    return sig_norm, bkg_norm


def _curves(sig_scores, bkg_scores, sig_w_phys, bkg_w_phys, thr_grid):
    y_true = np.concatenate(
        [
            np.ones(len(sig_scores), dtype=np.int32),
            np.zeros(len(bkg_scores), dtype=np.int32),
        ]
    )
    y_score = np.concatenate([sig_scores, bkg_scores])

    sw_phys = np.concatenate([sig_w_phys, bkg_w_phys])
    sig_w_norm, bkg_w_norm = _class_normalize_weights(sig_w_phys, bkg_w_phys)
    sw_norm = np.concatenate([sig_w_norm, bkg_w_norm])

    # Top-row PR comparison uses class-normalized weights for meaningful shape comparison.
    precision_norm, recall_norm, _ = precision_recall_curve(
        y_true, y_score, sample_weight=sw_norm
    )
    pr_auc_norm = average_precision_score(y_true, y_score, sample_weight=sw_norm)

    # Keep also the physically weighted PR-AUC for reporting.
    pr_auc_phys = average_precision_score(y_true, y_score, sample_weight=sw_phys)

    # Significance always uses physical expected-yield weights.
    srb = []
    for t in thr_grid:
        s = np.sum(sig_w_phys[sig_scores >= t])
        b = np.sum(bkg_w_phys[bkg_scores >= t])
        srb.append(s / np.sqrt(b + s + 1e-30))
    srb = np.asarray(srb, dtype=np.float64)

    return precision_norm, recall_norm, pr_auc_norm, pr_auc_phys, srb


# Trained ParT inputs (trigger-style labels from validation set).
part_scores_all = all_outputs_after_cuts
part_labels_all = all_labels_after_cuts.astype(int)
part_sig_scores = part_scores_all[part_labels_all == 1]
part_bkg_scores = part_scores_all[part_labels_all == 0]
part_bkg_w_raw = all_weights_after_cuts[part_labels_all == 0]

if len(part_sig_scores) == 0 or len(part_bkg_scores) == 0:
    raise ValueError(
        "Trained ParT comparison requires both signal and background samples after cuts."
    )

thr_grid = np.linspace(0.0, 1.0, 301)
all_tagger_labels = ["Offline PNet", "L1NG", "L1Ext", "Trained ParT"]

fig, axes = plt.subplots(
    2, len(all_tagger_labels), figsize=(7 * len(all_tagger_labels), 10), squeeze=False
)

mode_style = {
    "unweighted": {"linestyle": "-", "linewidth": 2.0},
    "qcd_only": {"linestyle": "--", "linewidth": 2.0},
    "full_physics": {"linestyle": ":", "linewidth": 2.4},
}

prsig_compare_results = {label: {} for label in all_tagger_labels}

for col, label in enumerate(all_tagger_labels):
    ax_pr = axes[0, col]
    ax_sig = axes[1, col]

    for mode in modes_to_compare:
        if label == "Trained ParT":
            # Trained ParT is naturally trigger-style (HH=1, QCD=0 labels).
            sig_w, bkg_w = _weights_for_mode_trigger_style(
                len(part_sig_scores),
                part_bkg_w_raw,
                mode,
            )
            sig_scores = part_sig_scores
            bkg_scores = part_bkg_scores
        else:
            prefix = tagger_cache_map[label]
            sig_scores, bkg_scores, sig_w, bkg_w = _cache_scores_weights(
                prefix,
                mode,
                ref_mode_for_prsig,
                pt_cut_for_prsig,
            )

        precision, recall, pr_auc_norm, pr_auc_phys, srb = _curves(
            sig_scores,
            bkg_scores,
            sig_w,
            bkg_w,
            thr_grid,
        )
        best_idx = int(np.argmax(srb))
        best_thr = float(thr_grid[best_idx])
        best_srb = float(srb[best_idx])

        prsig_compare_results[label][mode] = {
            "pr_auc_class_normalized": pr_auc_norm,
            "pr_auc_physics_weighted": pr_auc_phys,
            "best_s_over_sqrt_s_plus_b": best_srb,
            "best_threshold": best_thr,
        }

        style = mode_style[mode]
        ax_pr.plot(
            recall,
            precision,
            label=f"{mode} (PR-AUCnorm={pr_auc_norm:.5f})",
            **style,
        )
        ax_sig.plot(
            thr_grid,
            srb,
            label=f"{mode} (max={best_srb:.3e} @ {best_thr:.3f})",
            **style,
        )

    ax_pr.set_title(label)
    ax_pr.set_xlabel("Recall")
    if col == 0:
        ax_pr.set_ylabel("Precision")
    ax_pr.grid(alpha=0.3)

    ax_sig.set_xlabel("Threshold")
    if col == 0:
        ax_sig.set_ylabel(r"$S/\sqrt{S+B}$")
    ax_sig.grid(alpha=0.3)

axes[0, -1].legend(loc="lower left", fontsize=9)
axes[1, -1].legend(loc="upper right", fontsize=8)

pt_cut_text = "None" if pt_cut_for_prsig is None else f"{pt_cut_for_prsig:.1f} GeV"
fig.suptitle(
    (
        "PR (class-normalized) and significance (physics-weighted) "
        f"comparison across modes | cache ref_mode={ref_mode_for_prsig}, pT_cut={pt_cut_text}"
    ),
    fontsize=14,
)
plt.tight_layout()

if "save_fig_local" in globals():
    save_fig_local(fig, f"pr_significance_compare_modes_with_part_{ref_mode_for_prsig}")
else:
    plt.show()

print("\nSummary (mode comparison):")
for label in all_tagger_labels:
    print(f"\n{label}")
    for mode in modes_to_compare:
        r = prsig_compare_results[label][mode]
        print(
            f"  {mode:>12}: PR-AUCnorm={r['pr_auc_class_normalized']:.6f} | "
            f"PR-AUCphys={r['pr_auc_physics_weighted']:.6f} | "
            f"max S/sqrt(S+B)={r['best_s_over_sqrt_s_plus_b']:.3e} @ thr={r['best_threshold']:.3f}"
        )

# Expose results for downstream use.
prsig_mode_compare_results = prsig_compare_results

del cache

## Regenerate cache
Reference-tagger ROC loading is now cache-driven in the cell above.

When new sims arrive, regenerate the cache first, then rerun the cached ROC cell:

```bash
conda run -n hep-root-ml python -m data_pipeline.cache_reference_taggers \
  --config hh-bbbb-obj-config.json \
  --output data/reference_tagger_cache.npz
```

Notes:
- Offline UParT is not included unless you add it to TAGGER_REGISTRY in data_pipeline/cache_reference_taggers.py and regenerate.
- Tunable cached-jet pT filtering is controlled by cache_pt_cut_gev in the cache ROC cell (set to None to disable).
- full_physics is supported in trigger mode (HH-only signal) and in btag mode using the mixed positive-class convention (HH + QCD b-jets).
- In btag + full_physics mode, HH positives use signal_weight and QCD positives/negatives use QCD per-event weights.
- If notebook CWD is notebooks/, the cache path is ../data/reference_tagger_cache.npz.
- If notebook CWD is root-obj-perf/, the cache path is data/reference_tagger_cache.npz.

In [ ]:
from evaluation.efficiency import efficiency_table
from evaluation.roc import pr_auc_and_opt_s_over_root_b

# ============================================================
# PR CURVE + S/sqrt(B) VS THRESHOLD + PER-BIN EFFICIENCY TABLES
# ============================================================
import pandas as pd
from sklearn.metrics import precision_recall_curve, average_precision_score

# Use post-cuts arrays if available
labels = all_labels_after_cuts
scores = all_outputs_after_cuts
weights = all_weights_after_cuts  # QCD cross-section weights
jet_pt = val_jet_pt[val_cuts_mask]
jet_eta = val_jet_eta[val_cuts_mask]

# Plot output directory for new plots
run_id = config_part.get("exp_name", "unknown_run").replace("/", "_")
artifact_name = CONFIG_PART.get("wandb", {}).get("artifact_name", "unknown_artifact")
artifact_type = CONFIG_PART.get("wandb", {}).get("ckpt_type", "unknown_type")
plot_dir = f"../Updates/plots_{run_id}/{artifact_name}:{artifact_type}"
os.makedirs(plot_dir, exist_ok=True)


def save_fig_local(fig, name):
    filepath = os.path.join(plot_dir, f"{name}.png")
    fig.savefig(filepath, dpi=150, bbox_inches="tight")
    print(f"  Saved: {name}.png")


# ------------------------------
# 1) Precision-Recall curve
# ------------------------------
precision, recall, pr_thresholds = precision_recall_curve(
    labels, scores, sample_weight=all_weights_after_cuts
)
pr_auc = average_precision_score(labels, scores, sample_weight=all_weights_after_cuts)
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall, precision, color="purple")
ax.set_xlabel("Recall (Signal Efficiency)")
ax.set_ylabel("Precision")
ax.set_title(f"Precision-Recall Curve (post-cuts) | PR-AUC={pr_auc:.4f}")
ax.grid(True, alpha=0.3)
save_fig_local(fig, "precision_recall_curve")
plt.show()

# ------------------------------
# 2) S/sqrt(B) vs threshold
# ------------------------------
# Use unique thresholds from scores for stability
thr = np.linspace(0, 1, 200)
s_over_root_b = []
s_counts = []
b_counts = []

for t in thr:
    preds = scores >= t
    s = weights[(preds == 1) & (labels == 1)].sum()
    b = weights[(preds == 1) & (labels == 0)].sum()
    s_counts.append(s)
    b_counts.append(b)
    s_over_root_b.append(s / np.sqrt(b + 1e-9))

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(thr, s_over_root_b, color="darkgreen")
ax.set_xlabel("Threshold")
ax.set_ylabel(r"$S/\sqrt{B}$")
ax.set_title(r"$S/\sqrt{B}$ vs Threshold (post-cuts)")
ax.grid(True, alpha=0.3)
save_fig_local(fig, "s_over_root_b_vs_threshold")
plt.show()

# ------------------------------
# 3) Per-bin efficiency tables
# ------------------------------
pt_bins = [25, 100, 200, 300, 400, 500, np.inf]
eta_bins = [0.0, 0.5, 1.0, 1.5, 2.4]
working_points = pnet_wps


for wp in working_points:
    df = efficiency_table(
        pt_bins, eta_bins, labels, scores, jet_pt, jet_eta, wp, sample_weights=weights
    )
    print(f"\nPer-bin efficiencies @ threshold = {wp}")
    display(df)


# ------------------------------
# 4) PR-AUC and optimal S/sqrt(B) per kinematic bin
# ------------------------------
df_pr_srb = pr_auc_and_opt_s_over_root_b(
    pt_bins, eta_bins, labels, scores, jet_pt, jet_eta, thr, sample_weights=weights
)
print("\nPer-bin PR-AUC and optimal S/sqrt(B)")
display(df_pr_srb)

In [ ]:
# ============================================================
# LOAD FULL DATASET FOR CONSTITUENT ANALYSIS (BATCHED)
# ============================================================
from data_pipeline.datasets import L1JetDataset
from data_pipeline.root_loading import load_and_prepare_data, apply_custom_cuts

full_dataset = L1JetDataset(filepath=config_part["data_path"])
n_total = len(full_dataset)
BATCH_SIZE = 1000

# Accumulate batched data
x_chunks, y_chunks, mask_chunks = [], [], []
for start in range(0, n_total, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n_total)
    x_b, y_b, m_b, _ = full_dataset[start:end]
    x_chunks.append(x_b)
    y_chunks.append(y_b)
    mask_chunks.append(m_b)
    print(f"  Loaded batch {start}-{end} / {n_total}", end="\r")

x_full = torch.cat(x_chunks, dim=0)
y_full = torch.cat(y_chunks, dim=0)
mask_full = torch.cat(mask_chunks, dim=0)
del x_chunks, y_chunks, mask_chunks
print(f"\nDataset loaded: {len(x_full)} jets")

# Constituent kinematics
const_mass_full = x_full[:, :, 0].numpy()
const_pt_full = x_full[:, :, 1].numpy()
const_eta_full = x_full[:, :, 2].numpy()
const_phi_full = x_full[:, :, 3].numpy()

# Reconstruct jet kinematics using vector library
const_vectors_full = vector.array(
    {
        "pt": const_pt_full,
        "eta": const_eta_full,
        "phi": const_phi_full,
        "mass": const_mass_full,
    }
)
jet_vectors_full = const_vectors_full.sum(axis=1)

# Feature names for constituent plots (excluding particle IDs)
feature_names = [
    "Mass",
    "Pt",
    "Eta",
    "Phi",
    "Dxy",
    "Z0",
    "Charge",
    "Log Pt Rel",
    "Delta Eta",
    "Delta Phi",
    "PUPPI weight",
    "Log Delta R",
]
x_features = x_full[:, :, : len(feature_names)]

# Load reference jet collections for comparison
with open("hh-bbbb-obj-config.json", "r") as f:
    config = json.load(f)

events = load_and_prepare_data(
    config["file_pattern"],
    config["tree_name"],
    [config["offline"]["collection_name"], config["l1ng"]["collection_name"]],
    config["max_events"],
    correct_pt=True,
)

offline_jets = apply_custom_cuts(
    events[config["offline"]["collection_name"]], config, "offline", kinematic_only=True
)
l1ng_jets = apply_custom_cuts(
    events[config["l1ng"]["collection_name"]], config, "l1ng", kinematic_only=True
)

# Flatten for histograms
offline_pt = ak.to_numpy(ak.flatten(offline_jets.pt))
offline_eta = ak.to_numpy(ak.flatten(offline_jets.eta))
l1ng_pt = ak.to_numpy(ak.flatten(l1ng_jets.pt))
l1ng_eta = ak.to_numpy(ak.flatten(l1ng_jets.eta))

print(
    f"Reconstructed jet pT range: {jet_vectors_full.pt.min():.2f} - {jet_vectors_full.pt.max():.2f} GeV"
)
print(
    f"Reconstructed jet eta range: {jet_vectors_full.eta.min():.2f} - {jet_vectors_full.eta.max():.2f}"
)

In [ ]:
# ============================================================
# PARTICLE TYPE DECODING
# ============================================================
PARTICLE_TYPE_NAMES = ["Charged Hadron", "Neutral Hadron", "Photon", "Electron", "Muon"]
PARTICLE_TYPE_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
N_PARTICLE_TYPES = 5

# Decode one-hot particle IDs (features 12-16) for full dataset
# IMPORTANT: mask out padded constituents first — argmax([0,0,0,0,0]) = 0
# which would falsely register as charged hadron
n_features = x_full.shape[2]
if n_features > 12:
    particle_type_full = np.argmax(x_full[:, :, 12:17].numpy(), axis=-1)  # (N_jets, 128)
    # Set padded positions to -1 so they are never counted
    mask_np_full = mask_full.numpy()
    particle_type_full[~mask_np_full] = -1

    # Compute per-jet particle composition
    particle_counts_full = np.zeros((len(x_full), N_PARTICLE_TYPES), dtype=int)
    for pid in range(N_PARTICLE_TYPES):
        particle_counts_full[:, pid] = (particle_type_full == pid).sum(axis=1)

    # Dominant particle type per jet (most common type)
    dominant_type_full = np.argmax(particle_counts_full, axis=1)

    print(f"Particle type distribution across all valid constituents:")
    total_valid = mask_np_full.sum()
    for pid in range(N_PARTICLE_TYPES):
        count = (particle_type_full == pid).sum()
        print(f"  {PARTICLE_TYPE_NAMES[pid]}: {count} ({100*count/total_valid:.1f}%)")
else:
    particle_type_full = None
    print("WARNING: Dataset has <= 12 features, no particle ID columns found")


## Generate and Save All Plots

This cell generates all analysis plots and saves them to `../Updates/plots_{run_id}/`

In [ ]:
run_id = config_part.get("exp_name", "unknown_run").replace("/", "_")
artifact_name = CONFIG_PART.get("wandb", {}).get("artifact_name", "unknown_artifact")
artifact_type = CONFIG_PART.get("wandb", {}).get("ckpt_type", "unknown_type")

print(
    f"Using plot directory: ../Updates/plots_{run_id}/{artifact_name}:{artifact_type}"
)

In [ ]:
from evaluation.roc import (
    calculate_trained_roc_2d_bins,
    calculate_auc_uncertainty_2d_bins,
)


import os
import gc
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

import evaluation.luminosity as luminosity_utils
from evaluation.luminosity import scale_qcd_weights_raw
from plotting.base import plot_roc_comparison

# ============================================================
# GENERATE AND SAVE ALL PLOTS
# ============================================================

# Fallback defaults if not set in previous cells
if "eval_weight_mode" not in globals():
    eval_weight_mode = "qcd_only"
    print("eval_weight_mode not found; defaulting to qcd_only")

if not all(
    k in globals()
    for k in ["_sigma_to_ngen", "_luminosity_fb", "_signal_xsec_pb", "_n_gen_signal"]
):
    with open("hh-bbbb-obj-config.json", "r") as f:
        _cfg = json.load(f)
    _sigma_to_ngen = {b["weight"]: b["n_gen"] for b in _cfg["QCD_background"].values()}
    _phys = _cfg["physics"]
    _luminosity_fb = _phys["luminosity_fb"]
    _signal_xsec_pb = _phys["signal_xsec_pb"]
    _n_gen_signal = _phys["n_gen_signal"]


# ============================================================
# GENERATE AND SAVE ALL PLOTS
# ============================================================
import os
import seaborn as sns
import pandas as pd
from sklearn.metrics import roc_auc_score

# Extract run_id from artifact path for plot directory
run_id = config_part.get("exp_name", "unknown_run").replace("/", "_")
artifact_name = CONFIG_PART.get("wandb", {}).get("artifact_name", "unknown_artifact")
artifact_type = CONFIG_PART.get("wandb", {}).get("ckpt_type", "unknown_type")
plot_dir = f"../Updates/plots_{run_id}/{artifact_name}:{artifact_type}"
os.makedirs(plot_dir, exist_ok=True)
print(f"Saving plots to: {plot_dir}")


def save_fig(fig, name):
    """Save figure to plot directory."""
    filepath = os.path.join(plot_dir, f"{name}.png")
    fig.savefig(filepath, dpi=150, bbox_inches="tight")
    print(f"  Saved: {name}.png")


# ============================================================
# 1. MODEL OUTPUT DISTRIBUTION
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(
    all_outputs_after_cuts[all_labels_after_cuts == 0],
    bins=50,
    range=(0, 1),
    label="Background",
    histtype="step",
    color="red",
    density=True,
)
ax.hist(
    all_outputs_after_cuts[all_labels_after_cuts == 1],
    bins=50,
    range=(0, 1),
    label="Signal",
    histtype="step",
    color="blue",
    density=True,
)
ax.set_xlabel("Model Output Score")
ax.set_ylabel("Density")
ax.set_title("Model Output Distribution on Validation Set")
ax.legend()
save_fig(fig, "model_output_distribution")
plt.show()

# ============================================================
# 2. ROC CURVE COMPARISON
# ============================================================
from plotting.base import plot_roc_comparison

fig_roc = plot_roc_comparison(
    [
        ("Offline PNet", offline_roc),
        ("L1NG", l1ng_roc),
        ("L1ExtJet", l1ext_roc),
        ("Trained ParT", (fpr_part, tpr_part, roc_auc_part, thresholds_part)),
    ],
    working_point=0.1,
    return_fig=True,
)
if fig_roc:
    save_fig(fig_roc, "roc_comparison")

# ============================================================
# 3. 2D AUC HEATMAP
# ============================================================
pt_ranges = [
    (25, 100),
    (100, 200),
    (200, 300),
    (300, 400),
    (400, 500),
    (500, np.inf),
    (25, np.inf),
]
eta_ranges = [(0, 0.5), (0.5, 1.0), (1.0, 1.5), (1.5, 2.4), (0, 1.5), (0, 2.4)]


# Use reconstructed jet kinematics
val_jet_pt_cuts = val_jet_pt[val_cuts_mask]
val_jet_eta_cuts = val_jet_eta[val_cuts_mask]

# AUC heatmap uses QCD weights if use_qcd_weights_for_roc is True; otherwise uniform weights
# So same roc_weights as used in the inclusive ROC calculation are passed here for consistency
auc_matrix, count_matrix = calculate_trained_roc_2d_bins(
    pt_ranges,
    eta_ranges,
    val_jet_pt_cuts,
    val_jet_eta_cuts,
    all_labels_after_cuts,
    all_outputs_after_cuts,
    weights=roc_weights,
)

# Plot heatmap
pt_labels = [f"[{low},{high})" for low, high in pt_ranges]
eta_labels = [f"[{low},{high})" for low, high in eta_ranges]
annot = np.empty_like(auc_matrix, dtype=object)
for i in range(auc_matrix.shape[0]):
    for j in range(auc_matrix.shape[1]):
        if np.isnan(auc_matrix[i, j]):
            annot[i, j] = f"N={int(count_matrix[i, j])}\n(N/A)"
        else:
            annot[i, j] = f"{auc_matrix[i, j]:.3f}\n(N={int(count_matrix[i, j])})"

fig, ax = plt.subplots(figsize=(12, 9))
df = pd.DataFrame(auc_matrix, index=pt_labels, columns=eta_labels)
sns.heatmap(
    df,
    annot=annot,
    fmt="",
    cmap="viridis",
    vmin=0.5,
    vmax=1.0,
    ax=ax,
    cbar_kws={"label": "AUC"},
)
ax.set_xlabel(r"$|\eta|$")
ax.set_ylabel("Jet pT [GeV]")
ax.set_title(r"Trained ParT: AUC in pT vs $|\eta|$ Bins")
plt.tight_layout()
save_fig(fig, "auc_heatmap_pt_eta")
plt.show()

# ============================================================
# AUC HEATMAP WITH BOOTSTRAP UNCERTAINTIES
# ============================================================


print("Computing AUC with bootstrap uncertainties...")
# Bootstrap AUC uses QCD weights
auc_matrix_u, unc_matrix_u, count_matrix_u = calculate_auc_uncertainty_2d_bins(
    pt_ranges,
    eta_ranges,
    val_jet_pt_cuts,
    val_jet_eta_cuts,
    all_labels_after_cuts,
    all_outputs_after_cuts,
    weights=roc_weights,
    n_boot=50,
)

annot_u = np.empty_like(auc_matrix_u, dtype=object)
for i in range(auc_matrix_u.shape[0]):
    for j in range(auc_matrix_u.shape[1]):
        if np.isnan(auc_matrix_u[i, j]):
            annot_u[i, j] = f"N={int(count_matrix_u[i, j])}\n(N/A)"
        elif np.isnan(unc_matrix_u[i, j]):
            annot_u[i, j] = f"{auc_matrix_u[i, j]:.3f}\n(N={int(count_matrix_u[i, j])})"
        else:
            annot_u[i, j] = (
                f"{auc_matrix_u[i, j]:.3f}±{unc_matrix_u[i, j]:.3f}\n(N={int(count_matrix_u[i, j])})"
            )

fig_unc, ax_unc = plt.subplots(figsize=(14, 10))
df_unc = pd.DataFrame(auc_matrix_u, index=pt_labels, columns=eta_labels)
sns.heatmap(
    df_unc,
    annot=annot_u,
    fmt="",
    cmap="viridis",
    vmin=0.5,
    vmax=1.0,
    ax=ax_unc,
    cbar_kws={"label": "AUC"},
    annot_kws={"fontsize": 9},
)
ax_unc.set_xlabel(r"$|\eta|$")
ax_unc.set_ylabel("Jet pT [GeV]")
ax_unc.set_title(r"Trained ParT: AUC with Bootstrap Uncertainty in pT vs $|\eta|$ Bins")
plt.tight_layout()
save_fig(fig_unc, "auc_heatmap_pt_eta_uncertainty")
plt.show()
print("AUC uncertainty heatmap saved!")

# ============================================================
# 4. CONSTITUENT FEATURE DISTRIBUTIONS
# ============================================================
signal_mask_full = (y_full == 1).numpy().flatten()
background_mask_full = (y_full == 0).numpy().flatten()
mask_np = mask_full.numpy()

for i, feat_name in enumerate(feature_names):
    fig, ax = plt.subplots(figsize=(10, 6))
    sig_vals = x_features[:, :, i].numpy()[(y_full == 1).numpy() & mask_np]
    bkg_vals = x_features[:, :, i].numpy()[(y_full == 0).numpy() & mask_np]
    sig_mask = all_labels_after_cuts == 1
    bkg_mask = all_labels_after_cuts == 0

    ax.hist(
        sig_vals.flatten(),
        bins=50,
        range=(np.percentile(sig_vals, 1), np.percentile(sig_vals, 99)),
        histtype="step",
        label="Signal (b-jets)",
        color="blue",
        density=True,
    )
    ax.hist(
        bkg_vals.flatten(),
        bins=50,
        range=(np.percentile(bkg_vals, 1), np.percentile(bkg_vals, 99)),
        histtype="step",
        label="Background",
        color="red",
        density=True,
    )
    ax.legend()
    ax.set_xlabel(f"Constituent {feat_name}")
    ax.set_ylabel("Density")
    ax.set_title(f"Constituent {feat_name} Distribution")
    save_fig(fig, f"constituent_{feat_name.lower().replace(' ', '_')}")
    plt.close(fig)

print(f"  Saved {len(feature_names)} constituent feature plots")

# ============================================================
# 5. JET KINEMATICS COMPARISON
# ============================================================
trained_pt = jet_vectors_full.pt
trained_eta = jet_vectors_full.eta

# pT comparison
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(
    trained_pt,
    bins=100,
    range=(0, 500),
    histtype="step",
    label=f"Trained AK4 Jets (N={len(trained_pt)})",
    color="green",
    density=True,
)
ax.hist(
    l1ng_pt,
    bins=100,
    range=(0, 500),
    histtype="step",
    label=f"L1NG Jets (N={len(l1ng_pt)})",
    color="blue",
    density=True,
)
ax.hist(
    offline_pt,
    bins=100,
    range=(0, 500),
    histtype="step",
    label=f"Offline Jets (N={len(offline_pt)})",
    color="red",
    density=True,
)
ax.axvline(25, color="black", linestyle="--", alpha=0.7, label="25 GeV cut")
ax.set_xlabel("Jet $p_T$ [GeV]")
ax.set_ylabel("Density")
ax.set_title("Jet $p_T$ Distribution Comparison")
ax.legend()
save_fig(fig, "jet_pt_comparison")
plt.show()

# eta comparison
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(
    trained_eta,
    bins=100,
    range=(-3, 3),
    histtype="step",
    label="Trained AK4 Jets",
    color="green",
    density=True,
)
ax.hist(
    l1ng_eta,
    bins=100,
    range=(-3, 3),
    histtype="step",
    label="L1NG Jets",
    color="blue",
    density=True,
)
ax.hist(
    offline_eta,
    bins=100,
    range=(-3, 3),
    histtype="step",
    label="Offline Jets",
    color="red",
    density=True,
)
ax.set_xlabel(r"Jet $\eta$")
ax.set_ylabel("Density")
ax.set_title(r"Jet $\eta$ Distribution Comparison")
ax.legend()
save_fig(fig, "jet_eta_comparison")
plt.show()

# ============================================================
# 6. SIGNAL VS BACKGROUND JET KINEMATICS
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# pT
ax = axes[0]
ax.hist(
    trained_pt[signal_mask_full],
    bins=50,
    range=(0, 500),
    histtype="step",
    label=f"Signal (N={signal_mask_full.sum()})",
    color="blue",
    density=True,
)
ax.hist(
    trained_pt[background_mask_full],
    bins=50,
    range=(0, 500),
    histtype="step",
    label=f"Background (N={background_mask_full.sum()})",
    color="red",
    density=True,
)

ax.set_xlabel("Jet $p_T$ [GeV]")
ax.set_ylabel("Density")
ax.set_title("Signal vs Background Jet $p_T$")
ax.legend()

# eta
ax = axes[1]
ax.hist(
    trained_eta[signal_mask_full],
    bins=50,
    range=(-3, 3),
    histtype="step",
    label="Signal",
    color="blue",
    density=True,
)
ax.hist(
    trained_eta[background_mask_full],
    bins=50,
    range=(-3, 3),
    histtype="step",
    label="Background",
    color="red",
    density=True,
)
ax.set_xlabel(r"Jet $\eta$")
ax.set_ylabel("Density")
ax.set_title(r"Signal vs Background Jet $\eta$")
ax.legend()

plt.tight_layout()
save_fig(fig, "signal_vs_background_kinematics")
plt.show()

# Signal vs background reweighting (pT) — uses kinematic training weights
# (not QCD weights) to visualize the effect of kinematic reweighting
# Use val_jet_pt_cuts / val_jet_eta_cuts (validation set after cuts) to match
# the shape of all_kinematic_weights_after_cuts
sig_mask_cuts = all_labels_after_cuts == 1
bkg_mask_cuts = all_labels_after_cuts == 0

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
bins = np.linspace(0, 1000, 50)
ax = axes[0]
ax.hist(
    val_jet_pt_cuts[sig_mask_cuts],
    bins=bins,
    histtype="step",
    label="Signal",
    color="blue",
    density=True,
    weights=all_kinematic_weights_after_cuts[sig_mask_cuts],
)
ax.hist(
    val_jet_pt_cuts[bkg_mask_cuts],
    bins=bins,
    histtype="step",
    label="Background",
    color="red",
    density=True,
    weights=all_kinematic_weights_after_cuts[bkg_mask_cuts],
)
ax.set_xlabel("Jet $p_T$ [GeV]")
ax.set_ylabel("Density")
ax.set_title("Signal vs Background Jet $p_T$ (Kinematic Reweighted)")

# eta
bins_eta = np.linspace(-3, 3, 50)
ax = axes[1]
ax.hist(
    val_jet_eta_cuts[sig_mask_cuts],
    bins=bins_eta,
    histtype="step",
    label="Signal",
    color="blue",
    density=True,
    weights=all_kinematic_weights_after_cuts[sig_mask_cuts],
)
ax.hist(
    val_jet_eta_cuts[bkg_mask_cuts],
    bins=bins_eta,
    histtype="step",
    label="Background",
    color="red",
    density=True,
    weights=all_kinematic_weights_after_cuts[bkg_mask_cuts],
)
ax.set_xlabel(r"Jet $\eta$")
ax.set_ylabel("Density")
ax.set_title(r"Signal vs Background Jet $\eta$ (Kinematic Reweighted)")

ax.legend()
save_fig(fig, "signal_vs_background_kinematics_reweighted")
plt.show()

print("Signal vs background reweighted kinematic plots saved!")


# ============================================================
# 7. 2D pT-eta DISTRIBUTIONS
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
for ax, (data_pt, data_eta, title) in zip(
    axes,
    [
        (trained_pt, trained_eta, "Trained AK4 Jets"),
        (l1ng_pt, l1ng_eta, "L1NG Jets"),
        (offline_pt, offline_eta, "Offline Jets"),
    ],
):
    h = ax.hist2d(
        data_eta,
        data_pt,
        bins=[50, 50],
        range=[[-3, 3], [0, 300]],
        cmap="viridis",
        norm=plt.matplotlib.colors.LogNorm(),
    )
    ax.set_xlabel(r"Jet $\eta$")
    ax.set_ylabel("Jet $p_T$ [GeV]")
    ax.set_title(f"{title}: $p_T$ vs $\\eta$")
    plt.colorbar(h[3], ax=ax, label="Counts")
plt.tight_layout()
save_fig(fig, "jet_pt_eta_2d_comparison" "")
plt.show()

print(f"\n{'='*60}")
print(f"All plots saved to: {plot_dir}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# CONSTITUENT FEATURE DISTRIBUTIONS — BY PARTICLE TYPE (NEW)
# ============================================================
if particle_type_full is not None:
    mask_np = mask_full.numpy()

    for i, feat_name in enumerate(feature_names):
        fig, (ax_all, ax_sig) = plt.subplots(1, 2, figsize=(18, 6))

        feat_vals = x_full[:, :, i].numpy()  # (N_jets, 128)

        for pid in range(N_PARTICLE_TYPES):
            # All jets
            type_mask = (particle_type_full == pid) & mask_np
            vals = feat_vals[type_mask]
            if len(vals) > 0:
                lo, hi = np.percentile(vals, [1, 99])
                ax_all.hist(vals, bins=50, range=(lo, hi), histtype="step",
                           label=PARTICLE_TYPE_NAMES[pid], color=PARTICLE_TYPE_COLORS[pid],
                           density=True, linewidth=1.5)

            # Signal jets only
            sig_mask = (y_full == 1).numpy().flatten()
            type_sig_mask = (particle_type_full == pid) & mask_np & sig_mask[:, None]
            vals_sig = feat_vals[type_sig_mask]
            if len(vals_sig) > 0:
                lo_s, hi_s = np.percentile(vals_sig, [1, 99])
                ax_sig.hist(vals_sig, bins=50, range=(lo_s, hi_s), histtype="step",
                           label=PARTICLE_TYPE_NAMES[pid], color=PARTICLE_TYPE_COLORS[pid],
                           density=True, linewidth=1.5)

        ax_all.set_xlabel(f"Constituent {feat_name}")
        ax_all.set_ylabel("Density")
        ax_all.set_title(f"{feat_name} — All Jets, by Particle Type")
        ax_all.legend(fontsize=9)

        ax_sig.set_xlabel(f"Constituent {feat_name}")
        ax_sig.set_ylabel("Density")
        ax_sig.set_title(f"{feat_name} — Signal (b-jets) Only, by Particle Type")
        ax_sig.legend(fontsize=9)

        plt.tight_layout()
        save_fig(fig, f"constituent_{feat_name.lower().replace(' ', '_')}_by_type")
        plt.close(fig)

    print(f"  Saved {len(feature_names)} constituent feature plots (by particle type)")


In [ ]:
# ============================================================
# PARTICLE COMPOSITION: SIGNAL vs BACKGROUND
# ============================================================
if particle_type_full is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    sig_jets = (y_full == 1).numpy().flatten()
    bkg_jets = (y_full == 0).numpy().flatten()

    # Mean particle counts per jet
    sig_counts_mean = particle_counts_full[sig_jets].mean(axis=0)
    bkg_counts_mean = particle_counts_full[bkg_jets].mean(axis=0)
    sig_counts_std = particle_counts_full[sig_jets].std(axis=0)
    bkg_counts_std = particle_counts_full[bkg_jets].std(axis=0)

    x_pos = np.arange(N_PARTICLE_TYPES)
    width = 0.35

    ax1.bar(x_pos - width/2, sig_counts_mean, width, yerr=sig_counts_std,
            label="Signal (b-jets)", color="blue", alpha=0.7, capsize=3)
    ax1.bar(x_pos + width/2, bkg_counts_mean, width, yerr=bkg_counts_std,
            label="Background", color="red", alpha=0.7, capsize=3)
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(PARTICLE_TYPE_NAMES, rotation=30, ha="right")
    ax1.set_ylabel("Mean Count per Jet")
    ax1.set_title("Particle Composition: Signal vs Background")
    ax1.legend()

    # Fraction plot
    sig_frac = sig_counts_mean / sig_counts_mean.sum()
    bkg_frac = bkg_counts_mean / bkg_counts_mean.sum()

    ax2.bar(x_pos - width/2, sig_frac, width, label="Signal", color="blue", alpha=0.7)
    ax2.bar(x_pos + width/2, bkg_frac, width, label="Background", color="red", alpha=0.7)
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(PARTICLE_TYPE_NAMES, rotation=30, ha="right")
    ax2.set_ylabel("Fraction of Constituents")
    ax2.set_title("Particle Type Fractions: Signal vs Background")
    ax2.legend()

    plt.tight_layout()
    save_fig(fig, "particle_composition_signal_vs_background")
    plt.show()


In [ ]:
# ============================================================
# PER-TYPE MULTIPLICITY DISTRIBUTIONS
# ============================================================
if particle_type_full is not None:
    fig, axes = plt.subplots(1, N_PARTICLE_TYPES, figsize=(4 * N_PARTICLE_TYPES, 5))

    sig_jets = (y_full == 1).numpy().flatten()
    bkg_jets = (y_full == 0).numpy().flatten()

    for pid in range(N_PARTICLE_TYPES):
        ax = axes[pid]
        sig_counts = particle_counts_full[sig_jets, pid]
        bkg_counts = particle_counts_full[bkg_jets, pid]

        max_count = max(sig_counts.max(), bkg_counts.max())
        bins = np.arange(0, min(max_count + 2, 50))

        ax.hist(sig_counts, bins=bins, histtype="step", label="Signal",
                color="blue", density=True, linewidth=1.5)
        ax.hist(bkg_counts, bins=bins, histtype="step", label="Background",
                color="red", density=True, linewidth=1.5)
        ax.set_xlabel(f"# {PARTICLE_TYPE_NAMES[pid]}s per Jet")
        ax.set_ylabel("Density")
        ax.set_title(PARTICLE_TYPE_NAMES[pid])
        ax.legend(fontsize=8)

    plt.suptitle("Constituent Multiplicity by Particle Type", fontsize=14)
    plt.tight_layout()
    save_fig(fig, "multiplicity_by_particle_type")
    plt.show()


In [ ]:
print(f"\n--- Saving regression head analysis plot ---")
signal_mask_reg = all_labels.squeeze() == 1
jet_pt_reg = all_jet_pt_val
gen_pt_reg = all_gen_pt_val

fig_reg, axes_reg = plt.subplots(2, 3, figsize=(18, 10))

ax = axes_reg[0, 0]
ax.hist(
    jet_pt_reg[signal_mask_reg],
    bins=60,
    range=(0, 500),
    histtype="step",
    density=True,
    label="Signal",
)
ax.hist(
    jet_pt_reg[~signal_mask_reg],
    bins=60,
    range=(0, 500),
    histtype="step",
    density=True,
    label="Background",
)
ax.set_xlabel("Reco jet $p_T$ [GeV]")
ax.set_ylabel("Density")
ax.set_title("Reco jet $p_T$ distribution")
ax.legend()

ax = axes_reg[0, 1]
ax.hist(
    gen_pt_reg[signal_mask_reg],
    bins=60,
    range=(0, 500),
    histtype="step",
    density=True,
    color="C0",
)
ax.set_xlabel("Gen $p_T$ [GeV]")
ax.set_ylabel("Density")
ax.set_title("Gen $p_T$ (signal jets only)")

ax = axes_reg[0, 2]
correction_reg = gen_pt_reg[signal_mask_reg] / (jet_pt_reg[signal_mask_reg] + 1e-9)
ax.hist(
    correction_reg, bins=60, range=(0, 3), histtype="step", density=True, color="C2"
)
ax.axvline(1.0, color="gray", linestyle="--", alpha=0.7, label="No correction")
ax.set_xlabel("gen $p_T$ / reco $p_T$")
ax.set_ylabel("Density")
ax.set_title("Target correction factor (signal)")
ax.legend()

if has_regression:
    reg_pt_reg = all_reg_pt
    ax = axes_reg[1, 0]
    ax.hist(
        reg_pt_reg[signal_mask_reg],
        bins=60,
        histtype="step",
        density=True,
        label="Signal",
    )
    ax.hist(
        reg_pt_reg[~signal_mask_reg],
        bins=60,
        histtype="step",
        density=True,
        label="Background",
    )
    ax.set_xlabel("Regression output")
    ax.set_ylabel("Density")
    ax.set_title("Regression head output")
    ax.legend()

    ax = axes_reg[1, 1]
    true_corr_reg = gen_pt_reg[signal_mask_reg] / (jet_pt_reg[signal_mask_reg] + 1e-9)
    pred_corr_reg = reg_pt_reg[signal_mask_reg]
    ax.scatter(true_corr_reg, pred_corr_reg, s=1, alpha=0.3)
    lim = max(true_corr_reg.max(), pred_corr_reg.max()) * 1.05
    ax.plot([0, lim], [0, lim], "r--", alpha=0.5, label="Ideal")
    ax.set_xlabel("True correction")
    ax.set_ylabel("Predicted correction")
    ax.set_title("Predicted vs true correction")
    ax.legend()

    ax = axes_reg[1, 2]
    corrected_pt_reg = jet_pt_reg[signal_mask_reg] * pred_corr_reg
    ax.scatter(
        gen_pt_reg[signal_mask_reg], corrected_pt_reg, s=1, alpha=0.3, label="Corrected"
    )
    ax.scatter(
        gen_pt_reg[signal_mask_reg],
        jet_pt_reg[signal_mask_reg],
        s=1,
        alpha=0.1,
        color="gray",
        label="Uncorrected",
    )
    ax.plot([0, 500], [0, 500], "r--", alpha=0.5)
    ax.set_xlabel("Gen $p_T$ [GeV]")
    ax.set_ylabel("Jet $p_T$ [GeV]")
    ax.set_xlim(0, 500)
    ax.set_ylim(0, 500)
    ax.set_title("Corrected vs uncorrected $p_T$")
    ax.legend()
else:
    for i in range(3):
        axes_reg[1, i].text(
            0.5,
            0.5,
            "No regression head",
            ha="center",
            va="center",
            transform=axes_reg[1, i].transAxes,
        )
        axes_reg[1, i].set_axis_off()

plt.tight_layout()
save_fig(fig_reg, "regression_head_analysis")
plt.show()

print(f"\nAll regression plots saved to: {plot_dir}")

In [ ]:
lim = 500
fig, ax = plt.subplots()
ax.scatter(
    gen_pt_reg[signal_mask_reg],
    jet_pt_reg[signal_mask_reg],
    s=1,
    alpha=0.1,
    color="black",
    label="Uncorrected",
)
ax.plot([0, lim], [0, lim], "r--", alpha=0.5)
ax.set_xlabel("Gen $p_T$ [GeV]")
ax.set_ylabel("Jet $p_T$ [GeV]")
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_title("Uncorrected $p_T$")
ax.legend()
plt.show()

fig, ax = plt.subplots()
corrected_pt_reg = jet_pt_reg[signal_mask_reg] * pred_corr_reg
ax.scatter(
    gen_pt_reg[signal_mask_reg], corrected_pt_reg, s=1, alpha=0.3, label="Corrected"
)
ax.plot([0, lim], [0, lim], "r--", alpha=0.5)
ax.set_xlabel("Gen $p_T$ [GeV]")
ax.set_ylabel("Jet $p_T$ [GeV]")
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_title("Corrected $p_T$")
ax.legend()
plt.show()

## Jet $p_T$ Resolution Analysis

Compute and compare the jet $p_T$ resolution **before** and **after** the regression correction.

**Predicted resolution** from the quantile regression head:

$$\sigma_{p_T} = \tfrac{1}{2}\,(q_{84} - q_{16})\;\times\;\text{jet\_pt}\;\times\;\text{jet\_scale}$$

where $q_{16}$, $q_{84}$ are the predicted 16th/84th percentiles of the $p_T$ response, and jet\_scale is the regression head output.

**Calculated resolution** is the Gaussian $\sigma$ of the $p_T$ response ($\text{reco}\;p_T / \text{gen}\;p_T$) fitted in bins of gen $p_T$ and $\eta$ — computed separately before (uncorrected) and after (corrected) the $p_T$ correction.

In [ ]:
from evaluation.resolution import gaussian, fit_response_in_bin, get_resolution_vs_var

# ============================================================
# JET pT RESOLUTION: PREDICTED (QUANTILE HEAD) vs CALCULATED
# ============================================================
from scipy.optimize import curve_fit


signal_mask_res = all_labels.squeeze() == 1
jet_pt_sig = all_jet_pt_val[signal_mask_res]
gen_pt_sig = all_gen_pt_val[signal_mask_res]
gen_eta_sig = val_jet_eta[signal_mask_res]  # use reco eta as proxy (matched)

# --- Uncorrected (raw) pT response ---
raw_response = jet_pt_sig / (gen_pt_sig + 1e-9)

# --- Corrected pT response (using regression head) ---
if has_regression:
    pred_scale = all_reg_pt[signal_mask_res]
    corrected_pt = jet_pt_sig * pred_scale
    corr_response = corrected_pt / (gen_pt_sig + 1e-9)
else:
    corrected_pt = jet_pt_sig
    corr_response = raw_response
    pred_scale = np.ones_like(jet_pt_sig)

# --- Predicted resolution from quantile head ---
if has_quantile:
    q16 = all_quantiles[signal_mask_res, 0]
    q84 = all_quantiles[signal_mask_res, 1]
    # Resolution = (1/2) * (q84 - q16) * jet_pt * jet_scale
    predicted_resolution_abs = 0.5 * (q84 - q16) * jet_pt_sig * pred_scale
    # Relative resolution (dimensionless) = (1/2) * (q84 - q16)
    predicted_resolution_rel = 0.5 * (q84 - q16)
    print(
        f"Predicted resolution (absolute) range: "
        f"{predicted_resolution_abs.min():.2f} – {predicted_resolution_abs.max():.2f} GeV"
    )
    print(
        f"Predicted resolution (relative)  range: "
        f"{predicted_resolution_rel.min():.4f} – {predicted_resolution_rel.max():.4f}"
    )

# --- Binning ---
pt_bins = np.linspace(25, 500, 30)
eta_bins = np.linspace(-2.4, 2.4, 20)

# Calculated resolution vs gen pT (before and after correction)
bc_pt, mu_raw_pt, sig_raw_pt, mu_raw_pt_err, sig_raw_pt_err = get_resolution_vs_var(
    gen_pt_sig, raw_response, pt_bins
)
bc_pt, mu_corr_pt, sig_corr_pt, mu_corr_pt_err, sig_corr_pt_err = get_resolution_vs_var(
    gen_pt_sig, corr_response, pt_bins
)

# Calculated resolution vs eta (before and after correction)
bc_eta, mu_raw_eta, sig_raw_eta, mu_raw_eta_err, sig_raw_eta_err = (
    get_resolution_vs_var(gen_eta_sig, raw_response, eta_bins)
)
bc_eta, mu_corr_eta, sig_corr_eta, mu_corr_eta_err, sig_corr_eta_err = (
    get_resolution_vs_var(gen_eta_sig, corr_response, eta_bins)
)

# Predicted (quantile) resolution averaged in bins
if has_quantile:

    def avg_in_bins(gen_var, values, var_bins):
        """Average values in bins of gen_var with std-of-mean error."""
        bc = 0.5 * (var_bins[1:] + var_bins[:-1])
        means, errs = [], []
        for i in range(len(var_bins) - 1):
            m = (gen_var > var_bins[i]) & (gen_var <= var_bins[i + 1])
            v = values[m]
            if len(v) > 5:
                means.append(np.mean(v))
                errs.append(np.std(v) / np.sqrt(len(v)))
            else:
                means.append(np.nan)
                errs.append(0.0)
        return bc, np.array(means), np.array(errs)

    _, pred_res_vs_pt, pred_res_vs_pt_err = avg_in_bins(
        gen_pt_sig, predicted_resolution_rel, pt_bins
    )
    _, pred_res_vs_eta, pred_res_vs_eta_err = avg_in_bins(
        gen_eta_sig, predicted_resolution_rel, eta_bins
    )

print("Resolution computation complete.")

In [ ]:
# ============================================================
# PLOT 1: pT Response Distributions (Before vs After Correction)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1a. Overall response distribution
ax = axes[0]
ax.hist(
    raw_response,
    bins=np.linspace(0, 2, 81),
    histtype="step",
    density=True,
    label="Uncorrected",
    color="C0",
    linewidth=1.5,
)
if has_regression:
    ax.hist(
        corr_response,
        bins=np.linspace(0, 2, 81),
        histtype="step",
        density=True,
        label="Corrected (regression)",
        color="C1",
        linewidth=1.5,
    )
ax.axvline(1.0, color="gray", linestyle="--", alpha=0.7, label="Ideal (1.0)")
ax.set_xlabel("$p_T$ Response (reco $p_T$ / gen $p_T$)")
ax.set_ylabel("Density")
ax.set_title("Jet $p_T$ Response Distribution (Signal Jets)")
ax.legend()
ax.set_xlim(0, 2)

# 1b. Response in selected gen pT bins
ax = axes[1]
pt_slices = [(0, 20), (20, 50), (50, 100), (100, 200), (200, 300), (300, 500)]
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(pt_slices)))
for (pt_lo, pt_hi), color in zip(pt_slices, colors):
    mask = (gen_pt_sig >= pt_lo) & (gen_pt_sig < pt_hi)
    if has_regression:
        ax.hist(
            corr_response[mask],
            bins=np.linspace(0, 2, 61),
            histtype="step",
            density=True,
            color=color,
            linewidth=1.5,
            label=f"Corrected [{pt_lo},{pt_hi}) GeV",
        )
    # ax.hist(
    #     raw_response[mask],
    #     bins=np.linspace(0, 2, 61),
    #     histtype="step",
    #     density=True,
    #     color=color,
    #     linewidth=1.0,
    #     linestyle="--",
    #     label=f"Uncorrected [{pt_lo},{pt_hi}) GeV",
    # )
ax.axvline(1.0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("$p_T$ Response")
ax.set_ylabel("Density")
ax.set_title("$p_T$ Response in gen $p_T$ Bins")
ax.legend(fontsize="x-small", ncol=1)
ax.set_xlim(0, 2)

plt.tight_layout()
save_fig(fig, "pt_response_distributions")
plt.show()

In [ ]:
fig, ax = plt.subplots()
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(pt_slices)))
for (pt_lo, pt_hi), color in zip(pt_slices, colors):
    mask = (gen_pt_sig >= pt_lo) & (gen_pt_sig < pt_hi)
    if has_regression:
        ax.hist(
            raw_response[mask],
            bins=np.linspace(0, 2, 61),
            histtype="step",
            density=True,
            color=color,
            linewidth=1.0,
            linestyle="--",
            label=f"Uncorrected [{pt_lo},{pt_hi}) GeV",
        )
ax.axvline(1.0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("$p_T$ Response")
ax.set_ylabel("Density")
ax.set_title("$p_T$ Response in gen $p_T$ Bins")
ax.legend(fontsize="x-small", ncol=1)
ax.set_xlim(0, 2)

plt.tight_layout()
# save_fig(fig, "pt_response_distributions")
plt.show()

In [ ]:
# ============================================================
# PLOT 2: Scale and Resolution vs gen pT (Before & After Correction)
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# --- Top panel: Scale (mean of response) vs gen pT ---
ax = axes[0]
ax.errorbar(
    bc_pt,
    mu_raw_pt,
    yerr=mu_raw_pt_err,
    marker="o",
    linestyle="-",
    capsize=4,
    label="Uncorrected",
    color="C0",
)
if has_regression:
    ax.errorbar(
        bc_pt,
        mu_corr_pt,
        yerr=mu_corr_pt_err,
        marker="s",
        linestyle="-",
        capsize=4,
        label="Corrected (regression)",
        color="C1",
    )
ax.axhline(1.0, color="black", linestyle="-", linewidth=0.8)
ax.set_ylabel("Jet $p_T$ Scale (Mean of Response)")
ax.set_title("Jet $p_T$ Scale & Resolution vs. Generated $p_T$ (Signal Jets)")
ax.legend()
ax.set_ylim(0.5, 1.5)

# --- Bottom panel: Resolution (sigma) vs gen pT ---
ax = axes[1]
ax.errorbar(
    bc_pt,
    sig_raw_pt,
    yerr=sig_raw_pt_err,
    marker="o",
    linestyle="-",
    capsize=4,
    label="Calculated (uncorrected)",
    color="C0",
)
if has_regression:
    ax.errorbar(
        bc_pt,
        sig_corr_pt,
        yerr=sig_corr_pt_err,
        marker="s",
        linestyle="-",
        capsize=4,
        label="Calculated (corrected)",
        color="C1",
    )
if has_quantile:
    ax.errorbar(
        bc_pt,
        pred_res_vs_pt,
        yerr=pred_res_vs_pt_err,
        marker="^",
        linestyle="--",
        capsize=4,
        label="Predicted (quantile head)",
        color="C2",
    )
ax.set_xlabel("Generated Jet $p_T$ [GeV]")
ax.set_ylabel("Resolution ($\\sigma$ of Response)")
ax.legend()
ax.set_ylim(0, 1.0)

plt.tight_layout()
save_fig(fig, "resolution_vs_gen_pt")
plt.show()

In [ ]:
# ============================================================
# PLOT 3: Scale and Resolution vs eta (Before & After Correction)
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# --- Top: Scale vs eta ---
ax = axes[0]
ax.errorbar(
    bc_eta,
    mu_raw_eta,
    yerr=mu_raw_eta_err,
    marker="o",
    linestyle="-",
    capsize=4,
    label="Uncorrected",
    color="C0",
)
if has_regression:
    ax.errorbar(
        bc_eta,
        mu_corr_eta,
        yerr=mu_corr_eta_err,
        marker="s",
        linestyle="-",
        capsize=4,
        label="Corrected (regression)",
        color="C1",
    )
ax.axhline(1.0, color="black", linestyle="-", linewidth=0.8)
ax.set_ylabel("Jet $p_T$ Scale (Mean of Response)")
ax.set_title("Jet $p_T$ Scale & Resolution vs. Jet $\\eta$ (Signal Jets)")
ax.legend()
ax.set_ylim(0.5, 1.5)

# --- Bottom: Resolution vs eta ---
ax = axes[1]
ax.errorbar(
    bc_eta,
    sig_raw_eta,
    yerr=sig_raw_eta_err,
    marker="o",
    linestyle="-",
    capsize=4,
    label="Calculated (uncorrected)",
    color="C0",
)
if has_regression:
    ax.errorbar(
        bc_eta,
        sig_corr_eta,
        yerr=sig_corr_eta_err,
        marker="s",
        linestyle="-",
        capsize=4,
        label="Calculated (corrected)",
        color="C1",
    )
if has_quantile:
    ax.errorbar(
        bc_eta,
        pred_res_vs_eta,
        yerr=pred_res_vs_eta_err,
        marker="^",
        linestyle="--",
        capsize=4,
        label="Predicted (quantile head)",
        color="C2",
    )
ax.set_xlabel("Jet $\\eta$")
ax.set_ylabel("Resolution ($\\sigma$ of Response)")
ax.legend()
ax.set_ylim(0)

plt.tight_layout()
save_fig(fig, "resolution_vs_eta")
plt.show()

In [ ]:
# ============================================================
# PLOT 4: 2D Response Maps (Before & After) + Predicted Resolution
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

# 4a. Uncorrected pT response vs gen pT
ax = axes[0]
h0 = ax.hist2d(
    gen_pt_sig,
    raw_response,
    bins=[np.linspace(25, 500, 51), np.linspace(0, 2, 51)],
    cmap="viridis",
    cmin=1,
)
ax.axhline(1.0, color="red", linestyle="--", alpha=0.7)
ax.set_xlabel("Generated $p_T$ [GeV]")
ax.set_ylabel("$p_T$ Response (reco / gen)")
ax.set_title("Uncorrected $p_T$ Response")
plt.colorbar(h0[3], ax=ax, label="Counts")

# 4b. Corrected pT response vs gen pT
ax = axes[1]
h1 = ax.hist2d(
    gen_pt_sig,
    corr_response,
    bins=[np.linspace(25, 500, 51), np.linspace(0, 2, 51)],
    cmap="viridis",
    cmin=1,
)
ax.axhline(1.0, color="red", linestyle="--", alpha=0.7)
ax.set_xlabel("Generated $p_T$ [GeV]")
ax.set_ylabel("$p_T$ Response (corrected / gen)")
ax.set_title("Corrected $p_T$ Response")
plt.colorbar(h1[3], ax=ax, label="Counts")

# 4c. Predicted resolution vs gen pT (scatter)
ax = axes[2]
if has_quantile:
    sc = ax.scatter(
        gen_pt_sig,
        predicted_resolution_rel,
        c=jet_pt_sig,
        s=1,
        alpha=0.3,
        cmap="plasma",
    )
    plt.colorbar(sc, ax=ax, label="Reco Jet $p_T$ [GeV]")
    ax.set_xlabel("Generated $p_T$ [GeV]")
    ax.set_ylabel("Predicted Relative Resolution")
    ax.set_title("Quantile-Predicted Resolution vs. Gen $p_T$")
    ax.set_ylim(0, 0.5)
else:
    ax.text(
        0.5,
        0.5,
        "No quantile head",
        ha="center",
        va="center",
        transform=ax.transAxes,
        fontsize=14,
    )
    ax.set_axis_off()

plt.tight_layout()
save_fig(fig, "response_2d_maps")
plt.show()

In [ ]:
# ============================================================
# PLOT 5: Gaussian fits in selected gen pT slices
# ============================================================
pt_slices_fit = [(25, 100), (100, 150), (200, 250), (300, 400)]
n_slices = len(pt_slices_fit)
fig, axes = plt.subplots(2, n_slices, figsize=(5 * n_slices, 10))

response_bins = np.linspace(0, 2, 61)
x_fit = np.linspace(0, 2, 200)

for j, (pt_lo, pt_hi) in enumerate(pt_slices_fit):
    mask = (gen_pt_sig >= pt_lo) & (gen_pt_sig < pt_hi)

    # --- Top row: Uncorrected ---
    ax = axes[0, j]
    vals_raw = raw_response[mask]
    counts_raw, _ = np.histogram(vals_raw, bins=response_bins)
    (mu_r, sig_r, A_r), _ = fit_response_in_bin(vals_raw, response_bins)
    ax.stairs(counts_raw, response_bins, color="C0", linewidth=1.5, label="Uncorrected")
    if not np.isnan(mu_r):
        ax.plot(
            x_fit,
            gaussian(x_fit, mu_r, sig_r, A_r),
            "C0--",
            label=f"$\\mu$={mu_r:.3f}, $\\sigma$={abs(sig_r):.3f}",
        )
    ax.set_title(f"gen $p_T$ [{pt_lo},{pt_hi}) GeV\n(Uncorrected)")
    ax.set_xlabel("$p_T$ Response")
    ax.set_ylabel("Counts")
    ax.legend(fontsize="x-small")

    # --- Bottom row: Corrected ---
    ax = axes[1, j]
    if has_regression:
        vals_corr = corr_response[mask]
        counts_corr, _ = np.histogram(vals_corr, bins=response_bins)
        (mu_c, sig_c, A_c), _ = fit_response_in_bin(vals_corr, response_bins)
        ax.stairs(
            counts_corr, response_bins, color="C1", linewidth=1.5, label="Corrected"
        )
        if not np.isnan(mu_c):
            ax.plot(
                x_fit,
                gaussian(x_fit, mu_c, sig_c, A_c),
                "C1--",
                label=f"$\\mu$={mu_c:.3f}, $\\sigma$={abs(sig_c):.3f}",
            )
        ax.set_title(f"gen $p_T$ [{pt_lo},{pt_hi}) GeV\n(Corrected)")
    else:
        ax.text(
            0.5,
            0.5,
            "No regression head",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
    ax.set_xlabel("$p_T$ Response")
    ax.set_ylabel("Counts")
    ax.legend(fontsize="x-small")

plt.tight_layout()
save_fig(fig, "response_gaussian_fits_pt_slices")
plt.show()

In [ ]:
# ============================================================
# PLOT 6: Absolute pT Resolution (GeV) and Improvement Summary
# ============================================================
if has_quantile:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # 6a. Absolute resolution: predicted vs calculated, before & after
    ax = axes[0]
    # Calculated absolute resolution = sigma_relative * gen_pt for each bin
    abs_raw_res = sig_raw_pt * bc_pt
    abs_corr_res = sig_corr_pt * bc_pt
    ax.errorbar(
        bc_pt,
        abs_raw_res,
        marker="o",
        linestyle="-",
        capsize=4,
        label="Calculated (uncorrected)",
        color="C0",
    )
    if has_regression:
        ax.errorbar(
            bc_pt,
            abs_corr_res,
            marker="s",
            linestyle="-",
            capsize=4,
            label="Calculated (corrected)",
            color="C1",
        )
    _, pred_abs_vs_pt, pred_abs_vs_pt_err = avg_in_bins(
        gen_pt_sig, predicted_resolution_abs, pt_bins
    )
    ax.errorbar(
        bc_pt,
        pred_abs_vs_pt,
        yerr=pred_abs_vs_pt_err,
        marker="^",
        linestyle="--",
        capsize=4,
        label="Predicted (quantile head)",
        color="C2",
    )
    ax.set_xlabel("Generated Jet $p_T$ [GeV]")
    ax.set_ylabel("Absolute $p_T$ Resolution [GeV]")
    ax.set_title("Absolute Jet $p_T$ Resolution vs. Gen $p_T$")
    ax.legend()
    ax.set_ylim(0)

    # 6b. Fractional improvement: (sigma_raw - sigma_corr) / sigma_raw
    ax = axes[1]
    if has_regression:
        valid = (~np.isnan(sig_raw_pt)) & (~np.isnan(sig_corr_pt)) & (sig_raw_pt > 0)
        improvement = (sig_raw_pt[valid] - sig_corr_pt[valid]) / sig_raw_pt[valid] * 100
        ax.bar(
            bc_pt[valid],
            improvement,
            width=np.diff(pt_bins).mean() * 0.7,
            color="C3",
            alpha=0.7,
        )
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_xlabel("Generated Jet $p_T$ [GeV]")
        ax.set_ylabel("Resolution Improvement [%]")
        ax.set_title("Resolution Improvement from Regression Correction")
        # Print summary
        mean_improvement = np.nanmean(improvement)
        ax.text(
            0.05,
            0.95,
            f"Mean improvement: {mean_improvement:.1f}%",
            transform=ax.transAxes,
            va="top",
            fontsize=12,
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
        )
    else:
        ax.text(
            0.5,
            0.5,
            "No regression head",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )

    plt.tight_layout()
    save_fig(fig, "absolute_resolution_and_improvement")
    plt.show()
else:
    print("Skipping absolute resolution plots (no quantile regression head).")

print(f"\nAll resolution plots saved to: {plot_dir}")

## Di-Higgs Mass Reconstruction with ParT B-Tag Scores

Load L1 PF and PUPPI candidates from ROOT files, cluster with FastJet, extract constituent features, score each jet with the trained ParT model, apply a working-point b-tag cut, correct jet $p_T$ with the regression head, and reconstruct $m_{H_1}$, $m_{H_2}$, $m_{HH}$ via $D_{HH}$ minimisation.

In [ ]:
# ============================================================
# DI-HIGGS MASS RECONSTRUCTION WITH TRAINED ParT
# ============================================================
# Restructured so clustering happens ONCE (no b-tag threshold).
# All jets are scored and their per-event lightweight data stored
# in `dh_raw`.  A downstream cell then sweeps b-tag cuts to find
# the optimal working point and builds `part_dihiggs_result`.
#
# What is stored per event (events with ≥4 clustered jets):
#   Signal: min_btag (4th jet score), lead_m, sub_m, hh_m,
#           is_pure (all 4 gen-matched), pair_ok / pair_swap
#           (algorithm chose correct / swapped gen pairing)
#   QCD   : min_btag, lead_m, sub_m, hh_m, raw σ_bin weight
# ============================================================

from evaluation.dihiggs import pair_from_4jets, find_gen_b_pairs_with_indices
from evaluation.luminosity import load_physics_config

_physics = load_physics_config()
LUMINOSITY_FB = _physics["luminosity_fb"]
SIGNAL_XSEC_PB = _physics["signal_xsec_pb"]
N_GEN_SIGNAL = _physics["n_gen_signal"]

import os, gc
import fastjet
import glob as glob_module
from itertools import combinations
from data_pipeline.make_particle_dataset import cluster_candidates
from data_pipeline.root_loading import (
    load_and_prepare_data,
    select_gen_b_quarks_from_higgs,
    apply_custom_cuts,
    one_hot_encode_l1_puppi,
)
from evaluation.jet_matching import get_purity_mask_cross_matched

# --- Fixed configuration (WP chosen later) ---
apply_pt_correction = True
dataset_used = config_part.get("training", {}).get("data", {}).get("use_dataset", "pf")
if dataset_used == "pf":
    collection_key = "l1extpf"
elif dataset_used == "puppi":
    collection_key = "l1extpuppi"
else:
    collection_key = "l1barrelextpf"
collection_name = config[collection_key]["collection_name"]
print(
    f"Model trained on: {dataset_used} → clustering {collection_key} ({collection_name})"
)
print(f"pT correction: {'enabled' if apply_pt_correction else 'disabled'}")


# ============================================================
# HELPER: cluster → extract features → ParT inference → scored jets
# ============================================================
def cluster_and_score(
    events,
    cfg,
    collection_key,
    model,
    device,
    config_part,
    n_constituents,
    apply_pt_correction=True,
):
    """
    Cluster L1 candidates, extract ParT features, run inference,
    apply pT regression correction, and return scored jets
    with event structure preserved.

    Returns: scored_jets (ak.Array), has_reg (bool)
    """
    # Cluster
    clustered_jets = cluster_candidates(events, cfg, collection_key, dist_param=0.4)
    sorted_indices = ak.argsort(clustered_jets.pt, axis=1, ascending=False)
    l1_clustered = clustered_jets[sorted_indices]
    del clustered_jets, sorted_indices
    matched_cands = l1_clustered.constituents
    const_pt_sort = ak.argsort(matched_cands.pt, axis=2, ascending=False)
    matched_cands = matched_cands[const_pt_sort]
    del const_pt_sort

    # Extract features
    j_pt = l1_clustered.pt[:, :, None]
    j_eta = l1_clustered.eta[:, :, None]
    j_phi = l1_clustered.phi[:, :, None]

    m_pt = matched_cands.vector.pt
    m_eta = matched_cands.vector.eta
    m_phi = matched_cands.vector.phi
    m_mass = matched_cands.vector.mass
    m_dxy = matched_cands.dxy
    m_z0 = matched_cands.z0
    m_charge = matched_cands.charge
    m_w = matched_cands.puppiWeight
    m_id = matched_cands.id

    log_pt_rel = np.log(np.maximum(m_pt, 1e-3) / np.maximum(j_pt, 1e-3))
    deta = m_eta - j_eta
    dphi = m_phi - j_phi
    dphi = (dphi + np.pi) % (2 * np.pi) - np.pi
    log_dr = np.log(np.maximum(np.sqrt(deta**2 + dphi**2), 1e-3))

    def pad_and_fill(arr, target=n_constituents):
        return ak.fill_none(ak.pad_none(arr, target, axis=2, clip=True), 0.0)

    feature_list = [
        pad_and_fill(m_mass),
        pad_and_fill(m_pt),
        pad_and_fill(m_eta),
        pad_and_fill(m_phi),
        pad_and_fill(m_dxy),
        pad_and_fill(m_z0),
        pad_and_fill(m_charge),
        pad_and_fill(log_pt_rel),
        pad_and_fill(deta),
        pad_and_fill(dphi),
        pad_and_fill(m_w),
        pad_and_fill(log_dr),
        pad_and_fill(m_id),
    ]
    del m_pt, m_eta, m_phi, m_mass, m_dxy, m_z0, m_charge, m_w, m_id
    del log_pt_rel, deta, dphi, log_dr, j_pt, j_eta, j_phi

    n_jets_per_event = ak.num(l1_clustered, axis=1)
    n_actual_constituents = ak.num(matched_cands, axis=2)
    n_actual_flat = ak.to_numpy(ak.flatten(n_actual_constituents, axis=1))
    del matched_cands, n_actual_constituents

    x_ini = np.stack(
        [ak.to_numpy(ak.flatten(f, axis=1)) for f in feature_list], axis=-1
    )
    del feature_list

    flat_ids = x_ini[..., -1]
    one_hot_ids = one_hot_encode_l1_puppi(flat_ids, n_classes=5)
    X_feat = np.concatenate([x_ini[..., :-1], one_hot_ids], axis=-1)
    del flat_ids, one_hot_ids

    particle_mask = np.zeros((X_feat.shape[0], n_constituents), dtype=bool)
    for i in range(X_feat.shape[0]):
        n_real = min(n_actual_flat[i], n_constituents)
        particle_mask[i, :n_real] = True
    del n_actual_flat

    # Jet 4-vectors from constituent sum
    const_vecs = vector.array(
        {
            "pt": x_ini[:, :, 1],
            "eta": x_ini[:, :, 2],
            "phi": x_ini[:, :, 3],
            "mass": x_ini[:, :, 0],
        }
    )
    del x_ini
    jet_4v = const_vecs.sum(axis=1)
    del const_vecs
    flat_jet_pt = np.array(jet_4v.pt)
    flat_jet_eta = np.array(jet_4v.eta)
    flat_jet_phi = np.array(jet_4v.phi)
    flat_jet_mass = np.array(jet_4v.mass)
    del jet_4v
    gc.collect()

    # Inference
    batch_size = config_part.get("training", {}).get("batch_size", 512)
    all_scores, all_reg = [], []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(X_feat), batch_size):
            end = min(start + batch_size, len(X_feat))
            xb = torch.tensor(X_feat[start:end], dtype=torch.float32).to(device)
            mb = torch.tensor(particle_mask[start:end], dtype=torch.bool).to(device)
            out = model(xb, particle_mask=mb)
            scores = (
                torch.nn.functional.sigmoid(out["classification"])
                .squeeze()
                .cpu()
                .numpy()
            )
            all_scores.append(scores)
            if "pt" in out:
                all_reg.append(out["pt"].squeeze().cpu().numpy())
    del X_feat, particle_mask
    gc.collect()

    all_scores = np.concatenate(all_scores)
    has_reg = len(all_reg) > 0
    if has_reg:
        all_reg = np.concatenate(all_reg)

    # pT correction
    corrected_pt = (
        flat_jet_pt * all_reg if (has_reg and apply_pt_correction) else flat_jet_pt
    )
    del all_reg

    corr_vecs = vector.array(
        {
            "pt": corrected_pt,
            "eta": flat_jet_eta,
            "phi": flat_jet_phi,
            "mass": flat_jet_mass * (corrected_pt / (flat_jet_pt + 1e-9)),
        }
    )
    del corrected_pt, flat_jet_pt, flat_jet_eta, flat_jet_phi, flat_jet_mass

    # Rebuild event structure
    n_jets_np = ak.to_numpy(n_jets_per_event)
    del l1_clustered, n_jets_per_event
    cumulative = np.concatenate([[0], np.cumsum(n_jets_np)])
    evt_pts, evt_etas, evt_phis, evt_masses, evt_scores = [], [], [], [], []
    for i in range(len(n_jets_np)):
        s, e = cumulative[i], cumulative[i + 1]
        evt_pts.append(corr_vecs.pt[s:e])
        evt_etas.append(corr_vecs.eta[s:e])
        evt_phis.append(corr_vecs.phi[s:e])
        evt_masses.append(corr_vecs.mass[s:e])
        evt_scores.append(all_scores[s:e])
    del corr_vecs, all_scores, cumulative, n_jets_np

    scored_jets = ak.zip(
        {
            "pt": ak.Array(evt_pts),
            "eta": ak.Array(evt_etas),
            "phi": ak.Array(evt_phis),
            "mass": ak.Array(evt_masses),
            "btag_score": ak.Array(evt_scores),
        }
    )
    del evt_pts, evt_etas, evt_phis, evt_masses, evt_scores
    scored_jets["vector"] = ak.zip(
        {
            "pt": scored_jets.pt,
            "eta": scored_jets.eta,
            "phi": scored_jets.phi,
            "mass": scored_jets.mass,
        },
        with_name="Momentum4D",
    )

    return scored_jets, has_reg


# ============================================================
# PHASE 1A — Signal: cluster, score, gen-match (NO threshold)
# ============================================================
SIGNAL_CHUNK_SIZE = 20000
max_signal_events = 200000

_sig_min_btag_list = []  # min btag of 4th jet, per event with ≥4 jets
_sig_lead_m_list = []
_sig_sub_m_list = []
_sig_hh_m_list = []
_sig_is_pure_list = []  # all 4 jets gen-matched
_sig_pair_ok_list = []  # pure event + D_HH chose correct pairing
_sig_pair_sw_list = []  # pure event + D_HH chose correct but h1/h2 swapped

pair_correct_total = 0
swap_correct_total = 0
pair_total = 0
has_reg = False

_gen_pt_cut = config["gen"]["pt_cut"]
_gen_eta_cut = config["gen"]["eta_cut"]
_matching_cone = config["matching_cone_size"]

_offset_sig = 0
_ev_rem_sig = max_signal_events
_cidx_sig = 0

print("=" * 60)
print("Phase 1A — Signal clustering (no b-tag threshold)")
print("=" * 60)
while _ev_rem_sig > 0:
    _cs = min(SIGNAL_CHUNK_SIZE, _ev_rem_sig)
    try:
        _ev = load_and_prepare_data(
            config["file_pattern"],
            config["tree_name"],
            [collection_name, "GenPart"],
            max_events=_cs,
            correct_pt=False,
            CONFIG=config,
            entry_start=_offset_sig,
        )
    except Exception as _e:
        print(f"  Signal chunk {_cidx_sig}: {_e}")
        break
    _n = len(_ev)
    if _n == 0:
        del _ev
        gc.collect()
        break
    _ev_rem_sig -= _n
    _offset_sig += _n
    _cidx_sig += 1

    _scored, has_reg = cluster_and_score(
        _ev,
        config,
        collection_key,
        model,
        device,
        config_part,
        all_constituents.shape[1],
        apply_pt_correction,
    )
    # Sort by b-tag score, take top-4 with NO threshold
    _srt = _scored[ak.argsort(_scored.btag_score, ascending=False)]
    _has4 = ak.num(_srt) >= 4
    _t4 = _srt[_has4][:, :4]
    del _scored, _srt

    # Gen-matching for purity
    _gb = select_gen_b_quarks_from_higgs(_ev)
    _gb = _gb[(_gb.pt > _gen_pt_cut) & (abs(_gb.eta) < _gen_eta_cut)]
    _gh4 = _gb[_has4]
    _gp_h4 = _ev.GenPart[_has4]  # keep for pairing eff before del _ev
    del _ev, _gb

    _drr = _t4[:, :, None].vector.deltaR(_gh4[:, None, :].vector)
    _ig = ak.argmin(_drr, axis=2)
    _mdr = ak.fill_none(ak.min(_drr, axis=2), np.inf)
    _drg = _gh4[:, :, None].vector.deltaR(_t4[:, None, :].vector)
    _ir = ak.argmin(_drg, axis=2)
    _back = _ir[_ig]
    _ri = ak.local_index(_t4, axis=1)
    _pjet = (ak.fill_none(_back, -1) == _ri) & (_mdr < _matching_cone)
    _ip = ak.to_numpy(ak.sum(_pjet, axis=1) == 4)
    del _drr, _ig, _mdr, _drg, _ir, _back, _ri, _pjet

    # Higgs masses + min btag — for ALL events with ≥4 jets
    _lead, _sub, _hh = pair_from_4jets(_t4)
    _min_b = ak.to_numpy(_t4.btag_score[:, 3])

    # Per-chunk pairing efficiency (pure events only)
    _pair_ok_chunk = np.zeros(int(_ip.sum()), dtype=bool)  # correct
    _pair_sw_chunk = np.zeros(int(_ip.sum()), dtype=bool)  # swapped
    if _ip.sum() > 0:
        _pj4 = _t4[_ip]
        _pgb = _gh4[_ip]
        _pgp = _gp_h4[_ip]

        # D_HH: find which permutation was chosen
        _j = [_pj4[:, i] for i in range(4)]
        _perms = [([0, 1], [2, 3]), ([0, 2], [1, 3]), ([0, 3], [1, 2])]
        _m1 = ak.concatenate(
            [(_j[a].vector + _j[b].vector).mass[:, None] for (a, b), _ in _perms],
            axis=1,
        )
        _m2 = ak.concatenate(
            [(_j[c].vector + _j[d].vector).mass[:, None] for _, (c, d) in _perms],
            axis=1,
        )
        _dhh = abs(_m1 - (125 / 120) * _m2) / np.sqrt(1 + (125 / 120) ** 2)
        _bst = ak.argmin(_dhh, axis=1)
        del _m1, _m2, _dhh

        # Reconstruct lead/sub vectors to get pT ordering
        _hA = [_j[a].vector + _j[b].vector for (a, b), _ in _perms]
        _hB = [_j[c].vector + _j[d].vector for _, (c, d) in _perms]
        _c0, _c1 = _bst == 0, _bst == 1
        _rv1 = ak.where(_c0, _hA[0], ak.where(_c1, _hA[1], _hA[2]))
        _rv2 = ak.where(_c0, _hB[0], ak.where(_c1, _hB[1], _hB[2]))
        _is_lead = _rv1.pt >= _rv2.pt
        del _rv1, _rv2, _hA, _hB

        # Corresponding gen-index pairs
        _gmpr = ak.argmin(
            _pj4[:, :, None].vector.deltaR(_pgb[:, None, :].vector), axis=2
        )
        _gp_idx = [
            (_gmpr[:, [0, 1]], _gmpr[:, [2, 3]]),
            (_gmpr[:, [0, 2]], _gmpr[:, [1, 3]]),
            (_gmpr[:, [0, 3]], _gmpr[:, [1, 2]]),
        ]
        _aA = ak.where(
            _c0[:, None],
            _gp_idx[0][0],
            ak.where(_c1[:, None], _gp_idx[1][0], _gp_idx[2][0]),
        )
        _aB = ak.where(
            _c0[:, None],
            _gp_idx[0][1],
            ak.where(_c1[:, None], _gp_idx[1][1], _gp_idx[2][1]),
        )
        del _gp_idx, _c0, _c1, _bst
        # Lead pair = the one with larger pT Higgs candidate
        _aL = ak.where(_is_lead[:, None], _aA, _aB)
        _aS = ak.where(_is_lead[:, None], _aB, _aA)
        del _aA, _aB, _is_lead

        # True gen pairing
        _, _, _h1i, _h2i = find_gen_b_pairs_with_indices(_gmpr, _pgb, _pgp)
        _h1s = ak.sort(_h1i, axis=1)
        _h2s = ak.sort(_h2i, axis=1)
        del _h1i, _h2i, _gmpr
        _aLs = ak.sort(_aL, axis=1)
        _aSs = ak.sort(_aS, axis=1)
        del _aL, _aS

        _correct = ak.all(_aLs == _h1s, axis=1) & ak.all(_aSs == _h2s, axis=1)
        _swapped = ak.all(_aLs == _h2s, axis=1) & ak.all(_aSs == _h1s, axis=1)
        _pair_ok_chunk = ak.to_numpy(_correct)
        _pair_sw_chunk = ak.to_numpy(_swapped)
        pair_correct_total += int(ak.sum(_correct))
        swap_correct_total += int(ak.sum(_swapped))
        pair_total += int(_ip.sum())
        del _pj4, _pgb, _pgp, _j, _perms, _h1s, _h2s, _aLs, _aSs, _correct, _swapped

    del _t4, _gp_h4, _gh4

    # Expand pairing results to full per-event arrays (False for impure events)
    _n4 = int(ak.sum(_has4))
    _pair_ok_full = np.zeros(_n4, dtype=bool)
    _pair_sw_full = np.zeros(_n4, dtype=bool)
    _pair_ok_full[_ip] = _pair_ok_chunk
    _pair_sw_full[_ip] = _pair_sw_chunk

    _sig_min_btag_list.append(_min_b)
    _sig_lead_m_list.append(ak.to_numpy(_lead.mass))
    _sig_sub_m_list.append(ak.to_numpy(_sub.mass))
    _sig_hh_m_list.append(ak.to_numpy(_hh.mass))
    _sig_is_pure_list.append(_ip)
    _sig_pair_ok_list.append(_pair_ok_full)
    _sig_pair_sw_list.append(_pair_sw_full)
    del (
        _lead,
        _sub,
        _hh,
        _min_b,
        _ip,
        _pair_ok_full,
        _pair_sw_full,
        _pair_ok_chunk,
        _pair_sw_chunk,
    )
    gc.collect()
    print(f"  Chunk {_cidx_sig}: {_n} events  |  total loaded: {_offset_sig}")

# Concatenate signal raw data
_sig_min_btag = np.concatenate(_sig_min_btag_list)
_sig_lead_m = np.concatenate(_sig_lead_m_list)
_sig_sub_m = np.concatenate(_sig_sub_m_list)
_sig_hh_m = np.concatenate(_sig_hh_m_list)
_sig_is_pure = np.concatenate(_sig_is_pure_list).astype(bool)
_sig_pair_ok = np.concatenate(_sig_pair_ok_list)
_sig_pair_sw = np.concatenate(_sig_pair_sw_list)
del _sig_min_btag_list, _sig_lead_m_list, _sig_sub_m_list, _sig_hh_m_list
del _sig_is_pure_list, _sig_pair_ok_list, _sig_pair_sw_list
gc.collect()

n_sig_loaded = (
    int(_sig_is_pure.sum()) if False else _offset_sig
)  # total HH events loaded
_pair_eff_all = pair_correct_total / max(pair_total, 1)
_pair_swap_all = swap_correct_total / max(pair_total, 1)
print(
    f"\nSignal: {_sig_is_pure.sum()} pure events  |  "
    f"pair_eff={_pair_eff_all:.2%}  swap={_pair_swap_all:.2%}  "
    f"total={_pair_eff_all+_pair_swap_all:.2%}"
)


# ============================================================
# PHASE 1B — QCD: cluster + score (NO threshold)
# ============================================================
QCD_CHUNK_SIZE = 17e4
qcd_config = config["QCD_background"]

_qcd_min_btag_list = []
_qcd_lead_m_list = []
_qcd_sub_m_list = []
_qcd_hh_m_list = []
_qcd_wt_list = []
_n_qcd_scanned = 0

print("\n" + "=" * 60)
print("Phase 1B — QCD clustering (no b-tag threshold)")
print("=" * 60)
for _bin_name, _bin_cfg in qcd_config.items():
    _qcd_file = _bin_cfg["file_pattern"]
    _qcd_tree = _bin_cfg["tree_name"]
    _max_ev = _bin_cfg.get("max_events") or config.get("max_events")
    _unlimited = _max_ev is None
    if _unlimited:
        _max_ev = float("inf")

    _qcd_cfg = dict(config)
    _qcd_cfg["file_pattern"] = _qcd_file
    _qcd_cfg["tree_name"] = _qcd_tree

    _off_bin, _ev_rem_bin = 0, _max_ev
    print(f"\n  {_bin_name}  (weight={_bin_cfg['weight']:.3e})")

    while _ev_rem_bin > 0:
        _cs = QCD_CHUNK_SIZE if _unlimited else min(QCD_CHUNK_SIZE, int(_ev_rem_bin))
        try:
            _qev = load_and_prepare_data(
                _qcd_file,
                _qcd_tree,
                [collection_name],
                max_events=_cs,
                correct_pt=False,
                CONFIG=_qcd_cfg,
                entry_start=_off_bin,
            )
        except Exception as _e:
            print(f"    chunk error: {_e}")
            break
        _nq = len(_qev)
        if _nq == 0:
            del _qev
            gc.collect()
            break
        _n_qcd_scanned += _nq
        _off_bin += _nq
        if not _unlimited:
            _ev_rem_bin -= _nq

        _qs, _ = cluster_and_score(
            _qev,
            _qcd_cfg,
            collection_key,
            model,
            device,
            config_part,
            all_constituents.shape[1],
            apply_pt_correction,
        )
        del _qev
        _qsrt = _qs[ak.argsort(_qs.btag_score, ascending=False)]
        del _qs
        _qhas4 = ak.num(_qsrt) >= 4
        _qt4 = _qsrt[_qhas4][:, :4]
        del _qsrt
        _n4q = int(ak.sum(_qhas4))
        del _qhas4
        if _n4q > 0:
            _ql, _qsb, _qhh = pair_from_4jets(_qt4)
            _qcd_min_btag_list.append(ak.to_numpy(_qt4.btag_score[:, 3]))
            _qcd_lead_m_list.append(ak.to_numpy(_ql.mass))
            _qcd_sub_m_list.append(ak.to_numpy(_qsb.mass))
            _qcd_hh_m_list.append(ak.to_numpy(_qhh.mass))
            _qcd_wt_list.append(np.full(_n4q, _bin_cfg["weight"], dtype=np.float64))
            del _ql, _qsb, _qhh
        del _qt4
        gc.collect()

    print(f"    scanned {_off_bin} events")

_qcd_min_btag = (
    np.concatenate(_qcd_min_btag_list) if _qcd_min_btag_list else np.array([])
)
_qcd_lead_m = np.concatenate(_qcd_lead_m_list) if _qcd_lead_m_list else np.array([])
_qcd_sub_m = np.concatenate(_qcd_sub_m_list) if _qcd_sub_m_list else np.array([])
_qcd_hh_m = np.concatenate(_qcd_hh_m_list) if _qcd_hh_m_list else np.array([])
_qcd_weights = np.concatenate(_qcd_wt_list) if _qcd_wt_list else np.array([])
_sigma_to_ngen = {v["weight"]: v["n_gen"] for v in qcd_config.values()}
del _qcd_min_btag_list, _qcd_lead_m_list, _qcd_sub_m_list, _qcd_hh_m_list, _qcd_wt_list
gc.collect()
print(
    f"\nQCD: {len(_qcd_min_btag):,} events with ≥4 jets  "
    f"(from {_n_qcd_scanned:,} scanned)"
)

# ============================================================
# Store raw pre-clustered data — used by the next cell
# ============================================================
dh_raw = {
    # Signal
    "sig_min_btag": _sig_min_btag,  # min btag of 4th jet, per event with ≥4 jets
    "sig_lead_m": _sig_lead_m,
    "sig_sub_m": _sig_sub_m,
    "sig_hh_m": _sig_hh_m,
    "sig_is_pure": _sig_is_pure,  # all 4 jets gen-matched
    "sig_pair_ok": _sig_pair_ok,  # True: pure + algorithm correct
    "sig_pair_swap": _sig_pair_sw,  # True: pure + algorithm correct (h1/h2 swapped)
    # QCD
    "qcd_min_btag": _qcd_min_btag,
    "qcd_lead_m": _qcd_lead_m,
    "qcd_sub_m": _qcd_sub_m,
    "qcd_hh_m": _qcd_hh_m,
    "qcd_weights": _qcd_weights,  # raw σ_bin (Convention C)
    # Metadata
    "sigma_to_ngen": _sigma_to_ngen,
    "collection_key": collection_key,
    "has_reg": has_reg,
    "n_sig_events_loaded": _offset_sig,
    "n_qcd_scanned": _n_qcd_scanned,
}

print(f"\n{'='*60}")
print(f"Phase 1 complete — raw data stored in `dh_raw`")
print(f"  Signal events with ≥4 jets : {len(_sig_min_btag):,}")
print(f"  Signal pure events          : {_sig_is_pure.sum():,}")
print(f"  QCD events with ≥4 jets    : {len(_qcd_min_btag):,}")
print(
    f"  Pairing eff (threshold=0)  : {_pair_eff_all:.2%}  "
    f"(+swap {_pair_swap_all:.2%} = {_pair_eff_all+_pair_swap_all:.2%})"
)
print(f"  pT regression              : {'applied' if has_reg else 'not available'}")
print(f"{'='*60}")
print("\nRun the next cell to sweep b-tag cuts and choose a working point.")

In [ ]:
# ============================================================
# PHASE 2: Significance sweep + WP selection
# ============================================================
# Sweeps 60 b-tag cut values using the pre-clustered dh_raw data.
# No re-inference needed — just numpy array filtering.
#
# After the table is printed, set WP_SELECTION to one of:
#   "tight" | "medium" | "loose" | "optimal"
# then re-run this cell to rebuild part_dihiggs_result.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import awkward as ak
from evaluation.dihiggs import R_hh_func, compute_significance_at_luminosity
from evaluation.luminosity import signal_weight, scale_qcd_weights_raw

R_HH_CUT = 55.0  # signal region — change here to propagate to all downstream cells
N_CUTS = 100
_btag_cuts = np.linspace(0.0, 0.99, N_CUTS)

# ── Sweep ────────────────────────────────────────────────────
_sig_sig = np.zeros(N_CUTS)
_S_arr = np.zeros(N_CUTS)
_B_arr = np.zeros(N_CUTS)

for _i, _cut in enumerate(_btag_cuts):
    _sm = (dh_raw["sig_min_btag"] >= _cut) & dh_raw["sig_is_pure"]
    _bm = dh_raw["qcd_min_btag"] >= _cut
    if _sm.sum() == 0 or _bm.sum() == 0:
        continue
    _r = compute_significance_at_luminosity(
        dh_raw["sig_lead_m"][_sm],
        dh_raw["sig_sub_m"][_sm],
        dh_raw["qcd_lead_m"][_bm],
        dh_raw["qcd_sub_m"][_bm],
        bkg_raw_weights=dh_raw["qcd_weights"][_bm],
        sigma_to_ngen=dh_raw["sigma_to_ngen"],
        n_gen_signal=N_GEN_SIGNAL,
        luminosity_fb=LUMINOSITY_FB,
        signal_xsec_pb=SIGNAL_XSEC_PB,
        region="circular",
        r_hh_cut=R_HH_CUT,
    )
    _sig_sig[_i] = _r["significance"]
    _S_arr[_i] = _r["S"]
    _B_arr[_i] = _r["B"]

_best_idx = int(np.argmax(_sig_sig))
_best_cut = float(_btag_cuts[_best_idx])

# ── Working-point options ─────────────────────────────────────
wp_options = {
    "tight": float(part_wps[0]),
    "medium": float(part_wps[1]),
    "loose": float(part_wps[2]),
    "optimal": _best_cut,
}

# Compute significance and counts at each WP for comparison
print(f"{'='*80}")
print(f"B-tag cut sweep  (R_HH < {R_HH_CUT} GeV,  L = {LUMINOSITY_FB:.0f} fb⁻¹)")
print(f"{'='*80}")
print(
    f"{'WP':<10} {'Cut':>7}  {'n_sig':>7}  {'n_qcd':>8}  "
    f"{'S':>9}  {'B':>10}  {'Signif.':>9}  {'PairEff':>9}"
)
print("-" * 80)
for _wp_name, _wp_cut in wp_options.items():
    _sm = (dh_raw["sig_min_btag"] >= _wp_cut) & dh_raw["sig_is_pure"]
    _bm = dh_raw["qcd_min_btag"] >= _wp_cut
    _n_s = _sm.sum()
    _n_b = _bm.sum()
    if _n_s == 0 or _n_b == 0:
        print(f"  {_wp_name:<8} {_wp_cut:>7.4f}  {'0':>7}  {'0':>8}  —")
        continue
    _r = compute_significance_at_luminosity(
        dh_raw["sig_lead_m"][_sm],
        dh_raw["sig_sub_m"][_sm],
        dh_raw["qcd_lead_m"][_bm],
        dh_raw["qcd_sub_m"][_bm],
        bkg_raw_weights=dh_raw["qcd_weights"][_bm],
        sigma_to_ngen=dh_raw["sigma_to_ngen"],
        n_gen_signal=N_GEN_SIGNAL,
        luminosity_fb=LUMINOSITY_FB,
        signal_xsec_pb=SIGNAL_XSEC_PB,
        region="circular",
        r_hh_cut=R_HH_CUT,
    )
    # Pairing efficiency at this WP (only for pure events above cut)
    _pure_at_wp = _sm  # already includes is_pure
    _eff_wp = dh_raw["sig_pair_ok"][_pure_at_wp].sum() / max(_pure_at_wp.sum(), 1)
    _star = " ★" if _wp_name == "optimal" else ""
    print(
        f"  {_wp_name:<8} {_wp_cut:>7.4f}  {_n_s:>7}  {_n_b:>8}  "
        f"{_r['S']:>9.1f}  {_r['B']:>10.2e}  {_r['significance']:>9.3f}  "
        f"{_eff_wp:>9.2%}{_star}"
    )
print(f"{'='*80}")
print(f"  Optimal cut: {_best_cut:.4f}  (max significance in sweep)")

# ── Quick significance vs cut plot ──────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(_btag_cuts, _sig_sig, color="mediumpurple", linewidth=2)
ax.axvline(
    _best_cut,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label=f"Optimal = {_best_cut:.3f}",
)
ax.scatter([_best_cut], [_sig_sig[_best_idx]], color="red", s=90, zorder=6)
for _wp_name, _wp_cut in wp_options.items():
    if _wp_name == "optimal":
        continue
    ax.axvline(
        _wp_cut, linestyle=":", linewidth=1.2, label=f"{_wp_name} = {_wp_cut:.3f}"
    )
ax.set_xlabel("ParT btag_score threshold  (all 4 jets ≥ cut)", fontsize=11)
ax.set_ylabel("Significance  $S/\\sqrt{S+B}$", fontsize=11)
ax.set_title(f"Significance vs b-tag cut  (R_HH < {R_HH_CUT} GeV)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ================================================================
# ← CHANGE THIS to switch working point, then re-run this cell
# ================================================================
WP_SELECTION = "loose"  # "tight" | "medium" | "loose" | "optimal"
# ================================================================

PART_BTAG_THRESHOLD = wp_options[WP_SELECTION]
print(f"\nSelected WP: {WP_SELECTION}  (threshold = {PART_BTAG_THRESHOLD:.4f})")

# ── Apply threshold → build part_dihiggs_result ─────────────
_sig_mask = (dh_raw["sig_min_btag"] >= PART_BTAG_THRESHOLD) & dh_raw["sig_is_pure"]
_qcd_mask = dh_raw["qcd_min_btag"] >= PART_BTAG_THRESHOLD
_tot_mask = (
    dh_raw["sig_min_btag"] >= PART_BTAG_THRESHOLD
)  # events with ≥4 jets above cut

n_signal = int(_sig_mask.sum())
n_qcd = int(_qcd_mask.sum())
n_total = int(_tot_mask.sum())

# Pairing efficiency at chosen WP
_pair_eff = dh_raw["sig_pair_ok"][_sig_mask].sum() / max(n_signal, 1)
_pair_swap = dh_raw["sig_pair_swap"][_sig_mask].sum() / max(n_signal, 1)


# Wrap numpy arrays as lightweight ak arrays (.mass) for downstream compatibility
def _mass_ak(arr):
    return ak.zip({"mass": ak.Array(arr.tolist())})


part_dihiggs_result = {
    "label": f"Trained ParT ({WP_SELECTION} = {PART_BTAG_THRESHOLD:.4f})",
    "n_total": n_total,
    "n_signal": n_signal,
    "n_qcd": n_qcd,
    "sig_lead": _mass_ak(dh_raw["sig_lead_m"][_sig_mask]),
    "sig_sub": _mass_ak(dh_raw["sig_sub_m"][_sig_mask]),
    "sig_hh": _mass_ak(dh_raw["sig_hh_m"][_sig_mask]),
    "qcd_lead": _mass_ak(dh_raw["qcd_lead_m"][_qcd_mask]),
    "qcd_sub": _mass_ak(dh_raw["qcd_sub_m"][_qcd_mask]),
    "qcd_hh": _mass_ak(dh_raw["qcd_hh_m"][_qcd_mask]),
    "qcd_weights": dh_raw["qcd_weights"][_qcd_mask],
    "sigma_to_ngen": dh_raw["sigma_to_ngen"],
    "collection_key": dh_raw["collection_key"],
    "wp": WP_SELECTION,
    "threshold": PART_BTAG_THRESHOLD,
    "has_regression": dh_raw["has_reg"],
    "pair_eff": float(_pair_eff),
    "eff_swapped": float(_pair_swap),
}

print(f"\n{'='*60}")
print(f"  part_dihiggs_result built at WP = {WP_SELECTION}")
print(f"  Signal events (pure, ≥4 jets above cut) : {n_signal}")
print(f"  QCD events (≥4 jets above cut)          : {n_qcd}")
print(
    f"  Pairing efficiency (at this WP)          : {_pair_eff:.2%}"
    f"  (+swap {_pair_swap:.2%} = {_pair_eff+_pair_swap:.2%})"
)
print(f"{'='*60}")
print("Run the next cell to plot mass distributions.")

In [ ]:
from evaluation.dihiggs import R_hh_func, compute_significance_at_luminosity

# ============================================================
# DI-HIGGS MASS DISTRIBUTION PLOTS + SAVE
# ============================================================
from matplotlib.patches import Ellipse

res = part_dihiggs_result
label = res["label"]
color = "purple"
qcd_weights = res.get("qcd_weights", np.ones(res["n_qcd"]))

sig_window_h = (90, 160)
sig_window_hh = (250, 550)

# --- 1. Signal vs QCD: 1×3 overlay (leading mH, subleading mH, mHH) ---
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
bins_h = np.linspace(0, 300, 61)
bins_hh = np.linspace(200, 800, 61)

# Leading Higgs
ax = axes[0]
if res["n_signal"] > 0:
    ax.hist(
        ak.to_numpy(res["sig_lead"].mass),
        bins=bins_h,
        histtype="stepfilled",
        alpha=0.3,
        color=color,
        label=f'Signal ({res["n_signal"]})',
        density=True,
    )
    ax.hist(
        ak.to_numpy(res["sig_lead"].mass),
        bins=bins_h,
        histtype="step",
        linewidth=2,
        color=color,
        density=True,
    )
if res["n_qcd"] > 0:
    ax.hist(
        ak.to_numpy(res["qcd_lead"].mass),
        bins=bins_h,
        histtype="step",
        linewidth=2,
        color="grey",
        linestyle="--",
        label=f'Simulated QCD ({res["n_qcd"]} events)',
        density=True,
        weights=qcd_weights,
    )
ax.axvline(125, color="green", linestyle=":", linewidth=1.5)
ax.axvspan(*sig_window_h, alpha=0.05, color="green")
ax.set_xlabel("Leading $m_H$ [GeV]")
ax.set_ylabel("Density")
ax.set_title(f"{label} — Leading Higgs")
ax.legend(fontsize=10)

# Subleading Higgs
ax = axes[1]
if res["n_signal"] > 0:
    ax.hist(
        ak.to_numpy(res["sig_sub"].mass),
        bins=bins_h,
        histtype="stepfilled",
        alpha=0.3,
        color=color,
        label="Signal",
        density=True,
    )
    ax.hist(
        ak.to_numpy(res["sig_sub"].mass),
        bins=bins_h,
        histtype="step",
        linewidth=2,
        color=color,
        density=True,
    )
if res["n_qcd"] > 0:
    ax.hist(
        ak.to_numpy(res["qcd_sub"].mass),
        bins=bins_h,
        histtype="step",
        linewidth=2,
        color="grey",
        linestyle="--",
        label="Simulated QCD",
        density=True,
        weights=qcd_weights,
    )
ax.axvline(125, color="green", linestyle=":", linewidth=1.5)
ax.axvspan(*sig_window_h, alpha=0.05, color="green")
ax.set_xlabel("Subleading $m_H$ [GeV]")
ax.set_ylabel("Density")
ax.set_title(f"{label} — Subleading Higgs")
ax.legend(fontsize=10)

# mHH
ax = axes[2]
if res["n_signal"] > 0:
    ax.hist(
        ak.to_numpy(res["sig_hh"].mass),
        bins=bins_hh,
        histtype="stepfilled",
        alpha=0.3,
        color=color,
        label="Signal",
        density=True,
    )
    ax.hist(
        ak.to_numpy(res["sig_hh"].mass),
        bins=bins_hh,
        histtype="step",
        linewidth=2,
        color=color,
        density=True,
    )
if res["n_qcd"] > 0:
    ax.hist(
        ak.to_numpy(res["qcd_hh"].mass),
        bins=bins_hh,
        histtype="step",
        linewidth=2,
        color="grey",
        linestyle="--",
        label="Simulated QCD",
        density=True,
        weights=qcd_weights,
    )
ax.axvspan(*sig_window_hh, alpha=0.05, color="green")
ax.set_xlabel("$m_{HH}$ [GeV]")
ax.set_ylabel("Density")
ax.set_title(f"{label} — $m_{{HH}}$")
ax.legend(fontsize=10)

reg_tag = (
    "pT-corrected"
    if res["has_regression"] and apply_pt_correction
    else "uncorrected pT"
)
fig.suptitle(f"Di-Higgs Reconstruction — {label} ({reg_tag})", fontsize=16, y=1.01)
plt.tight_layout()
save_fig(fig, f"dihiggs_mass_1d_{WP_SELECTION}")
plt.show()


# --- 2. 2D mH1 vs mH2 (Signal vs QCD) ---
bins_2d = np.linspace(0, 300, 61)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
for ax_idx, (category, lead_key, sub_key, n_events, w) in enumerate(
    [
        ("Signal", "sig_lead", "sig_sub", res["n_signal"], None),
        ("Simulated QCD", "qcd_lead", "qcd_sub", res["n_qcd"], qcd_weights),
    ]
):
    ax = axes[ax_idx]
    if n_events > 0:
        lead_mass = ak.to_numpy(res[lead_key].mass)
        sub_mass = ak.to_numpy(res[sub_key].mass)
        r_hh_vals = R_hh_func(lead_mass, sub_mass)
        sel = r_hh_vals < 550.0

        if w is not None:
            h = ax.hist2d(
                lead_mass[sel],
                sub_mass[sel],
                bins=[bins_2d, bins_2d],
                cmap="viridis",
                # weights=w[sel],
            )
        else:
            h = ax.hist2d(
                lead_mass[sel],
                sub_mass[sel],
                bins=[bins_2d, bins_2d],
                cmap="viridis",
            )
        ax.axvline(
            125, color="red", linestyle="--", linewidth=1.5, label="$m_H$ = 125 GeV"
        )
        ax.axhline(120, color="red", linestyle="--", linewidth=1.5)
        ellipse = Ellipse(
            xy=(125, 120),
            width=110,
            height=96,
            angle=0,
            edgecolor="yellow",
            facecolor="none",
            linestyle="--",
            linewidth=2,
            label="$R_{HH}$ = 55 GeV",
        )
        ax.add_patch(ellipse)
        n_sel = int(np.sum(sel))
        ax.set_title(
            f"{label} — {category}\n({n_sel} events inside $R_{{HH}}$ / {n_events} total)"
        )
        fig.colorbar(
            h[3], ax=ax, label="Weighted Events" if w is not None else "Events"
        )
    else:
        ax.text(
            0.5,
            0.5,
            f"No {category.lower()} events",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontsize=14,
        )
        ax.set_title(f"{label} — {category}")
    ax.set_xlabel("Leading Higgs Mass [GeV]")
    ax.set_ylabel("Subleading Higgs Mass [GeV]")
    ax.legend(loc="upper right", fontsize=10)

fig.suptitle(f"2D $m_{{H1}}$ vs $m_{{H2}}$ — {label} ({reg_tag})", fontsize=16, y=1.02)
plt.tight_layout()
save_fig(fig, f"dihiggs_mass_2d_{WP_SELECTION}")
plt.show()

# --- 3. Significance summary (with weighted QCD) ---
print(f"\n{'='*70}")
print(
    f"{'Tagger':<30} {'S':>6} {'B (wtd)':>10} {'S/√(S+B)':>10}  |  {'S (window)':>10} {'B (window)':>12} {'S/√(S+B)':>10}"
)
print(
    f"{'':30} {'(all)':>6} {'(all)':>10} {'(all)':>10}  |  {str(sig_window_hh):>10} {str(sig_window_hh):>12} {'(window)':>10}"
)
print("-" * 90)
if res["n_signal"] > 0 and res["n_qcd"] > 0:
    sig_m = ak.to_numpy(res["sig_hh"].mass)
    bkg_m = ak.to_numpy(res["qcd_hh"].mass)
    sigma_to_ngen = res.get("sigma_to_ngen", {})
    sig_mh1 = ak.to_numpy(res["sig_lead"].mass)
    sig_mh2 = ak.to_numpy(res["sig_sub"].mass)
    bkg_mh1 = ak.to_numpy(res["qcd_lead"].mass)
    bkg_mh2 = ak.to_numpy(res["qcd_sub"].mass)

    result_mhh = compute_significance_at_luminosity(
        sig_mh1,
        sig_mh2,
        bkg_mh1,
        bkg_mh2,
        bkg_raw_weights=qcd_weights,
        sigma_to_ngen=sigma_to_ngen,
        n_gen_signal=N_GEN_SIGNAL,
        luminosity_fb=LUMINOSITY_FB,
        signal_xsec_pb=SIGNAL_XSEC_PB,
        region="rectangular",
        rect_window=sig_window_hh,
    )
    S_win = result_mhh["S"]
    B_win = result_mhh["B"]
    signif_win = result_mhh["significance"]

    # Inclusive counts (no region cut)
    from evaluation.luminosity import (
        signal_weight as _sw,
        scale_qcd_weights_raw as _sqr,
    )

    S_all = float(
        np.sum(_sw(len(sig_mh1), LUMINOSITY_FB, SIGNAL_XSEC_PB, N_GEN_SIGNAL))
    )
    B_all = float(np.sum(_sqr(qcd_weights, sigma_to_ngen, LUMINOSITY_FB)))
    signif_all = S_all / np.sqrt(S_all + B_all) if (S_all + B_all) > 0 else 0

    print(
        f"{label:<30} {S_all:>6.0f} {B_all:>10.1e} {signif_all:>10.4f}  |  {S_win:>10.0f} {B_win:>12.1e} {signif_win:>10.4f}"
    )
    print(
        f"\n  Note: S and B are luminosity-scaled expected event counts at {LUMINOSITY_FB:.0f} fb⁻¹."
    )
    print(f"  Unweighted QCD events: {res['n_qcd']}")
else:
    print(f"{label:<30}  Insufficient events for significance")
print("=" * 90)
print(f"\nWorking point: {WP_SELECTION} (threshold={PART_BTAG_THRESHOLD:.4f})")
print(
    f"pT regression: {'applied' if res['has_regression'] and apply_pt_correction else 'not applied'}"
)
print(f"Collection: {res['collection_key']}")

print(f"\nAll di-Higgs plots saved to: {plot_dir}")

In [ ]:
from plotting.dihiggs_plots import plot_mass_1d, plot_mass_2d
from evaluation.dihiggs import compute_significance_at_luminosity
from evaluation.luminosity import signal_weight, scale_qcd_weights_raw, load_physics_config

# Fallback in case these are not already defined in your session
try:
    LUMINOSITY_FB
    SIGNAL_XSEC_PB
    N_GEN_SIGNAL
except NameError:
    _phys = load_physics_config("hh-bbbb-obj-config.json")
    LUMINOSITY_FB = _phys["luminosity_fb"]
    SIGNAL_XSEC_PB = _phys["signal_xsec_pb"]
    N_GEN_SIGNAL = _phys["n_gen_signal"]

res = part_dihiggs_result
label = res["label"]
qcd_weights_raw = res.get("qcd_weights", np.ones(res["n_qcd"]))
sigma_to_ngen = res.get("sigma_to_ngen", {})

sig_window_h = (90, 160)
sig_window_hh = (250, 550)

if res["n_signal"] > 0:
    sig_lead_m = ak.to_numpy(res["sig_lead"].mass)
    sig_sub_m = ak.to_numpy(res["sig_sub"].mass)
    sig_hh_m = ak.to_numpy(res["sig_hh"].mass)
else:
    sig_lead_m = np.array([])
    sig_sub_m = np.array([])
    sig_hh_m = np.array([])

if res["n_qcd"] > 0:
    bkg_lead_m = ak.to_numpy(res["qcd_lead"].mass)
    bkg_sub_m = ak.to_numpy(res["qcd_sub"].mass)
    bkg_hh_m = ak.to_numpy(res["qcd_hh"].mass)
else:
    bkg_lead_m = np.array([])
    bkg_sub_m = np.array([])
    bkg_hh_m = np.array([])

sig_weights_expected = signal_weight(
    len(sig_lead_m),
    luminosity_fb=LUMINOSITY_FB,
    signal_xsec_pb=SIGNAL_XSEC_PB,
    n_gen_signal=N_GEN_SIGNAL,
)
bkg_weights_expected = scale_qcd_weights_raw(
    qcd_weights_raw,
    sigma_to_ngen,
    luminosity_fb=LUMINOSITY_FB,
)

_reg_applied = bool(res.get("has_regression", False) and globals().get("apply_pt_correction", False))
_reg_tag = "pT-corrected" if _reg_applied else "uncorrected pT"

plot_label = f"{label} ({_reg_tag})"

# 1D mass plots
fig1 = plot_mass_1d(
    sig_lead_mass=sig_lead_m,
    sig_sub_mass=sig_sub_m,
    sig_hh_mass=sig_hh_m,
    qcd_lead_mass=bkg_lead_m,
    qcd_sub_mass=bkg_sub_m,
    qcd_hh_mass=bkg_hh_m,
    qcd_weights=bkg_weights_expected,
    sig_weights=sig_weights_expected,
    label=plot_label,
    sig_window_h=sig_window_h,
    sig_window_hh=sig_window_hh,
    density=True,
)
save_fig(fig1, f"dihiggs_mass_1d_{WP_SELECTION}")
plt.show()

# 2D mass-plane plots
fig2 = plot_mass_2d(
    sig_lead_mass=sig_lead_m,
    sig_sub_mass=sig_sub_m,
    qcd_lead_mass=bkg_lead_m,
    qcd_sub_mass=bkg_sub_m,
    qcd_weights=bkg_weights_expected,
    sig_weights=sig_weights_expected,
    label=plot_label,
    r_hh_cut=55.0,
    mh_centers=(125.0, 120.0),
)
save_fig(fig2, f"dihiggs_mass_2d_{WP_SELECTION}")
plt.show()

# Significance: rectangular and circular regions
print("\n" + "=" * 70)
print(
    f"{'Tagger':<30} {'S':>6} {'B (wtd)':>10} {'S/sqrt(S+B)':>12}  |  {'S (rect)':>10} {'B (rect)':>12} {'S/sqrt(S+B)':>12}"
)
print("-" * 100)

if len(sig_lead_m) > 0 and len(bkg_lead_m) > 0:
    sig_rect = compute_significance_at_luminosity(
        sig_lead_m,
        sig_sub_m,
        bkg_lead_m,
        bkg_sub_m,
        bkg_raw_weights=qcd_weights_raw,
        sigma_to_ngen=sigma_to_ngen,
        n_gen_signal=N_GEN_SIGNAL,
        luminosity_fb=LUMINOSITY_FB,
        signal_xsec_pb=SIGNAL_XSEC_PB,
        region="rectangular",
        rect_window=sig_window_hh,
    )

    sig_circ = compute_significance_at_luminosity(
        sig_lead_m,
        sig_sub_m,
        bkg_lead_m,
        bkg_sub_m,
        bkg_raw_weights=qcd_weights_raw,
        sigma_to_ngen=sigma_to_ngen,
        n_gen_signal=N_GEN_SIGNAL,
        luminosity_fb=LUMINOSITY_FB,
        signal_xsec_pb=SIGNAL_XSEC_PB,
        region="circular",
        r_hh_cut=55.0,
    )

    S_all = float(np.sum(sig_weights_expected))
    B_all = float(np.sum(bkg_weights_expected))
    signif_all = S_all / np.sqrt(S_all + B_all) if (S_all + B_all) > 0 else 0.0

    print(
        f"{label:<30} {S_all:>6.0f} {B_all:>10.1e} {signif_all:>12.4f}  |  {sig_rect['S']:>10.0f} {sig_rect['B']:>12.1e} {sig_rect['significance']:>12.4f}"
    )
    print(
        f"{'(R_HH<55 circular)':<30} {'':>6} {'':>10} {'':>12}  |  {sig_circ['S']:>10.0f} {sig_circ['B']:>12.1e} {sig_circ['significance']:>12.4f}"
    )
    print(f"\nUnweighted QCD events: {res['n_qcd']}")
else:
    print(f"{label:<30} Insufficient events for significance")

print("=" * 100)
print(f"\nWorking point: {WP_SELECTION} (threshold={PART_BTAG_THRESHOLD:.4f})")
print(f"pT regression: {'applied' if _reg_applied else 'not applied'}")
print(f"Collection: {res['collection_key']}")
print(f"All di-Higgs plots saved to: {plot_dir}")

In [ ]:
# ============================================================
# R_HH SIGNAL-REGION ANALYSIS FOR TRAINED ParT (SINGLE TAGGER)
# ============================================================
from matplotlib.patches import Ellipse

# Use trained ParT result object created in the cell above
res = part_dihiggs_result
label = res["label"]
color = "purple"
qcd_weights = res.get("qcd_weights", np.ones(res["n_qcd"]))
sig_wt = res.get("sig_weights")

# Define R_HH and a dedicated signal-region cut
# You can tighten/loosen this value without changing the rest of the code.
R_HH_CUT = 55.0

print("=" * 90)
from evaluation.luminosity import signal_weight, scale_qcd_weights_raw
from evaluation.dihiggs import compute_significance_at_luminosity

print(f"Trained ParT significance using R_HH signal region")
print(f"  R_HH = sqrt((m_H1 - 125)^2 + (m_H2 - 120)^2)")
print(f"  Signal region: R_HH < {R_HH_CUT:.1f} GeV")
print("=" * 90)
print(
    f"{'Tagger':<30} {'S (all)':>10} {'B (all,wtd)':>14} {'S/√(S+B)':>12}  |  "
    f"{'S (R_HH)':>10} {'B (R_HH,wtd)':>14} {'S/√(S+B)':>12}"
)
print("-" * 120)

if res["n_signal"] > 0 and res["n_qcd"] > 0:
    sig_lead_m = ak.to_numpy(res["sig_lead"].mass)
    sig_sub_m = ak.to_numpy(res["sig_sub"].mass)
    bkg_lead_m = ak.to_numpy(res["qcd_lead"].mass)
    bkg_sub_m = ak.to_numpy(res["qcd_sub"].mass)

    # Inclusive significance
    sigma_to_ngen = res.get("sigma_to_ngen", {})
    _sw = signal_weight(len(sig_lead_m), LUMINOSITY_FB, SIGNAL_XSEC_PB, N_GEN_SIGNAL)
    _bw = scale_qcd_weights_raw(qcd_weights, sigma_to_ngen, LUMINOSITY_FB)
    S_all = float(np.sum(_sw))
    B_all = float(np.sum(_bw))
    signif_all = S_all / np.sqrt(S_all + B_all) if (S_all + B_all) > 0 else 0.0

    # R_HH signal region significance
    sig_rhh = R_hh_func(sig_lead_m, sig_sub_m)
    bkg_rhh = R_hh_func(bkg_lead_m, bkg_sub_m)

    sig_sel = sig_rhh < R_HH_CUT
    bkg_sel = bkg_rhh < R_HH_CUT

    result_rhh = compute_significance_at_luminosity(
        sig_lead_m,
        sig_sub_m,
        bkg_lead_m,
        bkg_sub_m,
        bkg_raw_weights=qcd_weights,
        sigma_to_ngen=sigma_to_ngen,
        n_gen_signal=N_GEN_SIGNAL,
        luminosity_fb=LUMINOSITY_FB,
        signal_xsec_pb=SIGNAL_XSEC_PB,
        region="circular",
        r_hh_cut=R_HH_CUT,
    )
    S_rhh = result_rhh["S"]
    B_rhh = result_rhh["B"]
    signif_rhh = result_rhh["significance"]

    print(
        f"{label:<30} {S_all:>10.0f} {B_all:>14.1e} {signif_all:>12.4f}  |  "
        f"{S_rhh:>10.0f} {B_rhh:>14.1e} {signif_rhh:>12.4f}"
    )

    print(
        f"\n  Note: S and B are luminosity-scaled expected event counts at {LUMINOSITY_FB:.0f} fb⁻¹."
    )
    print(f"  Unweighted QCD events: {res['n_qcd']}")
else:
    print(f"{label:<30} Insufficient events for significance")

print("=" * 120)

# ------------------------------------------------------------
# 2) R_HH distributions: signal vs weighted QCD
# ------------------------------------------------------------
rhh_bins = np.linspace(0, 300, 61)
fig, ax = plt.subplots(figsize=(10, 7))

if res["n_signal"] > 0:
    sig_rhh = R_hh_func(
        ak.to_numpy(res["sig_lead"].mass), ak.to_numpy(res["sig_sub"].mass)
    )
    ax.hist(
        sig_rhh,
        bins=rhh_bins,
        histtype="stepfilled",
        alpha=0.30,
        color=color,
        label=f'Signal ({res["n_signal"]})',
    )
    ax.hist(sig_rhh, bins=rhh_bins, histtype="step", linewidth=2, color=color)

if res["n_qcd"] > 0:
    bkg_rhh = R_hh_func(
        ak.to_numpy(res["qcd_lead"].mass), ak.to_numpy(res["qcd_sub"].mass)
    )
    ax.hist(
        bkg_rhh,
        bins=rhh_bins,
        histtype="step",
        linewidth=2,
        color="grey",
        linestyle="--",
        label=f'QCD ({res["n_qcd"]} events, weighted)',
        weights=qcd_weights,
    )

ax.axvline(
    R_HH_CUT,
    color="red",
    linestyle=":",
    linewidth=2,
    label=f"$R_{{HH}}$ cut = {R_HH_CUT:.0f} GeV",
)
ax.axvspan(0, R_HH_CUT, alpha=0.06, color="red")
ax.set_xlabel("$R_{HH}$ [GeV]")
ax.set_ylabel("Events / 5 GeV (QCD weighted)")
ax.set_title(f"{label} — $R_{{HH}}$ distribution")
ax.set_yscale("log")
ax.grid(True, alpha=0.3, which="both")
ax.legend(fontsize=10)
plt.tight_layout()
save_fig(fig, f"rhh_distribution_{WP_SELECTION}")
plt.show()

# ------------------------------------------------------------
# 3) 2D mH1 vs mH2 with R_HH ellipse, signal + QCD panels
# ------------------------------------------------------------
bins_2d = np.linspace(0, 300, 61)
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax_idx, (category, lead_key, sub_key, n_events, w) in enumerate(
    [
        ("Signal", "sig_lead", "sig_sub", res["n_signal"], None),
        ("Simulated QCD", "qcd_lead", "qcd_sub", res["n_qcd"], qcd_weights),
    ]
):
    ax = axes[ax_idx]

    if n_events > 0:
        lead_mass = ak.to_numpy(res[lead_key].mass)
        sub_mass = ak.to_numpy(res[sub_key].mass)
        r_hh_vals = R_hh_func(lead_mass, sub_mass)
        sel = r_hh_vals < R_HH_CUT

        h = ax.hist2d(
            lead_mass,
            sub_mass,
            bins=[bins_2d, bins_2d],
            cmap="viridis",
            weights=w,
        )

        ax.axvline(
            125, color="red", linestyle="--", linewidth=1.5, label="$m_{H1}=125$ GeV"
        )
        ax.axhline(120, color="red", linestyle="--", linewidth=1.5)

        ellipse = Ellipse(
            xy=(125, 120),
            width=2 * R_HH_CUT,
            height=2 * R_HH_CUT,
            angle=0,
            edgecolor="yellow",
            facecolor="none",
            linestyle="--",
            linewidth=2,
            label=f"$R_{{HH}}$ = {R_HH_CUT:.0f} GeV",
        )
        ax.add_patch(ellipse)

        n_sel = int(np.sum(sel))
        if w is not None:
            w_sel = float(np.sum(w[sel]))
            w_tot = float(np.sum(w))
            ax.set_title(
                f"{label} — {category}\n"
                f"({n_sel}/{n_events} inside $R_{{HH}}$, weighted: {w_sel:.1e}/{w_tot:.1e})"
            )
        else:
            ax.set_title(
                f"{label} — {category}\n({n_sel}/{n_events} inside $R_{{HH}}$)"
            )

        fig.colorbar(
            h[3], ax=ax, label="Weighted Events" if w is not None else "Events"
        )
    else:
        ax.text(
            0.5,
            0.5,
            f"No {category.lower()} events",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontsize=14,
        )
        ax.set_title(f"{label} — {category}")

    ax.set_xlabel("Leading Higgs Mass [GeV]")
    ax.set_ylabel("Subleading Higgs Mass [GeV]")
    ax.legend(loc="upper right", fontsize=10)

reg_tag = (
    "pT-corrected"
    if res["has_regression"] and apply_pt_correction
    else "uncorrected pT"
)
fig.suptitle(
    f"2D $m_{{H1}}$ vs $m_{{H2}}$ — {label} ({reg_tag}), $R_{{HH}}<{R_HH_CUT:.0f}$",
    fontsize=16,
    y=1.02,
)
plt.tight_layout()
save_fig(fig, f"dihiggs_mass_2d_rhh_{WP_SELECTION}")
plt.show()

print(f"\nWorking point: {WP_SELECTION} (threshold={PART_BTAG_THRESHOLD:.4f})")
print(
    f"pT regression: {'applied' if res['has_regression'] and apply_pt_correction else 'not applied'}"
)
print(f"Collection: {res['collection_key']}")
print(f"All di-Higgs plots saved to: {plot_dir}")

In [ ]:
from evaluation.attention import (
    compute_pairwise_features,
    AttentionHook,
    forward_with_attention,
)

# ============================================================
# ATTENTION MAP VISUALIZATION & PAIRWISE FEATURE ANALYSIS
# ============================================================
import torch
import torch.nn as nn
from model.parT import to_rapidity, to_energy


# ============================================================
# 1. EXTRACT PAIRWISE FEATURES FOR VISUALIZATION
# ============================================================
# Modified forward that returns attention weights
print("=" * 60)
print("ATTENTION & PAIRWISE FEATURE ANALYSIS")
print("=" * 60)

# Select sample jets (signal and background)
n_samples = 5
signal_indices = np.where(all_labels == 1)[0][:n_samples]
background_indices = np.where(all_labels == 0)[0][:n_samples]
sample_indices = np.concatenate([signal_indices, background_indices])

# Get sample data
sample_x = all_constituents[sample_indices].to(device)
sample_mask = sample_x[:, :, 1] > 0  # pT > 0 means valid constituent

# Compute pairwise features
pairwise_feats, mask = compute_pairwise_features(sample_x.cpu(), sample_mask.cpu())

# Get attention maps
model.eval()
with torch.no_grad():
    attention_maps, u_ij = forward_with_attention(model, sample_x, sample_mask)

print(
    f"Analyzed {len(sample_indices)} jets ({n_samples} signal, {n_samples} background)"
)
print(f"Particle attention layers: {len(attention_maps['particle_attn'])}")
print(f"Class attention layers: {len(attention_maps['class_attn'])}")

# ============================================================
# 4. PLOT PAIRWISE FEATURE DISTRIBUTIONS
# ============================================================
# Use full dataset for distribution plots
print("\nComputing pairwise features on full validation set...")
n_jets_for_dist = min(1000, len(all_constituents))
dist_x = all_constituents[:n_jets_for_dist]
dist_mask = dist_x[:, :, 1] > 0
dist_labels = all_labels[:n_jets_for_dist]

pairwise_dist, _ = compute_pairwise_features(dist_x, dist_mask)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

feature_names_pairwise = [
    ("log_delta_R", r"$\log(\Delta R)$"),
    ("log_k_t", r"$\log(k_T)$"),
    ("log_z", r"$\log(z)$"),
    ("log_m_2", r"$\log(m^2)$"),
    ("d_dxy", r"$\Delta d_{xy}$"),
    ("d_z0", r"$\Delta z_0$"),
    ("q_ij", r"$q_i \cdot q_j$"),
]

for idx, (feat_key, feat_label) in enumerate(feature_names_pairwise):
    ax = axes[idx]

    feat = pairwise_dist[feat_key].numpy()
    mask_np = dist_mask.numpy()

    # Get upper triangular values (avoid double counting i,j and j,i)
    sig_vals = []
    bkg_vals = []

    for i in range(len(feat)):
        m = mask_np[i]
        n_valid = m.sum()
        if n_valid > 1:
            f = feat[i, :n_valid, :n_valid]
            upper_tri = f[np.triu_indices(n_valid, k=1)]
            if dist_labels[i] == 1:
                sig_vals.extend(upper_tri)
            else:
                bkg_vals.extend(upper_tri)

    sig_vals = np.array(sig_vals)
    bkg_vals = np.array(bkg_vals)

    # Filter out inf/nan
    sig_vals = sig_vals[np.isfinite(sig_vals)]
    bkg_vals = bkg_vals[np.isfinite(bkg_vals)]

    if len(sig_vals) > 0 and len(bkg_vals) > 0:
        range_min = min(np.percentile(sig_vals, 1), np.percentile(bkg_vals, 1))
        range_max = max(np.percentile(sig_vals, 99), np.percentile(bkg_vals, 99))

        ax.hist(
            sig_vals,
            bins=50,
            range=(range_min, range_max),
            histtype="step",
            label="Signal",
            color="blue",
            density=True,
        )
        ax.hist(
            bkg_vals,
            bins=50,
            range=(range_min, range_max),
            histtype="step",
            label="Background",
            color="red",
            density=True,
        )

    ax.set_xlabel(feat_label)
    ax.set_ylabel("Density")
    ax.set_title(f"Pairwise Feature: {feat_label}")
    ax.legend()

# Hide unused subplot
axes[-1].axis("off")

plt.tight_layout()
save_fig(fig, "pairwise_feature_distributions")
plt.show()


# ============================================================
# 5. ATTENTION MAP VISUALIZATION
# ============================================================
def plot_attention_maps(
    attention_maps, sample_idx, sample_mask, is_signal, layer_idx=0
):
    """Plot attention maps for a single jet."""
    n_heads = attention_maps["particle_attn"][layer_idx].shape[1]
    n_valid = sample_mask[sample_idx].sum().item()

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    attn = attention_maps["particle_attn"][layer_idx][sample_idx, :, :n_valid, :n_valid]

    for h in range(min(n_heads, 8)):
        ax = axes[h]
        attn_head = attn[h].numpy()

        im = ax.imshow(
            attn_head, cmap="viridis", aspect="auto", vmin=0, vmax=attn_head.max()
        )
        ax.set_xlabel("Key Particle")
        ax.set_ylabel("Query Particle")
        ax.set_title(f"Head {h+1}")
        plt.colorbar(im, ax=ax, fraction=0.046)

    label = "Signal (b-jet)" if is_signal else "Background"
    fig.suptitle(
        f"Particle Attention Maps - Layer {layer_idx+1} - {label}", fontsize=14
    )
    plt.tight_layout()
    return fig


# Plot attention maps for signal and background examples
print("\nPlotting attention maps...")

# Signal example
fig_sig = plot_attention_maps(
    attention_maps, 0, sample_mask.cpu(), is_signal=True, layer_idx=0
)
save_fig(fig_sig, "attention_map_signal_layer1")
plt.show()

# Background example
fig_bkg = plot_attention_maps(
    attention_maps, n_samples, sample_mask.cpu(), is_signal=False, layer_idx=0
)
save_fig(fig_bkg, "attention_map_background_layer1")
plt.show()

# ============================================================
# 6. CLASS ATTENTION VISUALIZATION
# ============================================================
fig, axes = plt.subplots(2, n_samples, figsize=(4 * n_samples, 8))

for i in range(n_samples):
    # Signal
    cls_attn = (
        attention_maps["class_attn"][-1][i, :, 0, 1:].mean(dim=0).numpy()
    )  # Average over heads
    n_valid = sample_mask[i].sum().item()

    ax = axes[0, i]
    ax.bar(range(n_valid), cls_attn[:n_valid])
    ax.set_xlabel("Constituent Index")
    ax.set_ylabel("Attention Weight")
    ax.set_title(f"Signal Jet {i+1}")

    # Background
    cls_attn = (
        attention_maps["class_attn"][-1][n_samples + i, :, 0, 1:].mean(dim=0).numpy()
    )
    n_valid = sample_mask[n_samples + i].sum().item()

    ax = axes[1, i]
    ax.bar(range(n_valid), cls_attn[:n_valid], color="red")
    ax.set_xlabel("Constituent Index")
    ax.set_ylabel("Attention Weight")
    ax.set_title(f"Background Jet {i+1}")

fig.suptitle(
    "Class Token Attention Weights (Which particles the classifier focuses on)",
    fontsize=14,
)
plt.tight_layout()
save_fig(fig, "class_attention_weights")
plt.show()

# ============================================================
# 7. AVERAGE ATTENTION PATTERNS
# ============================================================
# Compute average attention across more samples
n_avg = min(100, len(all_constituents))
avg_x = all_constituents[:n_avg].to(device)
avg_mask = avg_x[:, :, 1] > 0
avg_labels = all_labels[:n_avg]

with torch.no_grad():
    avg_attention_maps, _ = forward_with_attention(model, avg_x, avg_mask)

# Compute average attention as a function of Delta R
print("\nComputing attention vs Delta R correlation...")
pairwise_avg, _ = compute_pairwise_features(avg_x.cpu(), avg_mask.cpu())

delta_r_vals = []
attn_vals = []
labels_vals = []

for i in range(n_avg):
    n_valid = avg_mask[i].sum().item()
    if n_valid > 1:
        dr = pairwise_avg["delta_R"][i, :n_valid, :n_valid].numpy()
        attn = (
            avg_attention_maps["particle_attn"][0][i, :, :n_valid, :n_valid]
            .mean(dim=0)
            .numpy()
        )

        for j in range(n_valid):
            for k in range(j + 1, n_valid):
                delta_r_vals.append(dr[j, k])
                attn_vals.append(attn[j, k])
                labels_vals.append(avg_labels[i])

delta_r_vals = np.array(delta_r_vals)
attn_vals = np.array(attn_vals)
labels_vals = np.array(labels_vals)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hexbin plot
ax = axes[0]
hb = ax.hexbin(
    delta_r_vals,
    attn_vals,
    gridsize=50,
    cmap="viridis",
    extent=[0, 2, 0, np.percentile(attn_vals, 99)],
    mincnt=1,
)
ax.set_xlabel(r"$\Delta R$")
ax.set_ylabel("Attention Weight")
ax.set_title("Attention Weight vs Angular Distance")
plt.colorbar(hb, ax=ax, label="Count")

# Separate by signal/background
ax = axes[1]
sig_mask_v = labels_vals == 1
dr_bins = np.linspace(0, 2, 20)
sig_attn_means = []
bkg_attn_means = []

for j in range(len(dr_bins) - 1):
    bin_mask = (delta_r_vals >= dr_bins[j]) & (delta_r_vals < dr_bins[j + 1])
    sig_attn_means.append(
        np.mean(attn_vals[bin_mask & sig_mask_v])
        if (bin_mask & sig_mask_v).sum() > 0
        else np.nan
    )
    bkg_attn_means.append(
        np.mean(attn_vals[bin_mask & ~sig_mask_v])
        if (bin_mask & ~sig_mask_v).sum() > 0
        else np.nan
    )

dr_centers = (dr_bins[:-1] + dr_bins[1:]) / 2
ax.plot(dr_centers, sig_attn_means, "b-o", label="Signal", markersize=5)
ax.plot(dr_centers, bkg_attn_means, "r-o", label="Background", markersize=5)
ax.set_xlabel(r"$\Delta R$")
ax.set_ylabel("Mean Attention Weight")
ax.set_title("Attention vs Distance by Jet Type")
ax.legend()

plt.tight_layout()
save_fig(fig, "attention_vs_delta_r")
plt.show()

print(f"\n{'='*60}")
print("Attention and pairwise feature analysis complete!")
print(f"{'='*60}")

In [ ]:
# ============================================================
# CLASS ATTENTION — COLORED BY PARTICLE TYPE (NEW)
# ============================================================
from matplotlib.patches import Patch

if n_features > 12:
    fig, axes = plt.subplots(2, n_samples, figsize=(4 * n_samples, 8))

    sample_particle_types = np.argmax(
        all_constituents[sample_indices, :, 12:17].numpy(), axis=-1
    )  # (2*n_samples, 128)

    for i in range(n_samples):
        # Signal jet
        cls_attn = attention_maps["class_attn"][-1][i, :, 0, 1:].mean(dim=0).numpy()
        n_valid = sample_mask[i].sum().item()
        ptypes = sample_particle_types[i, :n_valid]
        colors = [PARTICLE_TYPE_COLORS[p] for p in ptypes]

        ax = axes[0, i]
        ax.bar(range(n_valid), cls_attn[:n_valid], color=colors)
        ax.set_xlabel("Constituent Index")
        ax.set_ylabel("Attention Weight")
        ax.set_title(f"Signal Jet {i+1}")

        # Background jet
        cls_attn = attention_maps["class_attn"][-1][n_samples + i, :, 0, 1:].mean(dim=0).numpy()
        n_valid = sample_mask[n_samples + i].sum().item()
        ptypes = sample_particle_types[n_samples + i, :n_valid]
        colors = [PARTICLE_TYPE_COLORS[p] for p in ptypes]

        ax = axes[1, i]
        ax.bar(range(n_valid), cls_attn[:n_valid], color=colors)
        ax.set_xlabel("Constituent Index")
        ax.set_ylabel("Attention Weight")
        ax.set_title(f"Background Jet {i+1}")

    # Add legend
    legend_elements = [Patch(facecolor=PARTICLE_TYPE_COLORS[p], label=PARTICLE_TYPE_NAMES[p])
                       for p in range(N_PARTICLE_TYPES)]
    fig.legend(handles=legend_elements, loc="upper center", ncol=5, fontsize=10,
               bbox_to_anchor=(0.5, 1.02))

    fig.suptitle("Class Token Attention — Colored by Particle Type", fontsize=14, y=1.05)
    plt.tight_layout()
    save_fig(fig, "class_attention_by_particle_type")
    plt.show()


In [ ]:
# ============================================================
# MEAN CLASS ATTENTION PER PARTICLE TYPE (AGGREGATED, NEW)
# ============================================================
if n_features > 12:
    # Use the larger avg_attention_maps computed earlier (n_avg jets)
    avg_cls_attn = avg_attention_maps["class_attn"][-1][:, :, 0, 1:].mean(dim=1).numpy()
    # avg_cls_attn shape: (n_avg, 128)

    avg_particle_types = np.argmax(
        all_constituents[:n_avg, :, 12:17].numpy(), axis=-1
    )  # (n_avg, 128)
    avg_mask_np_typed = (all_constituents[:n_avg, :, 1] > 0).numpy()
    avg_particle_types[~avg_mask_np_typed] = -1

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    for label_val, label_name, ax in [(1, "Signal", ax1), (0, "Background", ax2)]:
        jet_mask = avg_labels[:n_avg] == label_val
        mean_attn_per_type = []
        std_attn_per_type = []

        for pid in range(N_PARTICLE_TYPES):
            attn_vals = []
            for j in np.where(jet_mask)[0]:
                n_valid = int(avg_mask_np_typed[j].sum())
                cls_a = avg_cls_attn[j, :n_valid]
                ptypes_j = avg_particle_types[j, :n_valid]
                type_attn = cls_a[ptypes_j == pid]
                attn_vals.extend(type_attn)

            attn_vals = np.array(attn_vals) if attn_vals else np.array([0.0])
            mean_attn_per_type.append(attn_vals.mean())
            std_attn_per_type.append(attn_vals.std() / np.sqrt(len(attn_vals)))

        ax.bar(range(N_PARTICLE_TYPES), mean_attn_per_type,
               yerr=std_attn_per_type, color=PARTICLE_TYPE_COLORS,
               alpha=0.8, capsize=4)
        ax.set_xticks(range(N_PARTICLE_TYPES))
        ax.set_xticklabels(PARTICLE_TYPE_NAMES, rotation=30, ha="right")
        ax.set_ylabel("Mean Class Attention Weight")
        ax.set_title(f"{label_name} Jets")

    plt.suptitle("Mean Class Token Attention by Particle Type", fontsize=14)
    plt.tight_layout()
    save_fig(fig, "mean_class_attention_by_particle_type")
    plt.show()


In [ ]:
# ============================================================
# PARTICLE-TYPE ATTENTION AFFINITY MATRIX (NEW)
# ============================================================
if n_features > 12:
    # Compute 5x5 affinity matrices: how much does type_i attend to type_j?
    layer0_attn = avg_attention_maps["particle_attn"][0]  # (n_avg, heads, N, N)
    head_avg_attn = layer0_attn.mean(dim=1).numpy()  # (n_avg, N, N)

    # Reuse avg_particle_types and avg_mask_np_typed from previous cell
    affinity_sig = np.zeros((N_PARTICLE_TYPES, N_PARTICLE_TYPES))
    affinity_bkg = np.zeros((N_PARTICLE_TYPES, N_PARTICLE_TYPES))
    count_sig = np.zeros((N_PARTICLE_TYPES, N_PARTICLE_TYPES))
    count_bkg = np.zeros((N_PARTICLE_TYPES, N_PARTICLE_TYPES))

    for j in range(n_avg):
        n_valid = int(avg_mask_np_typed[j].sum())
        if n_valid < 2:
            continue
        attn_j = head_avg_attn[j, :n_valid, :n_valid]
        ptypes_j = avg_particle_types[j, :n_valid]

        for qi in range(n_valid):
            for ki in range(n_valid):
                if qi == ki:
                    continue
                q_type = ptypes_j[qi]
                k_type = ptypes_j[ki]
                if q_type < 0 or k_type < 0:
                    continue
                if avg_labels[j] == 1:
                    affinity_sig[q_type, k_type] += attn_j[qi, ki]
                    count_sig[q_type, k_type] += 1
                else:
                    affinity_bkg[q_type, k_type] += attn_j[qi, ki]
                    count_bkg[q_type, k_type] += 1

    # Normalize
    affinity_sig_norm = np.divide(affinity_sig, count_sig,
                                   out=np.zeros_like(affinity_sig), where=count_sig > 0)
    affinity_bkg_norm = np.divide(affinity_bkg, count_bkg,
                                   out=np.zeros_like(affinity_bkg), where=count_bkg > 0)
    affinity_diff = affinity_sig_norm - affinity_bkg_norm

    fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
    short_names = ["ChHad", "NeuHad", "Photon", "Electron", "Muon"]

    for ax, data, title, cmap in [
        (axes[0], affinity_sig_norm, "Signal (b-jets)", "viridis"),
        (axes[1], affinity_bkg_norm, "Background", "viridis"),
        (axes[2], affinity_diff, "Signal - Background", "RdBu_r"),
    ]:
        vmax = np.abs(data).max() if "-" in title else data.max()
        vmin = -vmax if "-" in title else 0
        im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_xticks(range(N_PARTICLE_TYPES))
        ax.set_yticks(range(N_PARTICLE_TYPES))
        ax.set_xticklabels(short_names, rotation=45, ha="right", fontsize=9)
        ax.set_yticklabels(short_names, fontsize=9)
        ax.set_xlabel("Key (attended to)")
        ax.set_ylabel("Query (attending)")
        ax.set_title(title)
        plt.colorbar(im, ax=ax, fraction=0.046)

        # Annotate cells
        for ii in range(N_PARTICLE_TYPES):
            for jj in range(N_PARTICLE_TYPES):
                ax.text(jj, ii, f"{data[ii, jj]:.4f}", ha="center", va="center",
                       fontsize=7, color="white" if abs(data[ii, jj]) > vmax * 0.5 else "black")

    plt.suptitle("Particle-Type Attention Affinity Matrix (Layer 0, Head-Averaged)", fontsize=14)
    plt.tight_layout()
    save_fig(fig, "particle_type_attention_affinity")
    plt.show()


In [ ]:
# ============================================================
# ATTENTION VS DELTA R — STRATIFIED BY PARTICLE TYPE (NEW)
# ============================================================
if n_features > 12:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Collect attention vs DeltaR per query particle type
    type_dr_attn = {pid: ([], []) for pid in range(N_PARTICLE_TYPES)}

    for i in range(n_avg):
        n_valid = int(avg_mask_np_typed[i].sum())
        if n_valid < 2:
            continue
        dr = pairwise_avg["delta_R"][i, :n_valid, :n_valid].numpy()
        attn = avg_attention_maps["particle_attn"][0][i, :, :n_valid, :n_valid].mean(dim=0).numpy()
        ptypes_i = avg_particle_types[i, :n_valid]

        for q in range(n_valid):
            q_type = ptypes_i[q]
            if q_type < 0:
                continue
            for k in range(n_valid):
                if q == k:
                    continue
                type_dr_attn[q_type][0].append(dr[q, k])
                type_dr_attn[q_type][1].append(attn[q, k])

    dr_bins = np.linspace(0, 2, 20)
    dr_centers = (dr_bins[:-1] + dr_bins[1:]) / 2

    for pid in range(N_PARTICLE_TYPES):
        dr_arr = np.array(type_dr_attn[pid][0])
        attn_arr = np.array(type_dr_attn[pid][1])
        if len(dr_arr) == 0:
            continue

        means = []
        for b in range(len(dr_bins) - 1):
            bin_mask = (dr_arr >= dr_bins[b]) & (dr_arr < dr_bins[b + 1])
            means.append(attn_arr[bin_mask].mean() if bin_mask.sum() > 0 else np.nan)

        axes[0].plot(dr_centers, means, "-o", color=PARTICLE_TYPE_COLORS[pid],
                    label=PARTICLE_TYPE_NAMES[pid], markersize=4, linewidth=1.5)

    axes[0].set_xlabel(r"$\Delta R$")
    axes[0].set_ylabel("Mean Attention Weight")
    axes[0].set_title("Attention vs Angular Distance by Query Particle Type")
    axes[0].legend(fontsize=9)

    # Right panel: signal jets only
    type_dr_attn_sig = {pid: ([], []) for pid in range(N_PARTICLE_TYPES)}

    for i in range(n_avg):
        if avg_labels[i] != 1:
            continue
        n_valid = int(avg_mask_np_typed[i].sum())
        if n_valid < 2:
            continue
        dr = pairwise_avg["delta_R"][i, :n_valid, :n_valid].numpy()
        attn = avg_attention_maps["particle_attn"][0][i, :, :n_valid, :n_valid].mean(dim=0).numpy()
        ptypes_i = avg_particle_types[i, :n_valid]

        for q in range(n_valid):
            q_type = ptypes_i[q]
            if q_type < 0:
                continue
            for k in range(n_valid):
                if q == k:
                    continue
                type_dr_attn_sig[q_type][0].append(dr[q, k])
                type_dr_attn_sig[q_type][1].append(attn[q, k])

    for pid in range(N_PARTICLE_TYPES):
        dr_arr = np.array(type_dr_attn_sig[pid][0])
        attn_arr = np.array(type_dr_attn_sig[pid][1])
        if len(dr_arr) == 0:
            continue

        means = []
        for b in range(len(dr_bins) - 1):
            bin_mask = (dr_arr >= dr_bins[b]) & (dr_arr < dr_bins[b + 1])
            means.append(attn_arr[bin_mask].mean() if bin_mask.sum() > 0 else np.nan)

        axes[1].plot(dr_centers, means, "-o", color=PARTICLE_TYPE_COLORS[pid],
                    label=PARTICLE_TYPE_NAMES[pid], markersize=4, linewidth=1.5)

    axes[1].set_xlabel(r"$\Delta R$")
    axes[1].set_ylabel("Mean Attention Weight")
    axes[1].set_title("Attention vs $\\Delta R$ — Signal Jets Only, by Query Type")
    axes[1].legend(fontsize=9)

    plt.tight_layout()
    save_fig(fig, "attention_vs_delta_r_by_particle_type")
    plt.show()


In [ ]:
from evaluation.attention import forward_with_activations, compute_separability

# ============================================================
# LAYER ACTIVATION VISUALIZATION
# ============================================================
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import torch.nn.functional as F


# ============================================================
# 1. FORWARD WITH LAYER ACTIVATIONS
# ============================================================
print("=" * 60)
print("LAYER ACTIVATION ANALYSIS")
print("=" * 60)

# Use a subset of validation data for visualization
n_vis = min(500, len(all_constituents))
vis_x = all_constituents[:n_vis].to(device)
vis_mask = vis_x[:, :, 1] > 0
vis_labels = all_labels[:n_vis]

print(f"Analyzing activations for {n_vis} jets...")
print(f"  Signal jets: {(vis_labels == 1).sum()}")
print(f"  Background jets: {(vis_labels == 0).sum()}")

with torch.no_grad():
    activations = forward_with_activations(model, vis_x, vis_mask)

# ============================================================
# 3. ACTIVATION MAGNITUDE DISTRIBUTIONS
# ============================================================
print("\nPlotting activation magnitude distributions...")

layer_names = (
    ["embedding"]
    + [f"particle_attn_{i+1}" for i in range(len(activations["particle_attn_layers"]))]
    + ["pre_cls"]
    + [f"cls_attn_{i+1}" for i in range(len(activations["cls_attn_layers"]))]
    + ["final_cls_token"]
)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

# Get activations for each layer
layer_activations = [activations["embedding"]]
layer_activations.extend(activations["particle_attn_layers"])
layer_activations.append(activations["pre_cls"])
layer_activations.extend(activations["cls_attn_layers"])
layer_activations.append(activations["final_cls_token"].unsqueeze(1))

for idx, (name, act) in enumerate(zip(layer_names, layer_activations)):
    if idx >= len(axes):
        break

    ax = axes[idx]

    # Flatten activations
    if len(act.shape) == 3:
        # For sequence activations, average over sequence dim
        act_flat = act.mean(dim=1).numpy()
    else:
        act_flat = act.numpy()

    # Plot distribution of activation magnitudes
    sig_act = act_flat[vis_labels == 1].flatten()
    bkg_act = act_flat[vis_labels == 0].flatten()

    range_min = min(np.percentile(sig_act, 1), np.percentile(bkg_act, 1))
    range_max = max(np.percentile(sig_act, 99), np.percentile(bkg_act, 99))

    ax.hist(
        sig_act,
        bins=50,
        range=(range_min, range_max),
        histtype="step",
        label="Signal",
        color="blue",
        density=True,
        alpha=0.8,
    )
    ax.hist(
        bkg_act,
        bins=50,
        range=(range_min, range_max),
        histtype="step",
        label="Background",
        color="red",
        density=True,
        alpha=0.8,
    )

    ax.set_xlabel("Activation Value")
    ax.set_ylabel("Density")
    ax.set_title(f"{name}\n(dim={act_flat.shape[-1]})")
    ax.legend(fontsize=8)

# Hide unused subplots
for idx in range(len(layer_names), len(axes)):
    axes[idx].axis("off")

plt.suptitle("Activation Magnitude Distributions Across Layers", fontsize=14)
plt.tight_layout()
save_fig(fig, "activation_magnitude_distributions")
plt.show()

# ============================================================
# 4. t-SNE VISUALIZATION OF LAYER REPRESENTATIONS
# ============================================================
print("\nComputing t-SNE projections...")

# Select key layers for t-SNE
tsne_layers = {
    "Input (processed)": activations["input_processed"].mean(dim=1).numpy(),
    "After Embedding": activations["embedding"].mean(dim=1).numpy(),
    f"After ParticleAttn-{len(activations['particle_attn_layers'])}": activations[
        "particle_attn_layers"
    ][-1]
    .mean(dim=1)
    .numpy(),
    "Final CLS Token": activations["final_cls_token"].numpy(),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (layer_name, layer_act) in enumerate(tsne_layers.items()):
    ax = axes[idx]

    # Apply PCA first to reduce dimensionality if needed
    if layer_act.shape[1] > 50:
        pca = PCA(n_components=50, random_state=42)
        layer_act_reduced = pca.fit_transform(layer_act)
    else:
        layer_act_reduced = layer_act

    # t-SNE
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, n_vis // 4))
    tsne_result = tsne.fit_transform(layer_act_reduced)

    # Plot
    ax.scatter(
        tsne_result[vis_labels == 0, 0],
        tsne_result[vis_labels == 0, 1],
        c="red",
        alpha=0.5,
        s=10,
        label="Background",
    )
    ax.scatter(
        tsne_result[vis_labels == 1, 0],
        tsne_result[vis_labels == 1, 1],
        c="blue",
        alpha=0.5,
        s=10,
        label="Signal",
    )

    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.set_title(layer_name)
    ax.legend()

plt.suptitle(
    "t-SNE Visualization: How Representations Evolve Through Layers", fontsize=14
)
plt.tight_layout()
save_fig(fig, "tsne_layer_evolution")
plt.show()

# ============================================================
# 5. NEURON ACTIVATION PATTERNS
# ============================================================
print("\nAnalyzing neuron activation patterns...")

# Focus on final CLS token activations
final_act = activations["final_cls_token"].numpy()
embed_dim = final_act.shape[1]

# Compute mean activation per neuron for signal vs background
sig_neuron_mean = final_act[vis_labels == 1].mean(axis=0)
bkg_neuron_mean = final_act[vis_labels == 0].mean(axis=0)
neuron_diff = sig_neuron_mean - bkg_neuron_mean

# Sort by difference
sorted_idx = np.argsort(neuron_diff)[::-1]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top discriminative neurons
ax = axes[0, 0]
top_n = 20
ax.barh(range(top_n), neuron_diff[sorted_idx[:top_n]], color="blue", alpha=0.7)
ax.set_yticks(range(top_n))
ax.set_yticklabels([f"Neuron {i}" for i in sorted_idx[:top_n]])
ax.set_xlabel("Mean Activation Difference (Signal - Background)")
ax.set_title(f"Top {top_n} Neurons Favoring Signal")
ax.invert_yaxis()

ax = axes[0, 1]
ax.barh(range(top_n), neuron_diff[sorted_idx[-top_n:]], color="red", alpha=0.7)
ax.set_yticks(range(top_n))
ax.set_yticklabels([f"Neuron {i}" for i in sorted_idx[-top_n:]])
ax.set_xlabel("Mean Activation Difference (Signal - Background)")
ax.set_title(f"Top {top_n} Neurons Favoring Background")
ax.invert_yaxis()

# Activation heatmap for most discriminative neurons
ax = axes[1, 0]
n_show = min(50, embed_dim)
top_neurons = sorted_idx[: n_show // 2].tolist() + sorted_idx[-n_show // 2 :].tolist()

# Sample jets for heatmap
n_jets_hm = 50
sig_jets = np.where(vis_labels == 1)[0][: n_jets_hm // 2]
bkg_jets = np.where(vis_labels == 0)[0][: n_jets_hm // 2]
jet_order = np.concatenate([sig_jets, bkg_jets])

heatmap_data = final_act[jet_order][:, top_neurons]
im = ax.imshow(
    heatmap_data.T,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-np.percentile(np.abs(heatmap_data), 95),
    vmax=np.percentile(np.abs(heatmap_data), 95),
)
ax.axvline(len(sig_jets) - 0.5, color="black", linestyle="--", linewidth=2)
ax.set_xlabel("Jet Index (Signal | Background)")
ax.set_ylabel("Neuron Index")
ax.set_title("Activation Heatmap (Most Discriminative Neurons)")
plt.colorbar(im, ax=ax, label="Activation")

# Correlation between neuron activations
ax = axes[1, 1]
top_act = final_act[:, sorted_idx[:30]]
corr = np.corrcoef(top_act.T)
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xlabel("Neuron Index")
ax.set_ylabel("Neuron Index")
ax.set_title("Correlation Between Top 30 Discriminative Neurons")
plt.colorbar(im, ax=ax, label="Correlation")

plt.tight_layout()
save_fig(fig, "neuron_activation_patterns")
plt.show()

# ============================================================
# 6. LAYER-WISE SEPARABILITY ANALYSIS
# ============================================================
print("\nComputing layer-wise class separability...")


separability_stats = []
layer_order = (
    ["input_processed", "embedding"]
    + [f"particle_attn_{i}" for i in range(len(activations["particle_attn_layers"]))]
    + ["pre_cls"]
    + [f"cls_attn_{i}" for i in range(len(activations["cls_attn_layers"]))]
    + ["final_cls_token"]
)

layer_acts = [
    activations["input_processed"].mean(dim=1).numpy(),
    activations["embedding"].mean(dim=1).numpy(),
]
for act in activations["particle_attn_layers"]:
    layer_acts.append(act.mean(dim=1).numpy())
layer_acts.append(activations["pre_cls"].mean(dim=1).numpy())
for act in activations["cls_attn_layers"]:
    layer_acts.append(act.mean(dim=1).numpy())
layer_acts.append(activations["final_cls_token"].numpy())

for name, act in zip(layer_order, layer_acts):
    mean_fisher, max_fisher, _ = compute_separability(act, vis_labels)
    separability_stats.append(
        {
            "layer": name,
            "mean_fisher": mean_fisher,
            "max_fisher": max_fisher,
            "dim": act.shape[1],
        }
    )

# Plot separability evolution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
x_pos = range(len(separability_stats))
ax.bar(
    x_pos, [s["mean_fisher"] for s in separability_stats], color="steelblue", alpha=0.7
)
ax.set_xticks(x_pos)
ax.set_xticklabels(
    [s["layer"].replace("_", "\n") for s in separability_stats],
    rotation=45,
    ha="right",
    fontsize=8,
)
ax.set_ylabel("Mean Fisher Discriminant Ratio")
ax.set_title("Average Class Separability Across Layers")

ax = axes[1]
ax.bar(
    x_pos, [s["max_fisher"] for s in separability_stats], color="darkorange", alpha=0.7
)
ax.set_xticks(x_pos)
ax.set_xticklabels(
    [s["layer"].replace("_", "\n") for s in separability_stats],
    rotation=45,
    ha="right",
    fontsize=8,
)
ax.set_ylabel("Max Fisher Discriminant Ratio")
ax.set_title("Best Single-Neuron Separability Across Layers")

plt.tight_layout()
save_fig(fig, "layer_separability_evolution")
plt.show()

# ============================================================
# 7. PER-CONSTITUENT ACTIVATION ANALYSIS
# ============================================================
print("\nAnalyzing per-constituent activations...")

# Look at how individual constituents are represented after particle attention
final_part_attn = activations["particle_attn_layers"][-1]  # [B, N, D]

# Compute activation magnitude per constituent
const_act_magnitude = torch.norm(final_part_attn, dim=-1).numpy()  # [B, N]

# Get constituent pT for correlation
const_pt_vis = vis_x[:, :, 1].cpu().numpy()
mask_vis = vis_mask.cpu().numpy()

# Flatten for scatter plot
pt_flat = []
act_flat = []
label_flat = []

for i in range(n_vis):
    n_valid = int(mask_vis[i].sum())
    pt_flat.extend(const_pt_vis[i, :n_valid])
    act_flat.extend(const_act_magnitude[i, :n_valid])
    label_flat.extend([vis_labels[i]] * n_valid)

pt_flat = np.array(pt_flat)
act_flat = np.array(act_flat)
label_flat = np.array(label_flat)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Constituent pT vs activation magnitude
ax = axes[0]
sig_mask_const = label_flat == 1
ax.scatter(
    pt_flat[~sig_mask_const],
    act_flat[~sig_mask_const],
    alpha=0.1,
    s=2,
    c="red",
    label="Background",
)
ax.scatter(
    pt_flat[sig_mask_const],
    act_flat[sig_mask_const],
    alpha=0.1,
    s=2,
    c="blue",
    label="Signal",
)
ax.set_xlabel("Constituent $p_T$ [GeV]")
ax.set_ylabel("Activation Magnitude (L2 norm)")
ax.set_title("Constituent Activation vs $p_T$")
ax.legend()
ax.set_xlim(0, np.percentile(pt_flat, 99))

# Histogram of activation magnitudes
ax = axes[1]
ax.hist(
    act_flat[sig_mask_const],
    bins=50,
    histtype="step",
    label="Signal",
    color="blue",
    density=True,
)
ax.hist(
    act_flat[~sig_mask_const],
    bins=50,
    histtype="step",
    label="Background",
    color="red",
    density=True,
)
ax.set_xlabel("Activation Magnitude")
ax.set_ylabel("Density")
ax.set_title("Per-Constituent Activation Distribution")
ax.legend()

plt.tight_layout()
save_fig(fig, "constituent_activation_analysis")
plt.show()

print(f"\n{'='*60}")
print("Layer activation analysis complete!")
print(f"{'='*60}")

In [ ]:
# ============================================================
# PER-CONSTITUENT ACTIVATION — BY PARTICLE TYPE (NEW)
# ============================================================
import gc

if n_features > 12:
    # Decode particle types for the visualization subset
    vis_particle_types = np.argmax(vis_x[:, :, 12:17].cpu().numpy(), axis=-1)
    vis_particle_types[~mask_vis] = -1

    # Flatten per-constituent data (reuse pt_flat, act_flat, label_flat from section 7)
    ptype_flat = []
    for i in range(n_vis):
        n_valid = int(mask_vis[i].sum())
        ptype_flat.extend(vis_particle_types[i, :n_valid])
    ptype_flat = np.array(ptype_flat)

    # --- Plot 1: Activation magnitude scatter + histogram by particle type ---
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    ax = axes[0]
    for pid in range(N_PARTICLE_TYPES):
        pmask = ptype_flat == pid
        if pmask.sum() == 0:
            continue
        ax.scatter(pt_flat[pmask], act_flat[pmask], alpha=0.05, s=2,
                  c=PARTICLE_TYPE_COLORS[pid], label=PARTICLE_TYPE_NAMES[pid])
    ax.set_xlabel("Constituent $p_T$ [GeV]")
    ax.set_ylabel("Activation Magnitude (L2 norm)")
    ax.set_title("Activation vs $p_T$ by Particle Type")
    ax.legend(markerscale=10, fontsize=9)
    ax.set_xlim(0, np.percentile(pt_flat, 99))

    ax = axes[1]
    for pid in range(N_PARTICLE_TYPES):
        pmask = ptype_flat == pid
        if pmask.sum() == 0:
            continue
        ax.hist(act_flat[pmask], bins=50, histtype="step",
               label=PARTICLE_TYPE_NAMES[pid], color=PARTICLE_TYPE_COLORS[pid],
               density=True, linewidth=1.5)
    ax.set_xlabel("Activation Magnitude")
    ax.set_ylabel("Density")
    ax.set_title("Per-Constituent Activation by Particle Type")
    ax.legend(fontsize=9)

    plt.tight_layout()
    save_fig(fig, "constituent_activation_by_particle_type")
    plt.show()


In [ ]:
# ============================================================
# ACTIVATION BOX PLOTS: PARTICLE TYPE x JET CLASS (NEW)
# ============================================================
if n_features > 12:
    fig, ax = plt.subplots(figsize=(14, 6))

    box_data = []
    box_labels_list = []
    box_positions = []
    box_colors = []

    pos = 0
    for pid in range(N_PARTICLE_TYPES):
        for label_val, label_name in [(1, "Sig"), (0, "Bkg")]:
            pmask = (ptype_flat == pid) & (label_flat == label_val)
            if pmask.sum() > 0:
                vals = act_flat[pmask]
                # Subsample for box plot speed
                if len(vals) > 5000:
                    vals = np.random.choice(vals, 5000, replace=False)
                box_data.append(vals)
                box_labels_list.append(f"{PARTICLE_TYPE_NAMES[pid]}\n({label_name})")
                box_positions.append(pos)
                box_colors.append("blue" if label_val == 1 else "red")
            pos += 1
        pos += 0.5  # Gap between particle types

    bp = ax.boxplot(box_data, positions=box_positions, widths=0.6,
                     patch_artist=True, showfliers=False)
    for patch, color in zip(bp["boxes"], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.5)

    ax.set_xticks(box_positions)
    ax.set_xticklabels(box_labels_list, fontsize=8, rotation=30, ha="right")
    ax.set_ylabel("Activation Magnitude (L2 norm)")
    ax.set_title("Activation Magnitude by Particle Type and Jet Class")

    plt.tight_layout()
    save_fig(fig, "activation_boxplot_by_type_and_class")
    plt.show()


In [ ]:
# ============================================================
# PER-CONSTITUENT t-SNE — BY PARTICLE TYPE (NEW)
# ============================================================
if n_features > 12:
    final_part_attn_np = activations["particle_attn_layers"][-1].numpy()  # (n_vis, 128, D)

    # Collect valid constituent embeddings with metadata
    embed_list = []
    ptype_list = []
    jet_label_list = []

    for i in range(n_vis):
        n_valid = int(mask_vis[i].sum())
        embed_list.append(final_part_attn_np[i, :n_valid, :])
        ptype_list.extend(vis_particle_types[i, :n_valid])
        jet_label_list.extend([vis_labels[i].item()] * n_valid)

    embed_all = np.concatenate(embed_list, axis=0)
    ptype_all = np.array(ptype_list)
    jet_label_all = np.array(jet_label_list)

    # Subsample for t-SNE (max 5000 points for speed)
    n_tsne = min(5000, len(embed_all))
    tsne_idx = np.random.choice(len(embed_all), n_tsne, replace=False)
    embed_sub = embed_all[tsne_idx]
    ptype_sub = ptype_all[tsne_idx]
    label_sub = jet_label_all[tsne_idx]

    # PCA then t-SNE
    if embed_sub.shape[1] > 50:
        pca = PCA(n_components=50, random_state=42)
        embed_sub = pca.fit_transform(embed_sub)

    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, n_tsne // 4))
    tsne_result = tsne.fit_transform(embed_sub)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

    # Left: colored by particle type
    for pid in range(N_PARTICLE_TYPES):
        pmask = ptype_sub == pid
        if pmask.sum() == 0:
            continue
        ax1.scatter(tsne_result[pmask, 0], tsne_result[pmask, 1],
                   c=PARTICLE_TYPE_COLORS[pid], alpha=0.3, s=5,
                   label=PARTICLE_TYPE_NAMES[pid])
    ax1.set_xlabel("t-SNE 1")
    ax1.set_ylabel("t-SNE 2")
    ax1.set_title("Per-Constituent Embedding — by Particle Type")
    ax1.legend(markerscale=5, fontsize=9)

    # Right: colored by jet class
    ax2.scatter(tsne_result[label_sub == 0, 0], tsne_result[label_sub == 0, 1],
               c="red", alpha=0.3, s=5, label="Background")
    ax2.scatter(tsne_result[label_sub == 1, 0], tsne_result[label_sub == 1, 1],
               c="blue", alpha=0.3, s=5, label="Signal")
    ax2.set_xlabel("t-SNE 1")
    ax2.set_ylabel("t-SNE 2")
    ax2.set_title("Per-Constituent Embedding — by Jet Class")
    ax2.legend(markerscale=5, fontsize=9)

    plt.suptitle("t-SNE of Per-Constituent Representations (Final Particle Attention Layer)", fontsize=13)
    plt.tight_layout()
    save_fig(fig, "constituent_tsne_by_particle_type")
    plt.show()

    del embed_all, embed_list, embed_sub, tsne_result
    gc.collect()


In [ ]:
# ============================================================
# FEATURE IMPORTANCE ANALYSIS
# ============================================================
from sklearn.metrics import roc_auc_score
import torch.nn.functional as F

print("=" * 60)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 60)

# Feature names for analysis
input_feature_names = [
    "Mass",
    "pT",
    "η",
    "φ",
    "d_xy",
    "z_0",
    "Charge",
    "log(pT_rel)",
    "Δη",
    "Δφ",
    "PUPPI weight",
    "log(ΔR)",
]

# Add particle ID features if present
n_features = all_constituents.shape[2]
if n_features > 12:
    for i in range(12, n_features):
        input_feature_names.append(f"ParticleID_{i-11}")

print(f"Analyzing {len(input_feature_names)} input features")

# ============================================================
# 1. GRADIENT-BASED FEATURE IMPORTANCE
# ============================================================
print("\n1. Computing gradient-based feature importance...")

n_grad_samples = min(500, len(all_constituents))
grad_x = all_constituents[:n_grad_samples].clone().to(device)
grad_x.requires_grad_(True)
grad_mask = grad_x[:, :, 1] > 0
grad_labels = all_labels[:n_grad_samples]

model.eval()

# Forward pass
tmep_outputs = model(grad_x, particle_mask=grad_mask)
outputs = tmep_outputs["classification"]

# Compute gradients w.r.t. inputs for both classes
# Gradient for positive class (signal)
grad_x.grad = None
loss_signal = outputs.sum()
loss_signal.backward(retain_graph=True)
grads_signal = grad_x.grad.clone().detach().cpu()

# Reset and compute for sigmoid output
grad_x.grad = None
grad_x.requires_grad_(True)
tmep_outputs = model(grad_x, particle_mask=grad_mask)
outputs = tmep_outputs["classification"]
probs = torch.sigmoid(outputs)
probs.sum().backward()
grads_prob = grad_x.grad.clone().detach().cpu()

# Compute importance as mean absolute gradient per feature
mask_np = grad_mask.cpu().numpy()
grad_importance = np.zeros(len(input_feature_names))
grad_importance_signal = np.zeros(len(input_feature_names))
grad_importance_background = np.zeros(len(input_feature_names))

for feat_idx in range(min(len(input_feature_names), grads_prob.shape[2])):
    # Get gradients for valid constituents only
    valid_grads = []
    sig_grads = []
    bkg_grads = []

    for i in range(n_grad_samples):
        n_valid = int(mask_np[i].sum())
        feat_grads = np.abs(grads_prob[i, :n_valid, feat_idx].numpy())
        valid_grads.extend(feat_grads)

        if grad_labels[i] == 1:
            sig_grads.extend(feat_grads)
        else:
            bkg_grads.extend(feat_grads)

    grad_importance[feat_idx] = np.mean(valid_grads)
    grad_importance_signal[feat_idx] = np.mean(sig_grads) if sig_grads else 0
    grad_importance_background[feat_idx] = np.mean(bkg_grads) if bkg_grads else 0

# Normalize
grad_importance = grad_importance / grad_importance.sum()
grad_importance_signal = grad_importance_signal / grad_importance_signal.sum()
grad_importance_background = (
    grad_importance_background / grad_importance_background.sum()
)

# ============================================================
# 2. PERMUTATION IMPORTANCE
# ============================================================
print("2. Computing permutation importance (this may take a minute)...")

n_perm_samples = min(1000, len(all_constituents))
perm_x = all_constituents[:n_perm_samples].to(device)
perm_mask = perm_x[:, :, 1] > 0
perm_labels = all_labels[:n_perm_samples]
perm_weights = all_qcd_weights_val[:n_perm_samples]

# Get baseline AUC
model.eval()
with torch.no_grad():
    tmep_outputs = model(perm_x, particle_mask=perm_mask)
    baseline_outputs = (
        torch.sigmoid(tmep_outputs["classification"]).squeeze().cpu().numpy()
    )


baseline_auc = roc_auc_score(perm_labels, baseline_outputs, sample_weight=perm_weights)
print(f"  Baseline AUC: {baseline_auc:.4f}")

perm_importance = np.zeros(len(input_feature_names))
n_permutations = 5

for feat_idx in range(min(len(input_feature_names), perm_x.shape[2])):
    auc_drops = []

    for _ in range(n_permutations):
        # Create permuted copy
        perm_x_copy = perm_x.clone()

        # Permute this feature across samples
        perm_indices = torch.randperm(n_perm_samples)
        perm_x_copy[:, :, feat_idx] = perm_x[perm_indices, :, feat_idx]

        # Evaluate
        with torch.no_grad():
            tmep_outputs = model(perm_x_copy, particle_mask=perm_mask)
            perm_outputs = (
                torch.sigmoid(tmep_outputs["classification"]).squeeze().cpu().numpy()
            )

        try:
            perm_auc = roc_auc_score(
                perm_labels, perm_outputs, sample_weight=perm_weights
            )
            auc_drops.append(baseline_auc - perm_auc)
        except:
            auc_drops.append(0)

    perm_importance[feat_idx] = np.mean(auc_drops)

    if (feat_idx + 1) % 4 == 0:
        print(
            f"    Processed {feat_idx + 1}/{min(len(input_feature_names), perm_x.shape[2])} features"
        )

# ============================================================
# 3. FEATURE ABLATION (ZEROING OUT)
# ============================================================
print("3. Computing feature ablation importance...")

ablation_importance = np.zeros(len(input_feature_names))

for feat_idx in range(min(len(input_feature_names), perm_x.shape[2])):
    # Create ablated copy
    ablated_x = perm_x.clone()
    ablated_x[:, :, feat_idx] = 0  # Zero out this feature

    # Evaluate
    with torch.no_grad():
        tmep_outputs = model(ablated_x, particle_mask=perm_mask)
        ablated_outputs = (
            torch.sigmoid(tmep_outputs["classification"]).squeeze().cpu().numpy()
        )

    try:
        ablated_auc = roc_auc_score(
            perm_labels, ablated_outputs, sample_weight=perm_weights
        )
        ablation_importance[feat_idx] = baseline_auc - ablated_auc
    except:
        ablation_importance[feat_idx] = 0

# ============================================================
# 4. STATISTICAL FEATURE SEPARABILITY
# ============================================================
print("4. Computing statistical feature separability...")

# Use validation dataset (all_constituents is already available)
stat_x = all_constituents[:, :, : len(input_feature_names)].numpy()
stat_mask = (all_constituents[:, :, 1] > 0).numpy()  # pT > 0 means valid
stat_labels = all_labels

# Compute Fisher discriminant ratio for each feature
fisher_scores = np.zeros(len(input_feature_names))
ks_scores = np.zeros(len(input_feature_names))

from scipy.stats import ks_2samp

for feat_idx in range(min(len(input_feature_names), stat_x.shape[2])):
    # Get signal and background values for valid constituents only
    # Extract feature values and mask for signal jets
    sig_feat_vals = stat_x[
        stat_labels == 1, :, feat_idx
    ]  # [n_sig_jets, n_constituents]
    sig_mask_vals = stat_mask[stat_labels == 1]  # [n_sig_jets, n_constituents]
    sig_vals = sig_feat_vals[sig_mask_vals].flatten()  # flatten valid constituents

    # Extract feature values and mask for background jets
    bkg_feat_vals = stat_x[stat_labels == 0, :, feat_idx]
    bkg_mask_vals = stat_mask[stat_labels == 0]
    bkg_vals = bkg_feat_vals[bkg_mask_vals].flatten()

    # Fisher discriminant ratio
    sig_mean, bkg_mean = sig_vals.mean(), bkg_vals.mean()
    sig_var, bkg_var = sig_vals.var() + 1e-8, bkg_vals.var() + 1e-8
    fisher_scores[feat_idx] = (sig_mean - bkg_mean) ** 2 / (sig_var + bkg_var)

    # Kolmogorov-Smirnov test (subsample for speed)
    n_subsample = min(10000, len(sig_vals), len(bkg_vals))
    sig_sample = (
        np.random.choice(sig_vals, n_subsample, replace=False)
        if len(sig_vals) > n_subsample
        else sig_vals
    )
    bkg_sample = (
        np.random.choice(bkg_vals, n_subsample, replace=False)
        if len(bkg_vals) > n_subsample
        else bkg_vals
    )
    ks_stat, _ = ks_2samp(sig_sample, bkg_sample)
    ks_scores[feat_idx] = ks_stat

# ============================================================
# 5. VISUALIZE RESULTS
# ============================================================
print("\nGenerating visualizations...")

# Limit to actual features we have
n_plot_features = min(len(input_feature_names), 12)
plot_feature_names = input_feature_names[:n_plot_features]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Gradient-based importance
ax = axes[0, 0]
sorted_idx = np.argsort(grad_importance[:n_plot_features])[::-1]
y_pos = np.arange(n_plot_features)
ax.barh(
    y_pos, grad_importance[:n_plot_features][sorted_idx], color="steelblue", alpha=0.8
)
ax.set_yticks(y_pos)
ax.set_yticklabels([plot_feature_names[i] for i in sorted_idx])
ax.set_xlabel("Normalized Mean |Gradient|")
ax.set_title("Gradient-Based Importance")
ax.invert_yaxis()

# 2. Permutation importance
ax = axes[0, 1]
sorted_idx = np.argsort(perm_importance[:n_plot_features])[::-1]
colors = [
    "green" if v > 0 else "red" for v in perm_importance[:n_plot_features][sorted_idx]
]
ax.barh(y_pos, perm_importance[:n_plot_features][sorted_idx], color=colors, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels([plot_feature_names[i] for i in sorted_idx])
ax.set_xlabel("AUC Drop When Permuted")
ax.set_title("Permutation Importance")
ax.axvline(0, color="black", linestyle="--", alpha=0.5)
ax.invert_yaxis()

# 3. Ablation importance
ax = axes[0, 2]
sorted_idx = np.argsort(ablation_importance[:n_plot_features])[::-1]
colors = [
    "green" if v > 0 else "red"
    for v in ablation_importance[:n_plot_features][sorted_idx]
]
ax.barh(
    y_pos, ablation_importance[:n_plot_features][sorted_idx], color=colors, alpha=0.8
)
ax.set_yticks(y_pos)
ax.set_yticklabels([plot_feature_names[i] for i in sorted_idx])
ax.set_xlabel("AUC Drop When Zeroed")
ax.set_title("Feature Ablation Importance")
ax.axvline(0, color="black", linestyle="--", alpha=0.5)
ax.invert_yaxis()

# 4. Fisher discriminant ratio
ax = axes[1, 0]
sorted_idx = np.argsort(fisher_scores[:n_plot_features])[::-1]
ax.barh(
    y_pos, fisher_scores[:n_plot_features][sorted_idx], color="darkorange", alpha=0.8
)
ax.set_yticks(y_pos)
ax.set_yticklabels([plot_feature_names[i] for i in sorted_idx])
ax.set_xlabel("Fisher Discriminant Ratio")
ax.set_title("Statistical Separability (Fisher)")
ax.invert_yaxis()

# 5. KS statistic
ax = axes[1, 1]
sorted_idx = np.argsort(ks_scores[:n_plot_features])[::-1]
ax.barh(y_pos, ks_scores[:n_plot_features][sorted_idx], color="purple", alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels([plot_feature_names[i] for i in sorted_idx])
ax.set_xlabel("KS Statistic")
ax.set_title("Statistical Separability (KS Test)")
ax.invert_yaxis()

# 6. Combined ranking
ax = axes[1, 2]


# Normalize all scores to [0, 1] and combine
def normalize_scores(scores):
    if scores.max() - scores.min() > 0:
        return (scores - scores.min()) / (scores.max() - scores.min())
    return scores


combined_score = (
    normalize_scores(grad_importance[:n_plot_features])
    + normalize_scores(np.maximum(perm_importance[:n_plot_features], 0))
    + normalize_scores(np.maximum(ablation_importance[:n_plot_features], 0))
    + normalize_scores(fisher_scores[:n_plot_features])
    + normalize_scores(ks_scores[:n_plot_features])
) / 5

sorted_idx = np.argsort(combined_score)[::-1]
ax.barh(y_pos, combined_score[sorted_idx], color="darkgreen", alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels([plot_feature_names[i] for i in sorted_idx])
ax.set_xlabel("Combined Importance Score")
ax.set_title("Overall Feature Ranking")
ax.invert_yaxis()

plt.suptitle(
    "Feature Importance Analysis for Signal vs Background Discrimination", fontsize=14
)
plt.tight_layout()
save_fig(fig, "feature_importance_analysis")
plt.show()

# ============================================================
# 6. SIGNAL VS BACKGROUND GRADIENT COMPARISON
# ============================================================
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(n_plot_features)
width = 0.35

ax.bar(
    x - width / 2,
    grad_importance_signal[:n_plot_features],
    width,
    label="Signal (b-jets)",
    color="blue",
    alpha=0.7,
)
ax.bar(
    x + width / 2,
    grad_importance_background[:n_plot_features],
    width,
    label="Background",
    color="red",
    alpha=0.7,
)

ax.set_xlabel("Feature")
ax.set_ylabel("Normalized Mean |Gradient|")
ax.set_title("Gradient Importance: Signal vs Background")
ax.set_xticks(x)
ax.set_xticklabels(plot_feature_names, rotation=45, ha="right")
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
save_fig(fig, "gradient_importance_by_class")
plt.show()

# ============================================================
# 7. FEATURE CORRELATION WITH MODEL OUTPUT
# ============================================================
print("\nComputing feature correlations with model output...")

# Get model predictions
with torch.no_grad():
    tmep_outputs = model(perm_x_copy, particle_mask=perm_mask)
    pred_outputs = torch.sigmoid(tmep_outputs["classification"]).squeeze().cpu().numpy()

# Compute mean feature values per jet (for valid constituents)
mean_features = np.zeros((n_perm_samples, n_plot_features))
for i in range(n_perm_samples):
    n_valid = int(perm_mask[i].sum().item())
    for f in range(n_plot_features):
        mean_features[i, f] = perm_x[i, :n_valid, f].cpu().numpy().mean()

# Correlation with predictions
feature_pred_corr = np.zeros(n_plot_features)
for f in range(n_plot_features):
    feature_pred_corr[f] = np.corrcoef(mean_features[:, f], pred_outputs)[0, 1]

fig, ax = plt.subplots(figsize=(12, 6))
colors = ["blue" if c > 0 else "red" for c in feature_pred_corr]
bars = ax.bar(range(n_plot_features), feature_pred_corr, color=colors, alpha=0.7)
ax.set_xticks(range(n_plot_features))
ax.set_xticklabels(plot_feature_names, rotation=45, ha="right")
ax.set_ylabel("Pearson Correlation")
ax.set_xlabel("Feature")
ax.set_title("Feature Correlation with Model Output (b-tag Score)")
ax.axhline(0, color="black", linestyle="--", alpha=0.5)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
save_fig(fig, "feature_correlation_with_output")
plt.show()

# ============================================================
# 8. SUMMARY TABLE
# ============================================================
print("\n" + "=" * 80)
print("FEATURE IMPORTANCE SUMMARY")
print("=" * 80)
print(
    f"{'Feature':<15} {'Gradient':>10} {'Permute':>10} {'Ablation':>10} {'Fisher':>10} {'KS':>10} {'Combined':>10}"
)
print("-" * 80)

for i in range(n_plot_features):
    print(
        f"{plot_feature_names[i]:<15} "
        f"{grad_importance[i]:>10.4f} "
        f"{perm_importance[i]:>10.4f} "
        f"{ablation_importance[i]:>10.4f} "
        f"{fisher_scores[i]:>10.4f} "
        f"{ks_scores[i]:>10.4f} "
        f"{combined_score[i]:>10.4f}"
    )

# Top features summary
print("\n" + "=" * 80)
print("TOP 5 MOST IMPORTANT FEATURES (by combined score):")
print("=" * 80)
top_5_idx = np.argsort(combined_score)[::-1][:5]
for rank, idx in enumerate(top_5_idx, 1):
    print(f"  {rank}. {plot_feature_names[idx]} (score: {combined_score[idx]:.4f})")

print(f"\n{'='*60}")
print("Feature importance analysis complete!")
print(f"{'='*60}")

In [ ]:
# ============================================================
# GRADIENT IMPORTANCE — BY PARTICLE TYPE (NEW)
# ============================================================
if n_features > 12:
    print("Computing gradient importance per particle type...")

    # Decode particle types for gradient samples
    grad_particle_types = np.argmax(
        all_constituents[:n_grad_samples, :, 12:17].numpy(), axis=-1
    )  # (n_grad_samples, 128)
    grad_particle_types[~mask_np] = -1

    # Compute mean |gradient| per feature per particle type
    grad_by_type = np.zeros((N_PARTICLE_TYPES, n_plot_features))

    for pid in range(N_PARTICLE_TYPES):
        for feat_idx in range(n_plot_features):
            grads_list = []
            for i in range(n_grad_samples):
                n_valid = int(mask_np[i].sum())
                ptypes_i = grad_particle_types[i, :n_valid]
                feat_grads = np.abs(grads_prob[i, :n_valid, feat_idx].numpy())
                type_grads = feat_grads[ptypes_i == pid]
                grads_list.extend(type_grads)

            grad_by_type[pid, feat_idx] = np.mean(grads_list) if grads_list else 0

    # Normalize per particle type (each row sums to 1)
    for pid in range(N_PARTICLE_TYPES):
        row_sum = grad_by_type[pid].sum()
        if row_sum > 0:
            grad_by_type[pid] /= row_sum

    # --- Plot 1: Grouped bar chart ---
    fig, ax = plt.subplots(figsize=(16, 7))

    x = np.arange(n_plot_features)
    total_width = 0.8
    bar_width = total_width / N_PARTICLE_TYPES

    for pid in range(N_PARTICLE_TYPES):
        offset = (pid - N_PARTICLE_TYPES / 2 + 0.5) * bar_width
        ax.bar(x + offset, grad_by_type[pid], bar_width,
               label=PARTICLE_TYPE_NAMES[pid], color=PARTICLE_TYPE_COLORS[pid], alpha=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels(plot_feature_names, rotation=45, ha="right")
    ax.set_ylabel("Normalized Mean |Gradient|")
    ax.set_title("Feature Importance by Particle Type (Gradient-Based)")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    save_fig(fig, "gradient_importance_by_particle_type")
    plt.show()

    # --- Plot 2: Heatmap ---
    fig, ax = plt.subplots(figsize=(14, 5))

    im = ax.imshow(grad_by_type, cmap="YlOrRd", aspect="auto")
    ax.set_yticks(range(N_PARTICLE_TYPES))
    ax.set_yticklabels(PARTICLE_TYPE_NAMES)
    ax.set_xticks(range(n_plot_features))
    ax.set_xticklabels(plot_feature_names, rotation=45, ha="right")
    ax.set_title("Feature Importance Heatmap by Particle Type")
    plt.colorbar(im, ax=ax, label="Normalized |Gradient|")

    # Annotate
    for ii in range(N_PARTICLE_TYPES):
        for jj in range(n_plot_features):
            ax.text(jj, ii, f"{grad_by_type[ii, jj]:.3f}", ha="center", va="center",
                   fontsize=7, color="white" if grad_by_type[ii, jj] > grad_by_type.max() * 0.5 else "black")

    plt.tight_layout()
    save_fig(fig, "gradient_importance_heatmap_by_particle_type")
    plt.show()


In [ ]:
# ============================================================
# MODEL BEHAVIOR ANALYSIS & MAXIMUM DISCRIMINATIVE POWER
# ============================================================
from sklearn.neighbors import KNeighborsClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, log_loss
from scipy.stats import entropy
import warnings

warnings.filterwarnings("ignore")

print("=" * 70)
print("MODEL BEHAVIOR ANALYSIS & MAXIMUM DISCRIMINATIVE POWER ESTIMATION")
print("=" * 70)

# ============================================================
# 1. ERROR ANALYSIS - UNDERSTANDING MISCLASSIFICATIONS
# ============================================================
print("\n1. ERROR ANALYSIS")
print("-" * 50)

# Get predictions at different thresholds
threshold = 0.5
# cls_outputs = torch.sigmoid(all_outputs["classification"]).squeeze().cpu().numpy()
predictions = (all_outputs > threshold).astype(int)
correct = predictions == all_labels
incorrect = ~correct

all_labels = all_labels.squeeze()

# Classify error types
false_positives = (predictions == 1) & (all_labels == 0)
false_negatives = (predictions == 0) & (all_labels == 1)
true_positives = (predictions == 1) & (all_labels == 1)
true_negatives = (predictions == 0) & (all_labels == 0)

print(f"Threshold: {threshold}")
print(
    f"  True Positives:  {true_positives.sum():>6} ({100*true_positives.sum()/all_labels.sum():.1f}% of signal)"
)
print(
    f"  True Negatives:  {true_negatives.sum():>6} ({100*true_negatives.sum()/(len(all_labels)-all_labels.sum()):.1f}% of background)"
)
print(
    f"  False Positives: {false_positives.sum():>6} (background misclassified as signal)"
)
print(
    f"  False Negatives: {false_negatives.sum():>6} (signal misclassified as background)"
)

# Analyze characteristics of misclassified samples
print("\n  Kinematic properties of misclassified jets:")

# Jet pT for different categories
fp_pt = val_jet_pt[false_positives]
fn_pt = val_jet_pt[false_negatives]
tp_pt = val_jet_pt[true_positives]
tn_pt = val_jet_pt[true_negatives]

print(
    f"    False Positives: mean pT = {fp_pt.mean():.1f} GeV, mean |η| = {np.abs(val_jet_eta[false_positives]).mean():.2f}"
)
print(
    f"    False Negatives: mean pT = {fn_pt.mean():.1f} GeV, mean |η| = {np.abs(val_jet_eta[false_negatives]).mean():.2f}"
)
print(
    f"    True Positives:  mean pT = {tp_pt.mean():.1f} GeV, mean |η| = {np.abs(val_jet_eta[true_positives]).mean():.2f}"
)
print(
    f"    True Negatives:  mean pT = {tn_pt.mean():.1f} GeV, mean |η| = {np.abs(val_jet_eta[true_negatives]).mean():.2f}"
)

# ============================================================
# 2. CONFIDENCE CALIBRATION
# ============================================================
print("\n2. CONFIDENCE CALIBRATION")
print("-" * 50)

# Compute calibration metrics
brier = brier_score_loss(all_labels, all_outputs)
logloss = log_loss(all_labels, np.clip(all_outputs, 1e-7, 1 - 1e-7))

print(f"  Brier Score: {brier:.4f} (lower is better, perfect = 0)")
print(f"  Log Loss: {logloss:.4f} (lower is better)")

# Calibration curve
prob_true, prob_pred = calibration_curve(
    all_labels, all_outputs, n_bins=10, strategy="uniform"
)

print(f"\n  Calibration by probability bin:")
print(f"  {'Predicted':>12} {'Actual':>12} {'Difference':>12}")
for pt, pp in zip(prob_true, prob_pred):
    print(f"  {pp:>12.3f} {pt:>12.3f} {pt-pp:>+12.3f}")

# Expected Calibration Error (ECE)
bin_edges = np.linspace(0, 1, 11)
ece = 0
for i in range(10):
    mask = (all_outputs >= bin_edges[i]) & (all_outputs < bin_edges[i + 1])
    if mask.sum() > 0:
        bin_acc = all_labels[mask].mean()
        bin_conf = all_outputs[mask].mean()
        ece += mask.sum() * np.abs(bin_acc - bin_conf)
ece /= len(all_labels)
print(f"\n  Expected Calibration Error (ECE): {ece:.4f}")

# ============================================================
# 3. HARD SAMPLE ANALYSIS
# ============================================================
print("\n3. HARD SAMPLE ANALYSIS")
print("-" * 50)

# Define "hard" samples as those with high uncertainty (close to 0.5)
uncertainty = np.abs(all_outputs - 0.5)
hard_threshold = 0.2  # samples with output between 0.3 and 0.7
hard_samples = uncertainty < hard_threshold
easy_samples = uncertainty >= 0.4

print(
    f"  Hard samples (output in [0.3, 0.7]): {hard_samples.sum()} ({100*hard_samples.mean():.1f}%)"
)
print(
    f"  Easy samples (output < 0.1 or > 0.9): {easy_samples.sum()} ({100*easy_samples.mean():.1f}%)"
)

# Accuracy on easy vs hard samples
hard_acc = (predictions[hard_samples] == all_labels[hard_samples]).mean()
easy_acc = (predictions[easy_samples] == all_labels[easy_samples]).mean()
print(f"\n  Accuracy on hard samples: {100*hard_acc:.1f}%")
print(f"  Accuracy on easy samples: {100*easy_acc:.1f}%")

# What makes samples hard?
print("\n  Characteristics of hard samples:")
hard_mask = all_constituents[:, :, 1] > 0
hard_n_const = hard_mask[hard_samples].sum(dim=1).float().mean().item()
easy_n_const = hard_mask[easy_samples].sum(dim=1).float().mean().item()
print(f"    Mean constituents (hard): {hard_n_const:.1f}")
print(f"    Mean constituents (easy): {easy_n_const:.1f}")
print(f"    Mean pT (hard): {val_jet_pt[hard_samples].mean():.1f} GeV")
print(f"    Mean pT (easy): {val_jet_pt[easy_samples].mean():.1f} GeV")

# ============================================================
# 4. MAXIMUM DISCRIMINATIVE POWER ESTIMATION
# ============================================================
print("\n4. MAXIMUM DISCRIMINATIVE POWER ESTIMATION")
print("-" * 50)

# Method 1: Class overlap estimation using k-NN
print("\n  Method 1: k-NN Class Overlap Estimation")

# Use a subset for speed
n_knn = min(50000, len(all_constituents))
knn_x = all_constituents[:n_knn]
knn_mask = knn_x[:, :, 1] > 0

# Compute mean features per jet for k-NN
knn_features = []
for i in range(n_knn):
    n_valid = int(knn_mask[i].sum().item())
    jet_feats = knn_x[i, :n_valid, :12].mean(dim=0).numpy()  # Mean over constituents
    knn_features.append(jet_feats)
knn_features = np.array(knn_features)
knn_labels = all_labels[:n_knn]

# Fit k-NN classifier with different k values
knn_aucs = []
for k in [1, 3, 5, 11, 21]:
    knn = KNeighborsClassifier(n_neighbors=k, weights="distance")
    # Use cross-validation-like split
    train_idx = np.random.choice(n_knn, int(0.7 * n_knn), replace=False)
    test_idx = np.setdiff1d(np.arange(n_knn), train_idx)

    knn.fit(knn_features[train_idx], knn_labels[train_idx])
    knn_proba = knn.predict_proba(knn_features[test_idx])[:, 1]
    knn_auc = roc_auc_score(
        knn_labels[test_idx],
        knn_proba,
        sample_weight=roc_weights[:n_knn][test_idx],
    )
    knn_aucs.append(knn_auc)
    print(f"    k={k:>2}: AUC = {knn_auc:.4f}")

best_knn_auc = max(knn_aucs)
print(f"    Best k-NN AUC (upper bound estimate): {best_knn_auc:.4f}")

# Method 2: Bayes error estimation via density overlap
print("\n  Method 2: Feature Distribution Overlap Analysis")

# Compute overlap for each feature
from scipy.stats import gaussian_kde

feature_overlaps = []
for feat_idx in range(12):
    sig_vals = knn_features[knn_labels == 1, feat_idx]
    bkg_vals = knn_features[knn_labels == 0, feat_idx]

    # Subsample for speed
    n_sub = min(1000, len(sig_vals), len(bkg_vals))
    sig_sub = np.random.choice(sig_vals, n_sub, replace=False)
    bkg_sub = np.random.choice(bkg_vals, n_sub, replace=False)

    try:
        # Compute KDE overlap
        x_range = np.linspace(
            min(sig_sub.min(), bkg_sub.min()), max(sig_sub.max(), bkg_sub.max()), 100
        )
        sig_kde = gaussian_kde(sig_sub)
        bkg_kde = gaussian_kde(bkg_sub)

        sig_pdf = sig_kde(x_range)
        bkg_pdf = bkg_kde(x_range)

        # Overlap coefficient (Bhattacharyya)
        overlap = np.sum(np.sqrt(sig_pdf * bkg_pdf)) * (x_range[1] - x_range[0])
        feature_overlaps.append(overlap)
    except:
        feature_overlaps.append(1.0)

feature_overlaps = np.array(feature_overlaps)
mean_overlap = feature_overlaps.mean()

# Lower overlap = more separable
estimated_max_auc = 1 - mean_overlap / 2  # Rough approximation
print(f"    Mean feature overlap: {mean_overlap:.4f}")
print(f"    Estimated max AUC (from overlap): {estimated_max_auc:.4f}")

# Method 3: Theoretical bounds from class distributions
print("\n  Method 3: Theoretical Bounds")

# Class balance
n_signal = all_labels_after_cuts.sum()
n_background = len(all_labels_after_cuts) - n_signal
class_ratio = n_signal / len(all_labels_after_cuts)
print(
    f"    Class balance: {100*class_ratio:.1f}% signal / {100*(1-class_ratio):.1f}% background"
)

# Random classifier AUC
print(f"    Random classifier AUC: 0.5000")
print(
    f"    Current model AUC: {roc_auc_score(all_labels_after_cuts, all_outputs_after_cuts, sample_weight=roc_weights):.4f}"
)
print(f"    Best k-NN AUC: {best_knn_auc:.4f}")

# Gap analysis
current_auc = roc_auc_score(all_labels_after_cuts, all_outputs_after_cuts, sample_weight=roc_weights)
gap_to_knn = best_knn_auc - current_auc
gap_to_perfect = 1.0 - current_auc

print(f"\n    Performance gaps:")
print(f"      Gap to k-NN (achievable): {gap_to_knn:+.4f}")
print(f"      Gap to perfect: {gap_to_perfect:+.4f}")

# ============================================================
# 5. IMPROVEMENT RECOMMENDATIONS
# ============================================================
print("\n5. IMPROVEMENT RECOMMENDATIONS")
print("-" * 50)

recommendations = []

# Check calibration
if ece > 0.05:
    recommendations.append(
        "• Model is poorly calibrated (ECE > 0.05). Consider temperature scaling or Platt calibration."
    )

# Check class balance impact
if class_ratio < 0.3 or class_ratio > 0.7:
    recommendations.append(
        "• Class imbalance detected. Consider focal loss or class weighting."
    )

# Check for overfitting potential
if best_knn_auc < current_auc:
    recommendations.append(
        "• Model outperforms k-NN, suggesting good feature learning."
    )
else:
    recommendations.append(
        f"• k-NN achieves higher AUC ({best_knn_auc:.4f} vs {current_auc:.4f}). Model may benefit from more capacity or training."
    )

# Check hard samples
if hard_samples.mean() > 0.3:
    recommendations.append(
        f"• {100*hard_samples.mean():.0f}% of samples are hard. Consider curriculum learning or sample weighting."
    )

# Check kinematic dependencies
pt_corr = np.corrcoef(val_jet_pt, all_outputs)[0, 1]
if abs(pt_corr) > 0.3:
    recommendations.append(
        f"• Strong pT-score correlation ({pt_corr:.2f}). Consider pT reweighting for robustness."
    )

for rec in recommendations:
    print(f"  {rec}")

if not recommendations:
    print(
        "  • Model appears well-optimized! Consider ensemble methods or architectural changes for further improvement."
    )

# ============================================================
# 6. VISUALIZATION
# ============================================================
print("\nGenerating diagnostic plots...")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Calibration plot
ax = axes[0, 0]
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.plot(prob_pred, prob_true, "bo-", label="Model")
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Fraction of Positives")
ax.set_title("Calibration Curve")
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Error analysis by pT
ax = axes[0, 1]
pt_bins = np.linspace(0, 300, 20)
fp_hist, _ = np.histogram(fp_pt, bins=pt_bins, density=True)
fn_hist, _ = np.histogram(fn_pt, bins=pt_bins, density=True)
tp_hist, _ = np.histogram(tp_pt, bins=pt_bins, density=True)
tn_hist, _ = np.histogram(tn_pt, bins=pt_bins, density=True)

pt_centers = (pt_bins[:-1] + pt_bins[1:]) / 2
ax.plot(pt_centers, fp_hist, "r-", label="False Positives", alpha=0.8)
ax.plot(pt_centers, fn_hist, "b-", label="False Negatives", alpha=0.8)
ax.set_xlabel("Jet pT [GeV]")
ax.set_ylabel("Density")
ax.set_title("pT Distribution of Misclassified Jets")
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Confidence distribution
ax = axes[0, 2]
ax.hist(
    all_outputs[all_labels == 1],
    bins=50,
    range=(0, 1),
    histtype="step",
    label="Signal",
    color="blue",
    density=True,
)
ax.hist(
    all_outputs[all_labels == 0],
    bins=50,
    range=(0, 1),
    histtype="step",
    label="Background",
    color="red",
    density=True,
)
ax.axvline(0.5, color="black", linestyle="--", alpha=0.5)
ax.axvspan(0.3, 0.7, alpha=0.1, color="gray", label="Hard region")
ax.set_xlabel("Model Output")
ax.set_ylabel("Density")
ax.set_title("Output Distribution with Hard Region")
ax.legend()

# 4. AUC vs pT
ax = axes[1, 0]
pt_bin_edges = [25, 50, 75, 100, 150, 200, 300, 500]
pt_aucs = []
pt_counts = []
for i in range(len(pt_bin_edges) - 1):
    mask = (val_jet_pt >= pt_bin_edges[i]) & (val_jet_pt < pt_bin_edges[i + 1])
    if mask.sum() > 50 and len(np.unique(all_labels[mask])) > 1:
        pt_aucs.append(
            roc_auc_score(
                all_labels[mask],
                all_outputs[mask],
                sample_weight=all_qcd_weights_val[mask],
            )
        )
        pt_counts.append(mask.sum())
    else:
        pt_aucs.append(np.nan)
        pt_counts.append(mask.sum())

pt_centers = [
    (pt_bin_edges[i] + pt_bin_edges[i + 1]) / 2 for i in range(len(pt_bin_edges) - 1)
]
ax.bar(range(len(pt_aucs)), pt_aucs, color="steelblue", alpha=0.7)
ax.set_xticks(range(len(pt_aucs)))
ax.set_xticklabels(
    [f"{pt_bin_edges[i]}-{pt_bin_edges[i+1]}" for i in range(len(pt_bin_edges) - 1)],
    rotation=45,
)
ax.set_xlabel("pT Range [GeV]")
ax.set_ylabel("AUC")
ax.set_title("AUC vs Jet pT")
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.3, axis="y")

# 5. Feature overlap visualization
ax = axes[1, 1]
feature_names_short = [
    "Mass",
    "pT",
    "η",
    "φ",
    "d_xy",
    "z_0",
    "Q",
    "log(pT_rel)",
    "Δη",
    "Δφ",
    "PUPPI",
    "log(ΔR)",
]
sorted_idx = np.argsort(feature_overlaps)
ax.barh(
    range(len(feature_overlaps)),
    feature_overlaps[sorted_idx],
    color="darkorange",
    alpha=0.7,
)
ax.set_yticks(range(len(feature_overlaps)))
ax.set_yticklabels([feature_names_short[i] for i in sorted_idx])
ax.set_xlabel("Distribution Overlap (lower = more separable)")
ax.set_title("Feature Separability")
ax.invert_yaxis()

# 6. Summary metrics
ax = axes[1, 2]
ax.axis("off")
summary_text = f"""
MODEL PERFORMANCE SUMMARY
{'='*40}

Current Performance:
  • AUC: {current_auc:.4f}
  • Brier Score: {brier:.4f}
  • Log Loss: {logloss:.4f}
  • ECE: {ece:.4f}

Error Analysis:
  • False Positive Rate: {100*false_positives.sum()/(len(all_labels)-all_labels.sum()):.2f}%
  • False Negative Rate: {100*false_negatives.sum()/all_labels.sum():.2f}%
  • Hard Samples: {100*hard_samples.mean():.1f}%

Maximum Performance Estimate:
  • k-NN Upper Bound: {best_knn_auc:.4f}
  • From Feature Overlap: {estimated_max_auc:.4f}
  • Improvement Potential: {max(0, best_knn_auc - current_auc):.4f}

Data Characteristics:
  • Samples: {len(all_labels):,}
  • Signal: {int(n_signal):,} ({100*class_ratio:.1f}%)
  • Background: {int(n_background):,} ({100*(1-class_ratio):.1f}%)
"""
ax.text(
    0.05,
    0.95,
    summary_text,
    transform=ax.transAxes,
    fontsize=11,
    verticalalignment="top",
    fontfamily="monospace",
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
)

plt.suptitle("Model Behavior Analysis & Maximum Discriminative Power", fontsize=14)
plt.tight_layout()
save_fig(fig, "model_behavior_analysis")
plt.show()

print(f"\n{'='*70}")
print("Model behavior analysis complete!")
print(f"{'='*70}")

In [ ]:
# ============================================================
# ERROR ANALYSIS — BY PARTICLE COMPOSITION (NEW)
# ============================================================
if particle_type_full is not None and n_features > 12:
    # Decode particle types for all validation jets
    val_particle_types = np.argmax(
        all_constituents[:, :, 12:17].numpy(), axis=-1
    )
    val_mask = (all_constituents[:, :, 1] > 0).numpy()
    val_particle_types[~val_mask] = -1

    val_particle_counts = np.zeros((len(all_constituents), N_PARTICLE_TYPES), dtype=int)
    for pid in range(N_PARTICLE_TYPES):
        val_particle_counts[:, pid] = (val_particle_types == pid).sum(axis=1)
    val_dominant_type = np.argmax(val_particle_counts, axis=1)

    # Misclassification rate by dominant particle type
    print("\n  Misclassification Rate by Dominant Particle Type:")
    print(f"  {'Type':<20} {'N_jets':>8} {'Error Rate':>12} {'FP Rate':>10} {'FN Rate':>10}")
    print("  " + "-" * 65)

    for pid in range(N_PARTICLE_TYPES):
        type_mask = val_dominant_type == pid
        n_type = type_mask.sum()
        if n_type == 0:
            continue

        n_incorrect = incorrect[type_mask].sum()
        n_fp = false_positives[type_mask].sum()
        n_fn = false_negatives[type_mask].sum()
        n_sig_type = (all_labels[type_mask] == 1).sum()
        n_bkg_type = (all_labels[type_mask] == 0).sum()

        err_rate = n_incorrect / n_type
        fp_rate = n_fp / n_bkg_type if n_bkg_type > 0 else 0
        fn_rate = n_fn / n_sig_type if n_sig_type > 0 else 0

        print(f"  {PARTICLE_TYPE_NAMES[pid]:<20} {n_type:>8} {err_rate:>12.4f} {fp_rate:>10.4f} {fn_rate:>10.4f}")

    # Composition of TP/FP/TN/FN categories
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    categories = [
        (true_positives, "True Positives"),
        (false_negatives, "False Negatives"),
        (true_negatives, "True Negatives"),
        (false_positives, "False Positives"),
    ]

    for ax, (cat_mask, cat_name) in zip(axes, categories):
        cat_counts = val_particle_counts[cat_mask].mean(axis=0)
        cat_fracs = cat_counts / cat_counts.sum() if cat_counts.sum() > 0 else cat_counts

        ax.bar(range(N_PARTICLE_TYPES), cat_fracs, color=PARTICLE_TYPE_COLORS, alpha=0.8)
        ax.set_xticks(range(N_PARTICLE_TYPES))
        ax.set_xticklabels(["ChHad", "NeuHad", "Photon", "Elec", "Muon"],
                          rotation=30, ha="right", fontsize=9)
        ax.set_ylabel("Mean Fraction")
        ax.set_title(f"{cat_name}\n(n={cat_mask.sum()})")
        ax.set_ylim(0, 1)

    plt.suptitle("Particle Composition of Classification Outcomes", fontsize=14)
    plt.tight_layout()
    save_fig(fig, "error_analysis_by_particle_type")
    plt.show()


In [ ]:
# Verify that feature distributions are preserved after stratified split
import matplotlib.pyplot as plt
import vector


from data_pipeline.combined_loader import CombinedJetDataLoader


def verify_distribution_preservation(
    combined_loader: CombinedJetDataLoader,
    n_bins: int = 50,
    figsize: Tuple[int, int] = (15, 10),
):
    """
    Verify that the stratified split preserves feature distributions
    by comparing original vs train vs validation distributions.
    """

    fig, axes = plt.subplots(2, 4, figsize=figsize)

    for row, (name, orig_ds, train_ds, val_ds) in enumerate(
        [
            (
                "PF",
                combined_loader.pf_dataset,
                combined_loader.train_pf,
                combined_loader.val_pf,
            ),
            (
                "PUPPI",
                combined_loader.puppi_dataset,
                combined_loader.train_puppi,
                combined_loader.val_puppi,
            ),
        ]
    ):
        # Get indices
        train_indices = np.array(train_ds.indices)
        val_indices = np.array(val_ds.indices)

        # Get constituent data
        X_orig = orig_ds.X.numpy()
        X_train = X_orig[train_indices]
        X_val = X_orig[val_indices]

        # Feature names (mass, pt, eta, phi)
        feature_names = ["Mass", "pT", "η", "φ"]

        for col, (feat_idx, feat_name) in enumerate(zip(range(4), feature_names)):
            ax = axes[row, col]

            # Get non-masked values for each split
            mask_orig = orig_ds.mask.numpy()
            mask_train = mask_orig[train_indices]
            mask_val = mask_orig[val_indices]

            vals_orig = X_orig[:, :, feat_idx][mask_orig].flatten()
            vals_train = X_train[:, :, feat_idx][mask_train].flatten()
            vals_val = X_val[:, :, feat_idx][mask_val].flatten()

            # Compute appropriate bins
            if feat_name == "pT":
                bins = np.linspace(0, np.percentile(vals_orig, 99), n_bins)
            elif feat_name == "Mass":
                bins = np.linspace(0, np.percentile(vals_orig, 99), n_bins)
            else:
                bins = np.linspace(vals_orig.min(), vals_orig.max(), n_bins)

            # Plot normalized histograms
            ax.hist(
                vals_orig,
                bins=bins,
                histtype="step",
                density=True,
                label="Original",
                color="black",
                linewidth=2,
            )
            ax.hist(
                vals_train,
                bins=bins,
                histtype="step",
                density=True,
                label="Train",
                color="blue",
                linewidth=1.5,
                linestyle="--",
            )
            ax.hist(
                vals_val,
                bins=bins,
                histtype="step",
                density=True,
                label="Val",
                color="red",
                linewidth=1.5,
                linestyle=":",
            )

            ax.set_xlabel(feat_name)
            ax.set_ylabel("Density")
            ax.set_title(f"{name} - {feat_name}")
            if row == 0 and col == 0:
                ax.legend()

    plt.tight_layout()
    plt.suptitle("Stratified Split: Feature Distribution Preservation", y=1.02)
    plt.show()


combined_loader_mode2 = CombinedJetDataLoader(
    pf_data_path=CONFIG_PART["training"]["data"]["pf_data_path"],
    puppi_data_path=config_part["training"]["data"]["puppi_data_path"],
    val_split=config_part["training"]["data"]["val_split"],
    batch_size=config_part["training"]["batch_size"],
    match_mode=None,
    num_workers=config_part["training"]["data"]["num_workers"],
    random_state=42,
)
# Run the verification
verify_distribution_preservation(combined_loader_mode2)

In [ ]:
plt.hist(jet_pt, bins=50, range=(0, 500), histtype="step", density=True)

train_loader_pf_comb, train_loader_puppi_comb = (
    combined_loader_mode2.get_train_loaders()
)
val_loader_pf_comb, val_loader_puppi_comb = combined_loader_mode2.get_val_loaders()
all_constituents_pf_comb_train = []
all_constituents_pf_comb_val = []
for batch in train_loader_pf_comb:
    all_constituents_pf_comb_train.append(batch[0].numpy())
for batch in val_loader_pf_comb:
    all_constituents_pf_comb_val.append(batch[0].numpy())
all_constituents_pf_comb_train = np.concatenate(all_constituents_pf_comb_train, axis=0)
all_constituents_pf_comb_val = np.concatenate(all_constituents_pf_comb_val, axis=0)

const_pf_comb_train_4_vec = vector.array(
    {
        "pt": all_constituents_pf_comb_train[:, :, 1],
        "eta": all_constituents_pf_comb_train[:, :, 2],
        "phi": all_constituents_pf_comb_train[:, :, 3],
        "mass": all_constituents_pf_comb_train[:, :, 0],
    }
)
const_pf_comb_val_4_vec = vector.array(
    {
        "pt": all_constituents_pf_comb_val[:, :, 1],
        "eta": all_constituents_pf_comb_val[:, :, 2],
        "phi": all_constituents_pf_comb_val[:, :, 3],
        "mass": all_constituents_pf_comb_val[:, :, 0],
    }
)
jet_pf_comb_train_4_vec = const_pf_comb_train_4_vec.sum(axis=1)
jet_pf_comb_val_4_vec = const_pf_comb_val_4_vec.sum(axis=1)
plt.hist(
    jet_pt,
    bins=50,
    range=(0, 500),
    histtype="step",
    density=True,
    label="Full dataset val",
)
plt.hist(
    jet_vectors_train.pt,
    bins=50,
    range=(0, 500),
    histtype="step",
    density=True,
    label="Full dataset Train",
)
plt.hist(
    jet_pf_comb_train_4_vec.pt,
    bins=50,
    range=(0, 500),
    histtype="step",
    density=True,
    label="Downsampled Train",
)
plt.hist(
    jet_pf_comb_val_4_vec.pt,
    bins=50,
    range=(0, 500),
    histtype="step",
    density=True,
    label="Downsampled Val",
)
plt.legend()
plt.show()

## H-Tagging ROC and AUC Heatmaps (AK8)

Use the **existing validation dataloader** (which already contains AK8 signal jets matched to gen b-quarks from Higgs and QCD background jets with cross-section weights) to compute:

1. H-tagging ROC curve (ParT H-tag score, signal = H-matched AK8 jets, background = QCD AK8 jets)
2. Per-bin AUC heatmaps in (pT, $|\eta|$) bins
3. Per-bin signal efficiency heatmaps at tight/medium/loose working points

In [ ]:
# ============================================================
# H-TAGGING ROC CURVE AND AUC HEATMAPS (AK8)
# ============================================================
# Uses existing validation outputs from earlier cells:
#   all_outputs         : ParT H-tag sigmoid scores (val set)
#   all_labels          : 1 = H-matched AK8 jet, 0 = QCD AK8 jet
#   all_qcd_weights_val : QCD cross-section weights (signal jets = 1)
#   val_jet_pt, val_jet_eta : reconstructed jet kinematics
# ============================================================

from sklearn.metrics import roc_curve, auc, roc_auc_score
import matplotlib.colors as mcolors
import os
import numpy as np
import matplotlib.pyplot as plt

# --- 1. H-tagging ROC curve ---
htag_labels = all_labels.reshape(-1)
htag_scores = all_outputs.reshape(-1)
htag_weights = all_qcd_weights_val.reshape(-1)

htag_fpr, htag_tpr, htag_thresholds = roc_curve(
    htag_labels, htag_scores, sample_weight=htag_weights
)
htag_auc = auc(htag_fpr, htag_tpr)

# Linear ROC
fig1, ax1 = plt.subplots(figsize=(9, 7))
ax1.plot(
    htag_fpr,
    htag_tpr,
    color="darkred",
    linewidth=2,
    label=f"ParT H-tag (AUC = {htag_auc:.4f})",
)
ax1.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax1.set_xlabel("False Positive Rate (QCD mistag)")
ax1.set_ylabel("True Positive Rate (H-tag efficiency)")
ax1.set_title("H-Tagging ROC Curve")
ax1.legend(loc="lower right")
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

# Log-x ROC matched to b-tagging cell 14 convention
fig2, ax2 = plt.subplots(figsize=(9, 7))
valid = htag_fpr > 0
ax2.plot(
    htag_fpr[valid],
    htag_tpr[valid],
    color="darkred",
    linewidth=2,
    label=f"ParT H-tag (AUC = {htag_auc:.4f})",
)
ax2.set_xlabel("Mistag Rate")
ax2.set_ylabel("H-Tagging Efficiency")
ax2.set_xscale("log")
ax2.set_xlim(1e-4, 1.0)
ax2.set_ylim(1e-4, 1.05)
ax2.set_title("H-Tagging ROC Curve")
ax2.legend(loc="lower right")
ax2.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.show()


# --- H-tag working points ---
def get_htag_wp(fpr_arr, tpr_arr, thresh_arr, target_mistag):
    idx = np.argmin(np.abs(fpr_arr - target_mistag))
    return fpr_arr[idx], tpr_arr[idx], thresh_arr[idx]


htag_wps = []
print(f"\nParT H-Tag Working Points (AUC = {htag_auc:.5f}):")
for wp_name, target in [("Tight", 0.001), ("Medium", 0.01), ("Loose", 0.1)]:
    fpr_wp, tpr_wp, thresh_wp = get_htag_wp(htag_fpr, htag_tpr, htag_thresholds, target)
    print(
        f"  {wp_name}: TPR={tpr_wp*100:.2f}%, FPR={fpr_wp:.4f}, "
        f"1/FPR={1/max(fpr_wp,1e-9):.1f}, thresh={thresh_wp:.4f}"
    )
    htag_wps.append(thresh_wp)

# --- 2. Per-bin AUC heatmap ---
# Binning matched to b-tagging cell 14 (for loose visual comparison on same axes)
htag_pt = val_jet_pt
htag_eta = val_jet_eta

pt_ranges_hm = [
    (25, 100),
    (100, 200),
    (200, 300),
    (300, 400),
    (400, 500),
    (500, np.inf),
    (25, np.inf),
]
eta_ranges_hm = [
    (0, 0.5),
    (0.5, 1.0),
    (1.0, 1.5),
    (1.5, 2.4),
    (0, 1.5),
    (0, 2.4),
]

n_pt = len(pt_ranges_hm)
n_eta = len(eta_ranges_hm)

auc_grid = np.full((n_pt, n_eta), np.nan)
count_grid = np.zeros((n_pt, n_eta), dtype=int)

for i, (pt_low, pt_high) in enumerate(pt_ranges_hm):
    for j, (eta_low, eta_high) in enumerate(eta_ranges_hm):
        mask = (
            (htag_pt >= pt_low)
            & (htag_pt < pt_high)
            & (np.abs(htag_eta) >= eta_low)
            & (np.abs(htag_eta) < eta_high)
        )
        bin_y = htag_labels[mask]
        bin_s = htag_scores[mask]
        bin_w = htag_weights[mask]
        count_grid[i, j] = len(bin_y)

        if len(np.unique(bin_y)) < 2 or len(bin_y) < 20:
            continue
        try:
            auc_grid[i, j] = roc_auc_score(bin_y, bin_s, sample_weight=bin_w)
        except ValueError:
            pass

pt_labels = [f"[{low},{high})" for low, high in pt_ranges_hm]
eta_labels = [f"[{low},{high})" for low, high in eta_ranges_hm]

# AUC heatmap
fig3, ax3 = plt.subplots(figsize=(9, 7))
im = ax3.imshow(auc_grid, aspect="auto", origin="lower", vmin=0.5, vmax=1.0)
for i in range(n_pt):
    for j in range(n_eta):
        val = auc_grid[i, j]
        txt = f"{val:.3f}" if not np.isnan(val) else "—"
        ax3.text(j, i, txt, ha="center", va="center", fontsize=12)
ax3.set_xticks(range(n_eta))
ax3.set_xticklabels(eta_labels, rotation=30, fontsize=12)
ax3.set_yticks(range(n_pt))
ax3.set_yticklabels(pt_labels, fontsize=12)
ax3.set_xlabel(r"$|\eta|$ bin")
ax3.set_ylabel(r"$p_T$ bin [GeV]")
ax3.set_title("H-Tag AUC per ($p_T$, $|\eta|$) bin")
plt.colorbar(im, ax=ax3, label="AUC")
plt.tight_layout()
plt.show()

# Jet count heatmap
fig4, ax4 = plt.subplots(figsize=(9, 7))
positive_counts = count_grid[count_grid > 0]
vmin_counts = max(1, positive_counts.min()) if positive_counts.size > 0 else 1
vmax_counts = max(1, count_grid.max())
im2 = ax4.imshow(
    count_grid,
    aspect="auto",
    origin="lower",
    norm=mcolors.LogNorm(vmin=vmin_counts, vmax=vmax_counts),
)
for i in range(n_pt):
    for j in range(n_eta):
        ax4.text(
            j,
            i,
            f"{count_grid[i, j]:,}",
            ha="center",
            va="center",
            fontsize=12,
            color="white",
        )
ax4.set_xticks(range(n_eta))
ax4.set_xticklabels(eta_labels, rotation=30, fontsize=12)
ax4.set_yticks(range(n_pt))
ax4.set_yticklabels(pt_labels, fontsize=12)
ax4.set_xlabel(r"$|\eta|$ bin")
ax4.set_ylabel(r"$p_T$ bin [GeV]")
ax4.set_title("Jet count per bin (signal + background)")
plt.colorbar(im2, ax=ax4, label="Count")
plt.tight_layout()
plt.show()

# --- 3. Signal efficiency heatmaps at each WP ---
wp_names = ["Tight", "Medium", "Loose"]
eff_heatmap_figs = []

for wp_idx, wp_name in enumerate(wp_names):
    fig_eff, ax5 = plt.subplots(figsize=(9, 7))
    thresh = htag_wps[wp_idx]
    eff_grid = np.full((n_pt, n_eta), np.nan)
    rej_grid = np.full((n_pt, n_eta), np.nan)

    for i, (pt_low, pt_high) in enumerate(pt_ranges_hm):
        for j, (eta_low, eta_high) in enumerate(eta_ranges_hm):
            mask = (
                (htag_pt >= pt_low)
                & (htag_pt < pt_high)
                & (np.abs(htag_eta) >= eta_low)
                & (np.abs(htag_eta) < eta_high)
            )
            bin_y = htag_labels[mask]
            bin_s = htag_scores[mask]
            bin_w = htag_weights[mask]

            sig_mask_bin = bin_y == 1
            bkg_mask_bin = bin_y == 0

            if sig_mask_bin.sum() > 0:
                eff_grid[i, j] = np.mean(bin_s[sig_mask_bin] >= thresh)

            if bkg_mask_bin.sum() > 0:
                bkg_pass = np.sum(bin_w[bkg_mask_bin] * (bin_s[bkg_mask_bin] >= thresh))
                bkg_total = np.sum(bin_w[bkg_mask_bin])
                bkg_eff = bkg_pass / bkg_total if bkg_total > 0 else 0.0
                rej_grid[i, j] = 1.0 / bkg_eff if bkg_eff > 0 else np.inf

    im = ax5.imshow(eff_grid, aspect="auto", origin="lower", vmin=0, vmax=1)
    for i in range(n_pt):
        for j in range(n_eta):
            val = eff_grid[i, j]
            if not np.isnan(val):
                rej = rej_grid[i, j]
                rej_str = f"\n1/FPR={rej:.0f}" if np.isfinite(rej) and rej < 1e6 else ""
                ax5.text(
                    j,
                    i,
                    f"{val:.2f}{rej_str}",
                    ha="center",
                    va="center",
                    fontsize=12,
                    color="white",
                )
            else:
                ax5.text(j, i, "—", ha="center", va="center", fontsize=12)

    ax5.set_xticks(range(n_eta))
    ax5.set_xticklabels(eta_labels, rotation=30, fontsize=12)
    ax5.set_yticks(range(n_pt))
    ax5.set_yticklabels(pt_labels, fontsize=12)
    ax5.set_xlabel(r"$|\eta|$ bin")
    ax5.set_ylabel(r"$p_T$ bin [GeV]")
    ax5.set_title(f"H-Tag Signal Efficiency — {wp_name} (thresh={thresh:.4f})")
    plt.colorbar(im, ax=ax5, label="Signal Efficiency")
    plt.tight_layout()
    plt.show()

    eff_heatmap_figs.append((wp_name, fig_eff))

# Save plots
try:
    run_id_htag = config_part.get("exp_name", "unknown_run").replace("/", "_")
    artifact_name_htag = CONFIG_PART.get("wandb", {}).get(
        "artifact_name", "unknown_artifact"
    )
    artifact_type_htag = CONFIG_PART.get("wandb", {}).get("ckpt_type", "unknown_type")
    plot_dir_htag = (
        f"../Updates/plots_{run_id_htag}/{artifact_name_htag}:{artifact_type_htag}"
    )
    os.makedirs(plot_dir_htag, exist_ok=True)

    fig1.savefig(
        os.path.join(plot_dir_htag, "htag_roc_linear.png"), dpi=150, bbox_inches="tight"
    )
    fig2.savefig(
        os.path.join(plot_dir_htag, "htag_roc_log.png"), dpi=150, bbox_inches="tight"
    )
    fig3.savefig(
        os.path.join(plot_dir_htag, "htag_auc_heatmap.png"),
        dpi=150,
        bbox_inches="tight",
    )
    fig4.savefig(
        os.path.join(plot_dir_htag, "htag_count_heatmap.png"),
        dpi=150,
        bbox_inches="tight",
    )

    for wp_name, fig_eff in eff_heatmap_figs:
        fig_eff.savefig(
            os.path.join(plot_dir_htag, f"htag_efficiency_heatmap_{wp_name}.png"),
            dpi=150,
            bbox_inches="tight",
        )

    print(f"Saved all H-tag individual fully-sized plots to {plot_dir_htag}")
except Exception as e:
    print(f"Could not save plots: {e}")

## Di-Higgs Mass Reconstruction with ParT H-Tag Scores (AK8)

Cluster L1 PUPPI/PF candidates with **anti-$k_T$ R=0.8** (AK8), extract constituent features, score each jet with the trained ParT H-tagger, apply $p_T$ correction from the regression head, and reconstruct $m_{H_1}$, $m_{H_2}$, $m_{HH}$.

Since this is a **Higgs tagger**, only **2 AK8 jets** are needed per event (one per Higgs). Jets are ranked by H-tag score and the top 2 passing the working-point threshold are selected. The leading/subleading Higgs are ordered by $p_T$.

In [ ]:
# ============================================================
# DI-HIGGS MASS RECONSTRUCTION WITH ParT H-TAG (AK8)
# ============================================================
# Step 1: Load ROOT data, cluster L1 PUPPI/PF candidates with AK8 (R=0.8)
# Step 2: Extract ParT features per jet, run inference
# Step 3: Apply H-tag WP cut + pT regression correction
# Step 4: Select top-2 H-tagged jets per event → mH1, mH2, mHH
#   Signal: from HH→4b ROOT files (top-2 H-tagged, cross-matched to gen Higgs)
#   Background: from QCD pT-binned ROOT files (top-2 H-tagged above WP)
# ============================================================

import os
import gc
import fastjet
import glob as glob_module
from data_pipeline.make_particle_dataset import cluster_candidates
from data_pipeline.root_loading import (
    load_and_prepare_data,
    select_gen_higgs,
    select_gen_b_quarks_from_higgs,
    apply_custom_cuts,
    one_hot_encode_l1_puppi,
)
from evaluation.jet_matching import get_purity_mask_cross_matched

# --- Configuration ---
apply_pt_correction_htag = True
HTAG_WP_SELECTION = "medium"  # Choose from: "tight", "medium", "loose"
htag_wp_index = {"tight": 0, "medium": 1, "loose": 2}[HTAG_WP_SELECTION]
HTAG_THRESHOLD = htag_wps[htag_wp_index]
AK8_DIST_PARAM = 0.8
N_CONSTITUENTS_AK8 = all_constituents.shape[1]  # match model input

print(f"H-Tag Working point: {HTAG_WP_SELECTION} → threshold = {HTAG_THRESHOLD:.4f}")
print(f"AK8 distance parameter: {AK8_DIST_PARAM}")
print(f"Constituents per jet: {N_CONSTITUENTS_AK8}")

# Use the same dataset config the model was trained on
dataset_used_htag = (
    config_part.get("training", {}).get("data", {}).get("use_dataset", "pf")
)
if dataset_used_htag == "pf":
    collection_key_htag = "l1extpf"
elif dataset_used_htag == "puppi":
    collection_key_htag = "l1extpuppi"
else:
    collection_key_htag = "l1barrelextpf"
print(f"Model trained on: {dataset_used_htag} → clustering {collection_key_htag}")

collection_name_htag = config[collection_key_htag]["collection_name"]


# ============================================================
# HELPER: cluster AK8 → extract features → ParT inference → scored jets
# ============================================================
def cluster_and_score_ak8(
    events,
    cfg,
    collection_key,
    model,
    device,
    config_part,
    n_constituents,
    apply_pt_correction=True,
):
    """
    Cluster L1 candidates with AK8 (R=0.8), extract ParT features,
    run inference, apply pT regression correction, and return scored
    jets with event structure preserved.
    """
    # Cluster with AK8
    clustered_jets = cluster_candidates(
        events, cfg, collection_key, dist_param=AK8_DIST_PARAM
    )
    sorted_indices = ak.argsort(clustered_jets.pt, axis=1, ascending=False)
    l1_clustered = clustered_jets[sorted_indices]
    del clustered_jets, sorted_indices
    matched_cands = l1_clustered.constituents
    const_pt_sort = ak.argsort(matched_cands.pt, axis=2, ascending=False)
    matched_cands = matched_cands[const_pt_sort]
    del const_pt_sort

    # Extract features (same as b-tag version)
    j_pt = l1_clustered.pt[:, :, None]
    j_eta = l1_clustered.eta[:, :, None]
    j_phi = l1_clustered.phi[:, :, None]

    m_pt = matched_cands.vector.pt
    m_eta = matched_cands.vector.eta
    m_phi = matched_cands.vector.phi
    m_mass = matched_cands.vector.mass
    m_dxy = matched_cands.dxy
    m_z0 = matched_cands.z0
    m_charge = matched_cands.charge
    m_w = matched_cands.puppiWeight
    m_id = matched_cands.id

    log_pt_rel = np.log(np.maximum(m_pt, 1e-3) / np.maximum(j_pt, 1e-3))
    deta = m_eta - j_eta
    dphi = m_phi - j_phi
    dphi = (dphi + np.pi) % (2 * np.pi) - np.pi
    log_dr = np.log(np.maximum(np.sqrt(deta**2 + dphi**2), 1e-3))

    def pad_and_fill(arr, target=n_constituents):
        return ak.fill_none(ak.pad_none(arr, target, axis=2, clip=True), 0.0)

    feature_list = [
        pad_and_fill(m_mass),
        pad_and_fill(m_pt),
        pad_and_fill(m_eta),
        pad_and_fill(m_phi),
        pad_and_fill(m_dxy),
        pad_and_fill(m_z0),
        pad_and_fill(m_charge),
        pad_and_fill(log_pt_rel),
        pad_and_fill(deta),
        pad_and_fill(dphi),
        pad_and_fill(m_w),
        pad_and_fill(log_dr),
        pad_and_fill(m_id),
    ]
    del m_pt, m_eta, m_phi, m_mass, m_dxy, m_z0, m_charge, m_w, m_id
    del log_pt_rel, deta, dphi, log_dr, j_pt, j_eta, j_phi

    n_jets_per_event = ak.num(l1_clustered, axis=1)
    n_actual_constituents = ak.num(matched_cands, axis=2)
    n_actual_flat = ak.to_numpy(ak.flatten(n_actual_constituents, axis=1))
    del matched_cands, n_actual_constituents

    x_ini = np.stack(
        [ak.to_numpy(ak.flatten(f, axis=1)) for f in feature_list], axis=-1
    )
    del feature_list

    flat_ids = x_ini[..., -1]
    one_hot_ids = one_hot_encode_l1_puppi(flat_ids, n_classes=5)
    X_feat = np.concatenate([x_ini[..., :-1], one_hot_ids], axis=-1)
    del flat_ids, one_hot_ids

    particle_mask = np.zeros((X_feat.shape[0], n_constituents), dtype=bool)
    for i in range(X_feat.shape[0]):
        n_real = min(n_actual_flat[i], n_constituents)
        particle_mask[i, :n_real] = True
    del n_actual_flat

    # Jet 4-vectors from constituent sum
    const_vecs = vector.array(
        {
            "pt": x_ini[:, :, 1],
            "eta": x_ini[:, :, 2],
            "phi": x_ini[:, :, 3],
            "mass": x_ini[:, :, 0],
        }
    )
    del x_ini
    jet_4v = const_vecs.sum(axis=1)
    del const_vecs
    flat_jet_pt = np.array(jet_4v.pt)
    flat_jet_eta = np.array(jet_4v.eta)
    flat_jet_phi = np.array(jet_4v.phi)
    flat_jet_mass = np.array(jet_4v.mass)
    del jet_4v
    gc.collect()

    # Inference
    batch_size = config_part.get("training", {}).get("batch_size", 512)
    all_scores, all_reg = [], []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(X_feat), batch_size):
            end = min(start + batch_size, len(X_feat))
            xb = torch.tensor(X_feat[start:end], dtype=torch.float32).to(device)
            mb = torch.tensor(particle_mask[start:end], dtype=torch.bool).to(device)
            out = model(xb, particle_mask=mb)
            scores = (
                torch.nn.functional.sigmoid(out["classification"])
                .squeeze()
                .cpu()
                .numpy()
            )
            all_scores.append(scores)
            if "pt" in out:
                all_reg.append(out["pt"].squeeze().cpu().numpy())
    del X_feat, particle_mask
    gc.collect()

    all_scores = np.concatenate(all_scores)
    has_reg = len(all_reg) > 0
    if has_reg:
        all_reg = np.concatenate(all_reg)

    # pT correction
    if has_reg and apply_pt_correction:
        corrected_pt = flat_jet_pt * all_reg
    else:
        corrected_pt = flat_jet_pt
    del all_reg

    corr_vecs = vector.array(
        {
            "pt": corrected_pt,
            "eta": flat_jet_eta,
            "phi": flat_jet_phi,
            "mass": flat_jet_mass * (corrected_pt / (flat_jet_pt + 1e-9)),
        }
    )
    del corrected_pt, flat_jet_pt, flat_jet_eta, flat_jet_phi, flat_jet_mass

    # Rebuild event structure
    n_jets_np = ak.to_numpy(n_jets_per_event)
    del l1_clustered, n_jets_per_event
    cumulative = np.concatenate([[0], np.cumsum(n_jets_np)])
    evt_pts, evt_etas, evt_phis, evt_masses, evt_scores = [], [], [], [], []
    for i in range(len(n_jets_np)):
        s, e = cumulative[i], cumulative[i + 1]
        evt_pts.append(corr_vecs.pt[s:e])
        evt_etas.append(corr_vecs.eta[s:e])
        evt_phis.append(corr_vecs.phi[s:e])
        evt_masses.append(corr_vecs.mass[s:e])
        evt_scores.append(all_scores[s:e])
    del corr_vecs, all_scores, cumulative, n_jets_np

    scored_jets = ak.zip(
        {
            "pt": ak.Array(evt_pts),
            "eta": ak.Array(evt_etas),
            "phi": ak.Array(evt_phis),
            "mass": ak.Array(evt_masses),
            "htag_score": ak.Array(evt_scores),
        }
    )
    del evt_pts, evt_etas, evt_phis, evt_masses, evt_scores
    scored_jets["vector"] = ak.zip(
        {
            "pt": scored_jets.pt,
            "eta": scored_jets.eta,
            "phi": scored_jets.phi,
            "mass": scored_jets.mass,
        },
        with_name="Momentum4D",
    )
    return scored_jets, has_reg


# ================================================================
# SIGNAL  — from HH→4b ROOT files (AK8 clustering, chunked)
# ================================================================
print("=" * 60)
print("Processing SIGNAL from HH→4b (AK8 H-tag)...")
print("=" * 60)
root_data_pattern = config["file_pattern"]
SIGNAL_CHUNK_SIZE = 20000
max_signal_events = 200000
print(f"ROOT data: {root_data_pattern}")
print(f"Collection: {collection_name_htag}")
print(f"Chunk size: {SIGNAL_CHUNK_SIZE}")

scored_jets_chunks_htag = []
gen_higgs_chunks = []
offset = 0
events_remaining = max_signal_events
chunk_idx = 0

while events_remaining > 0:
    chunk_size = min(SIGNAL_CHUNK_SIZE, events_remaining)
    try:
        chunk_events = load_and_prepare_data(
            root_data_pattern,
            config["tree_name"],
            [collection_name_htag, "GenPart"],
            max_events=chunk_size,
            correct_pt=False,
            CONFIG=config,
            entry_start=offset,
        )
    except Exception as e:
        print(f"  Error loading signal chunk {chunk_idx}: {e}")
        break

    n_loaded = len(chunk_events)
    if n_loaded == 0:
        del chunk_events
        gc.collect()
        break

    events_remaining -= n_loaded
    offset += n_loaded
    chunk_idx += 1

    # Cluster AK8 and score
    chunk_scored, has_reg_htag = cluster_and_score_ak8(
        chunk_events,
        config,
        collection_key_htag,
        model,
        device,
        config_part,
        N_CONSTITUENTS_AK8,
        apply_pt_correction_htag,
    )
    scored_jets_chunks_htag.append(chunk_scored)

    # Extract gen Higgs for cross-matching
    chunk_gen_higgs = select_gen_higgs(chunk_events)
    gen_higgs_chunks.append(chunk_gen_higgs)

    del chunk_events, chunk_scored, chunk_gen_higgs
    gc.collect()
    print(f"  Chunk {chunk_idx}: {n_loaded} events, {offset} total so far")

# Concatenate
scored_jets_htag = ak.concatenate(scored_jets_chunks_htag)
gen_higgs_all = ak.concatenate(gen_higgs_chunks)
del scored_jets_chunks_htag, gen_higgs_chunks
gc.collect()

n_events_total_htag = len(scored_jets_htag)
n_jets_total_htag = int(ak.sum(ak.num(scored_jets_htag, axis=1)))
print(
    f"Clustered & scored {n_jets_total_htag} AK8 jets across {n_events_total_htag} events"
)

# Sort jets by H-tag score (descending) and select top 2 above WP
jets_htag_sorted = scored_jets_htag[
    ak.argsort(scored_jets_htag.htag_score, ascending=False)
]
del scored_jets_htag
jets_htag_pass = jets_htag_sorted[jets_htag_sorted.htag_score > HTAG_THRESHOLD]
del jets_htag_sorted
print(f"Jets per event after H-tag cut: mean={ak.mean(ak.num(jets_htag_pass)):.1f}")

has_2_tagged = ak.num(jets_htag_pass) >= 2
sig_jets_2 = jets_htag_pass[has_2_tagged][:, :2]
del jets_htag_pass
print(f"Events with >=2 H-tagged AK8 jets: {ak.sum(has_2_tagged)}/{len(has_2_tagged)}")

# Cross-match AK8 jets to gen Higgs (dR matching)
# Signal = both AK8 jets match to distinct gen Higgs
gen_higgs_for_match = gen_higgs_all[has_2_tagged]
dr_reco_h = sig_jets_2[:, :, None].vector.deltaR(gen_higgs_for_match[:, None, :].vector)
idx_gen_for_reco_h = ak.argmin(dr_reco_h, axis=2)
min_dr_reco_h = ak.fill_none(ak.min(dr_reco_h, axis=2), np.inf)

dr_gen_h = gen_higgs_for_match[:, :, None].vector.deltaR(sig_jets_2[:, None, :].vector)
idx_reco_for_gen_h = ak.argmin(dr_gen_h, axis=2)
del dr_reco_h, dr_gen_h

back_check_h = idx_reco_for_gen_h[idx_gen_for_reco_h]
reco_idx_h = ak.local_index(sig_jets_2, axis=1)

# Use a larger cone for AK8 matching
ak8_matching_cone = AK8_DIST_PARAM
pure_mask_h = (ak.fill_none(back_check_h, -1) == reco_idx_h) & (
    min_dr_reco_h < ak8_matching_cone
)
del back_check_h, idx_reco_for_gen_h, min_dr_reco_h, reco_idx_h

# Signal = both jets cross-matched to distinct gen Higgs
signal_mask_evt_htag = ak.sum(pure_mask_h, axis=1) == 2
del pure_mask_h
n_signal_htag = int(ak.sum(signal_mask_evt_htag))
n_total_htag = int(ak.sum(has_2_tagged))
print(f"Signal events (both AK8 jets pure): {n_signal_htag}")
print(f"Total events with >=2 H-tagged jets: {n_total_htag}")

# Reconstruct di-Higgs: leading and subleading by pT
sig_jets_htag_2 = sig_jets_2[signal_mask_evt_htag][:, :2]
del sig_jets_2

if n_signal_htag > 0:
    h1_vec = sig_jets_htag_2[:, 0].vector
    h2_vec = sig_jets_htag_2[:, 1].vector

    # Order by pT: leading = higher pT
    is_lead = h1_vec.pt >= h2_vec.pt
    sig_lead_htag = ak.where(is_lead, h1_vec, h2_vec)
    sig_sub_htag = ak.where(is_lead, h2_vec, h1_vec)
    sig_hh_htag = h1_vec + h2_vec

    print(
        f"\nSignal mH (lead): mean={ak.mean(sig_lead_htag.mass):.1f}, "
        f"median={np.median(ak.to_numpy(sig_lead_htag.mass)):.1f} GeV"
    )
    print(
        f"Signal mH (sub):  mean={ak.mean(sig_sub_htag.mass):.1f}, "
        f"median={np.median(ak.to_numpy(sig_sub_htag.mass)):.1f} GeV"
    )
    print(
        f"Signal mHH: mean={ak.mean(sig_hh_htag.mass):.1f}, "
        f"median={np.median(ak.to_numpy(sig_hh_htag.mass)):.1f} GeV"
    )
else:
    sig_lead_htag = sig_sub_htag = sig_hh_htag = ak.Array([])
    print("\nNo signal events found!")

del (
    sig_jets_htag_2,
    gen_higgs_for_match,
    gen_higgs_all,
    has_2_tagged,
    signal_mask_evt_htag,
)
gc.collect()


# ================================================================
# QCD BACKGROUND — from QCD pT-binned ROOT files (AK8, chunked)
# ================================================================
print("\n" + "=" * 60)
print("Processing QCD BACKGROUND (AK8 H-tag)...")
print("=" * 60)
QCD_CHUNK_SIZE_HTAG = 5000
qcd_config_htag = config["QCD_background"]
all_qcd_lead_htag, all_qcd_sub_htag, all_qcd_hh_htag = [], [], []
all_qcd_weights_htag_list = []
n_qcd_total_htag = 0
n_qcd_events_processed_htag = 0

for bin_name, bin_cfg in qcd_config_htag.items():
    print(f"\n--- QCD bin: {bin_name}  (weight={bin_cfg['weight']:.3e}) ---")
    qcd_file_pattern = bin_cfg["file_pattern"]
    max_events_bin = 20000

    qcd_cfg_htag = dict(config)
    qcd_cfg_htag["file_pattern"] = qcd_file_pattern
    qcd_cfg_htag["tree_name"] = bin_cfg["tree_name"]

    offset = 0
    events_remaining = max_events_bin
    chunk_idx = 0

    while events_remaining > 0:
        chunk_size = min(QCD_CHUNK_SIZE_HTAG, events_remaining)
        try:
            qcd_events = load_and_prepare_data(
                qcd_file_pattern,
                bin_cfg["tree_name"],
                [collection_name_htag],
                max_events=chunk_size,
                correct_pt=False,
                CONFIG=qcd_cfg_htag,
                entry_start=offset,
            )
        except Exception as e:
            print(f"  Error loading {bin_name} chunk {chunk_idx}: {e}")
            break

        n_loaded = len(qcd_events)
        if n_loaded == 0:
            del qcd_events
            gc.collect()
            break

        n_qcd_events_processed_htag += n_loaded
        events_remaining -= n_loaded
        offset += n_loaded
        chunk_idx += 1

        qcd_scored_htag, _ = cluster_and_score_ak8(
            qcd_events,
            qcd_cfg_htag,
            collection_key_htag,
            model,
            device,
            config_part,
            N_CONSTITUENTS_AK8,
            apply_pt_correction_htag,
        )
        del qcd_events
        gc.collect()

        # Apply same selection: top-2 H-tagged jets above WP
        qcd_htag_sorted = qcd_scored_htag[
            ak.argsort(qcd_scored_htag.htag_score, ascending=False)
        ]
        del qcd_scored_htag
        qcd_htag_pass = qcd_htag_sorted[qcd_htag_sorted.htag_score > HTAG_THRESHOLD]
        del qcd_htag_sorted
        has_2_tagged_qcd = ak.num(qcd_htag_pass) >= 2
        qcd_2jets = qcd_htag_pass[has_2_tagged_qcd][:, :2]

        n_events_chunk = int(ak.sum(has_2_tagged_qcd))
        del qcd_htag_pass, has_2_tagged_qcd

        if n_events_chunk > 0:
            q_h1 = qcd_2jets[:, 0].vector
            q_h2 = qcd_2jets[:, 1].vector
            q_is_lead = q_h1.pt >= q_h2.pt
            q_lead = ak.where(q_is_lead, q_h1, q_h2)
            q_sub = ak.where(q_is_lead, q_h2, q_h1)
            q_hh = q_h1 + q_h2

            all_qcd_lead_htag.append(q_lead)
            all_qcd_sub_htag.append(q_sub)
            all_qcd_hh_htag.append(q_hh)
            all_qcd_weights_htag_list.append(
                np.full(n_events_chunk, bin_cfg["weight"], dtype=np.float64)
            )
            n_qcd_total_htag += n_events_chunk

        del qcd_2jets
        gc.collect()

    print(
        f"  {bin_name}: {chunk_idx} chunks, {offset} events loaded, "
        f"{n_qcd_total_htag} total reconstructed so far"
    )

# Combine QCD results
if n_qcd_total_htag > 0:
    qcd_lead_htag = ak.concatenate(all_qcd_lead_htag)
    qcd_sub_htag = ak.concatenate(all_qcd_sub_htag)
    qcd_hh_htag = ak.concatenate(all_qcd_hh_htag)
    qcd_weights_htag = np.concatenate(all_qcd_weights_htag_list)
    print(
        f"\nTotal QCD: {n_qcd_total_htag} reconstructed events from "
        f"{n_qcd_events_processed_htag} processed"
    )
    print(
        f"QCD weights: min={qcd_weights_htag.min():.1e}, max={qcd_weights_htag.max():.1e}"
    )
    print(
        f"QCD mHH: weighted mean={np.average(ak.to_numpy(qcd_hh_htag.mass), weights=qcd_weights_htag):.1f}, "
        f"unweighted median={np.median(ak.to_numpy(qcd_hh_htag.mass)):.1f} GeV"
    )
else:
    qcd_lead_htag = qcd_sub_htag = qcd_hh_htag = ak.Array([])
    qcd_weights_htag = np.array([], dtype=np.float64)
    print("\nNo QCD background events found!")

del all_qcd_lead_htag, all_qcd_sub_htag, all_qcd_hh_htag, all_qcd_weights_htag_list

# Build sigma → n_gen mapping for luminosity scaling
sigma_to_ngen_htag = {
    bin_cfg["weight"]: bin_cfg["n_gen"] for bin_cfg in qcd_config_htag.values()
}
gc.collect()

# Store results
htag_dihiggs_result = {
    "label": f"ParT H-Tag ({HTAG_WP_SELECTION})",
    "n_total": n_total_htag,
    "n_signal": n_signal_htag,
    "n_qcd": n_qcd_total_htag,
    "sig_lead": sig_lead_htag,
    "sig_sub": sig_sub_htag,
    "sig_hh": sig_hh_htag,
    "qcd_lead": qcd_lead_htag,
    "qcd_sub": qcd_sub_htag,
    "qcd_hh": qcd_hh_htag,
    "qcd_weights": qcd_weights_htag,  # raw σ_bin (Convention C)
    "sigma_to_ngen": sigma_to_ngen_htag,
    "collection_key": collection_key_htag,
    "wp": HTAG_WP_SELECTION,
    "threshold": HTAG_THRESHOLD,
    "has_regression": has_reg_htag,
}

print(f"\n{'='*60}")
print(f"Di-Higgs reconstruction (H-tag, AK8) complete")
print(f"  Collection: {collection_key_htag} ({dataset_used_htag})")
print(f"  Working point: {HTAG_WP_SELECTION} (threshold={HTAG_THRESHOLD:.4f})")
print(f"  Signal (both AK8 jets pure): {n_signal_htag} events")
print(f"  QCD background: {n_qcd_total_htag} events")
print(f"  pT regression: {'applied' if has_reg_htag else 'not available'}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# DI-HIGGS MASS DISTRIBUTION PLOTS (H-TAG, AK8) + SAVE
# ============================================================

# Full-size H-tag di-Higgs plots + significance (S/sqrt(S+B))
from evaluation.dihiggs import compute_significance_at_luminosity
from evaluation.luminosity import signal_weight, scale_qcd_weights_raw

res_h = htag_dihiggs_result
label_h = res_h["label"]
color_h = "teal"
qcd_w_h = res_h.get("qcd_weights", np.ones(res_h["n_qcd"]))
sig_wt_h = res_h.get("sig_weights")

sig_window_h = (90, 160)
sig_window_hh = (250, 550)
bins_mh = np.linspace(0, 300, 61)
bins_mhh = np.linspace(200, 800, 61)

run_id_htag = config_part.get("exp_name", "unknown_run").replace("/", "_")
artifact_name_htag = CONFIG_PART.get("wandb", {}).get(
    "artifact_name", "unknown_artifact"
)
artifact_type_htag = CONFIG_PART.get("wandb", {}).get("ckpt_type", "unknown_type")
plot_dir_htag = (
    f"../Updates/plots_{run_id_htag}/{artifact_name_htag}:{artifact_type_htag}"
)
os.makedirs(plot_dir_htag, exist_ok=True)


def save_htag_fig(fig_obj, name):
    out_path = os.path.join(plot_dir_htag, name)
    fig_obj.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved: {out_path}")


# 1) Leading mH
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111)
if res_h["n_signal"] > 0:
    ax.hist(
        ak.to_numpy(res_h["sig_lead"].mass),
        bins=bins_mh,
        histtype="stepfilled",
        alpha=0.3,
        color=color_h,
        label=f'Signal ({res_h["n_signal"]})',
        density=True,
    )
    ax.hist(
        ak.to_numpy(res_h["sig_lead"].mass),
        bins=bins_mh,
        histtype="step",
        linewidth=2,
        color=color_h,
        density=True,
    )
if res_h["n_qcd"] > 0:
    ax.hist(
        ak.to_numpy(res_h["qcd_lead"].mass),
        bins=bins_mh,
        histtype="step",
        linewidth=2,
        color="grey",
        linestyle="--",
        label=f'QCD ({res_h["n_qcd"]} events)',
        density=True,
    )
ax.axvline(125, color="green", linestyle=":", linewidth=1.5)
ax.axvspan(*sig_window_h, alpha=0.05, color="green")
ax.set_xlabel("Leading $m_H$ [GeV]")
ax.set_ylabel("Density")
ax.set_title(f"{label_h} - Leading Higgs (AK8)")
ax.legend(fontsize=10)
plt.tight_layout()
save_htag_fig(fig, "htag_dihiggs_mass_leading_full.png")
plt.show()

# 2) Subleading mH
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111)
if res_h["n_signal"] > 0:
    ax.hist(
        ak.to_numpy(res_h["sig_sub"].mass),
        bins=bins_mh,
        histtype="stepfilled",
        alpha=0.3,
        color=color_h,
        label="Signal",
        density=True,
    )
    ax.hist(
        ak.to_numpy(res_h["sig_sub"].mass),
        bins=bins_mh,
        histtype="step",
        linewidth=2,
        color=color_h,
        density=True,
    )
if res_h["n_qcd"] > 0:
    ax.hist(
        ak.to_numpy(res_h["qcd_sub"].mass),
        bins=bins_mh,
        histtype="step",
        linewidth=2,
        color="grey",
        linestyle="--",
        label="QCD",
        density=True,
    )
ax.axvline(125, color="green", linestyle=":", linewidth=1.5)
ax.axvspan(*sig_window_h, alpha=0.05, color="green")
ax.set_xlabel("Subleading $m_H$ [GeV]")
ax.set_ylabel("Density")
ax.set_title(f"{label_h} - Subleading Higgs (AK8)")
ax.legend(fontsize=10)
plt.tight_layout()
save_htag_fig(fig, "htag_dihiggs_mass_subleading_full.png")
plt.show()

# 3) mHH
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111)
if res_h["n_signal"] > 0:
    ax.hist(
        ak.to_numpy(res_h["sig_hh"].mass),
        bins=bins_mhh,
        histtype="stepfilled",
        alpha=0.3,
        color=color_h,
        label="Signal",
        density=True,
    )
    ax.hist(
        ak.to_numpy(res_h["sig_hh"].mass),
        bins=bins_mhh,
        histtype="step",
        linewidth=2,
        color=color_h,
        density=True,
    )
if res_h["n_qcd"] > 0:
    ax.hist(
        ak.to_numpy(res_h["qcd_hh"].mass),
        bins=bins_mhh,
        histtype="step",
        linewidth=2,
        color="grey",
        linestyle="--",
        label="QCD",
        density=True,
    )
ax.axvspan(*sig_window_hh, alpha=0.05, color="green")
ax.set_xlabel("$m_{HH}$ [GeV]")
ax.set_ylabel("Density")
ax.set_title(f"{label_h} - Di-Higgs Mass (AK8)")
ax.legend(fontsize=10)
plt.tight_layout()
save_htag_fig(fig, "htag_dihiggs_mass_mhh_full.png")
plt.show()

# 4) Signal scatter
if res_h["n_signal"] > 0:
    fig = plt.figure(figsize=(9, 8))
    ax = fig.add_subplot(111)
    ax.scatter(
        ak.to_numpy(res_h["sig_lead"].mass),
        ak.to_numpy(res_h["sig_sub"].mass),
        s=2,
        alpha=0.3,
        color=color_h,
    )
    ax.axvline(125, color="green", linestyle=":", alpha=0.5)
    ax.axhline(125, color="green", linestyle=":", alpha=0.5)
    ax.set_xlabel("Leading $m_H$ [GeV]")
    ax.set_ylabel("Subleading $m_H$ [GeV]")
    ax.set_title(f"Signal - $m_{{H_1}}$ vs $m_{{H_2}}$ ({res_h['n_signal']} events)")
    ax.set_xlim(0, 300)
    ax.set_ylim(0, 300)
    plt.tight_layout()
    save_htag_fig(fig, "htag_dihiggs_mass_scatter_signal_full.png")
    plt.show()

# 5) QCD scatter
if res_h["n_qcd"] > 0:
    fig = plt.figure(figsize=(9, 8))
    ax = fig.add_subplot(111)
    ax.scatter(
        ak.to_numpy(res_h["qcd_lead"].mass),
        ak.to_numpy(res_h["qcd_sub"].mass),
        s=2,
        alpha=0.3,
        color="grey",
    )
    ax.axvline(125, color="green", linestyle=":", alpha=0.5)
    ax.axhline(125, color="green", linestyle=":", alpha=0.5)
    ax.set_xlabel("Leading $m_H$ [GeV]")
    ax.set_ylabel("Subleading $m_H$ [GeV]")
    ax.set_title(
        f"QCD Background - $m_{{H_1}}$ vs $m_{{H_2}}$ ({res_h['n_qcd']} events)"
    )
    ax.set_xlim(0, 300)
    ax.set_ylim(0, 300)
    plt.tight_layout()
    save_htag_fig(fig, "htag_dihiggs_mass_scatter_qcd_full.png")
    plt.show()

# Significance summary
print(f"\n{'='*70}")
print(
    f"{'Tagger':<30} {'S':>6} {'B (wtd)':>10} {'S/sqrt(S+B)':>12}  |  {'S (window)':>10} {'B (window)':>12} {'S/sqrt(S+B)':>12}"
)
print(
    f"{'':30} {'(all)':>6} {'(all)':>10} {'(all)':>12}  |  {str(sig_window_hh):>10} {str(sig_window_hh):>12} {'(window)':>12}"
)
print("-" * 95)
if res_h["n_signal"] > 0 and res_h["n_qcd"] > 0:
    sig_m = ak.to_numpy(res_h["sig_hh"].mass)
    bkg_m = ak.to_numpy(res_h["qcd_hh"].mass)
    sigma_to_ngen_h = res_h.get("sigma_to_ngen", {})
    sig_mh1_h = ak.to_numpy(res_h["sig_lead"].mass)
    sig_mh2_h = ak.to_numpy(res_h["sig_sub"].mass)
    bkg_mh1_h = ak.to_numpy(res_h["qcd_lead"].mass)
    bkg_mh2_h = ak.to_numpy(res_h["qcd_sub"].mass)

    result_mhh_h = compute_significance_at_luminosity(
        sig_mh1_h,
        sig_mh2_h,
        bkg_mh1_h,
        bkg_mh2_h,
        bkg_raw_weights=qcd_w_h,
        sigma_to_ngen=sigma_to_ngen_h,
        n_gen_signal=N_GEN_SIGNAL,
        luminosity_fb=LUMINOSITY_FB,
        signal_xsec_pb=SIGNAL_XSEC_PB,
        region="rectangular",
        rect_window=sig_window_hh,
    )
    S_win = result_mhh_h["S"]
    B_win = result_mhh_h["B"]
    signif_win = result_mhh_h["significance"]

    _sw_h = signal_weight(len(sig_mh1_h), LUMINOSITY_FB, SIGNAL_XSEC_PB, N_GEN_SIGNAL)
    _bw_h = scale_qcd_weights_raw(qcd_w_h, sigma_to_ngen_h, LUMINOSITY_FB)
    S_all = float(np.sum(_sw_h))
    B_all = float(np.sum(_bw_h))
    signif_all = S_all / np.sqrt(S_all + B_all) if (S_all + B_all) > 0 else 0.0

    print(
        f"{label_h:<30} {S_all:>6.0f} {B_all:>10.1e} {signif_all:>12.4f}  |  {S_win:>10.0f} {B_win:>12.1e} {signif_win:>12.4f}"
    )
    print(
        f"\n  Note: S and B are luminosity-scaled expected event counts at {LUMINOSITY_FB:.0f} fb⁻¹."
    )
    print(f"  Unweighted QCD events: {res_h['n_qcd']}")
else:
    print(f"{label_h:<30}  Insufficient events for significance")
print("=" * 95)
print(f"\nAll full-size H-tag di-Higgs plots saved to: {plot_dir_htag}")

In [ ]:
# ============================================================
# 6) r_HH signal region (AK8): plotting + significance
# ============================================================
# Uses existing variables from the previous cell:
#   res_h, qcd_w_h, label_h, sig_window_hh, plot_dir_htag, save_htag_fig

from evaluation.dihiggs import compute_significance_at_luminosity
from evaluation.luminosity import signal_weight, scale_qcd_weights_raw

# --- Config ---
MH_CENTER = 125.0
# If R_HH_CUT already exists in your notebook, use it; otherwise default to 30 GeV
try:
    r_hh_cut = float(R_HH_CUT)
except NameError:
    r_hh_cut = 30.0

if res_h["n_signal"] > 0 and res_h["n_qcd"] > 0:
    # Arrays
    sig_lead_m = ak.to_numpy(res_h["sig_lead"].mass)
    sig_sub_m = ak.to_numpy(res_h["sig_sub"].mass)
    sig_hh_m = ak.to_numpy(res_h["sig_hh"].mass)

    bkg_lead_m = ak.to_numpy(res_h["qcd_lead"].mass)
    bkg_sub_m = ak.to_numpy(res_h["qcd_sub"].mass)
    bkg_hh_m = ak.to_numpy(res_h["qcd_hh"].mass)

    # r_HH definition in (mH1, mH2) plane
    sig_rhh = np.sqrt((sig_lead_m - MH_CENTER) ** 2 + (sig_sub_m - MH_CENTER) ** 2)
    bkg_rhh = np.sqrt((bkg_lead_m - MH_CENTER) ** 2 + (bkg_sub_m - MH_CENTER) ** 2)

    # Masks for regions
    sig_mask_rhh = sig_rhh < r_hh_cut
    bkg_mask_rhh = bkg_rhh < r_hh_cut

    sig_mask_mhh = (sig_hh_m >= sig_window_hh[0]) & (sig_hh_m <= sig_window_hh[1])
    bkg_mask_mhh = (bkg_hh_m >= sig_window_hh[0]) & (bkg_hh_m <= sig_window_hh[1])

    # r_HH only
    sigma_to_ngen_h = res_h.get("sigma_to_ngen", {})
    sig_mh1_h = ak.to_numpy(res_h["sig_lead"].mass)
    sig_mh2_h = ak.to_numpy(res_h["sig_sub"].mass)
    bkg_mh1_h = ak.to_numpy(res_h["qcd_lead"].mass)
    bkg_mh2_h = ak.to_numpy(res_h["qcd_sub"].mass)

    result_rhh_h = compute_significance_at_luminosity(
        sig_mh1_h,
        sig_mh2_h,
        bkg_mh1_h,
        bkg_mh2_h,
        bkg_raw_weights=qcd_w_h,
        sigma_to_ngen=sigma_to_ngen_h,
        n_gen_signal=N_GEN_SIGNAL,
        luminosity_fb=LUMINOSITY_FB,
        signal_xsec_pb=SIGNAL_XSEC_PB,
        region="circular",
        r_hh_cut=R_HH_CUT_HTAG,
    )
    S_rhh = result_rhh_h["S"]
    B_rhh = result_rhh_h["B"]
    signif_rhh = result_rhh_h["significance"]

    # mHH window only (already in your table, repeated here for completeness)
    result_mhh_h2 = compute_significance_at_luminosity(
        sig_mh1_h,
        sig_mh2_h,
        bkg_mh1_h,
        bkg_mh2_h,
        bkg_raw_weights=qcd_w_h,
        sigma_to_ngen=sigma_to_ngen_h,
        n_gen_signal=N_GEN_SIGNAL,
        luminosity_fb=LUMINOSITY_FB,
        signal_xsec_pb=SIGNAL_XSEC_PB,
        region="rectangular",
        rect_window=sig_window_hh_htag,
    )
    S_win = result_mhh_h2["S"]
    B_win = result_mhh_h2["B"]
    signif_win = result_mhh_h2["significance"]

    # Combined region: r_HH + mHH window
    sig_mask_comb = sig_mask_rhh & sig_mask_mhh
    bkg_mask_comb = bkg_mask_rhh & bkg_mask_mhh
    # Combined R_HH + mHH: use the intersected masks with pre-scaled weights
    _sw_h = signal_weight(len(sig_mh1_h), LUMINOSITY_FB, SIGNAL_XSEC_PB, N_GEN_SIGNAL)
    _bw_h = scale_qcd_weights_raw(qcd_w_h, sigma_to_ngen_h, LUMINOSITY_FB)
    S_comb = float(np.sum(_sw_h[sig_mask_comb]))
    B_comb = float(np.sum(_bw_h[bkg_mask_comb]))
    signif_comb = S_comb / np.sqrt(S_comb + B_comb) if (S_comb + B_comb) > 0 else 0.0

    # --- Plot A: signal/background in mH1-mH2 with r_HH circle ---
    fig = plt.figure(figsize=(9, 8))
    ax = fig.add_subplot(111)
    ax.scatter(bkg_lead_m, bkg_sub_m, s=2, alpha=0.20, color="grey", label="QCD")
    ax.scatter(sig_lead_m, sig_sub_m, s=3, alpha=0.35, color="teal", label="Signal")
    ax.axvline(MH_CENTER, color="green", linestyle=":", alpha=0.6)
    ax.axhline(MH_CENTER, color="green", linestyle=":", alpha=0.6)

    circle = plt.Circle(
        (MH_CENTER, MH_CENTER),
        r_hh_cut,
        fill=False,
        color="crimson",
        lw=2,
        label=f"r_HH < {r_hh_cut:g}",
    )
    ax.add_patch(circle)

    ax.set_xlabel("Leading $m_H$ [GeV]")
    ax.set_ylabel("Subleading $m_H$ [GeV]")
    ax.set_title(f"{label_h} - AK8 $m_{{H_1}}$ vs $m_{{H_2}}$ with $r_{{HH}}$ SR")
    ax.set_xlim(0, 300)
    ax.set_ylim(0, 300)
    ax.legend(fontsize=10, loc="upper right")
    plt.tight_layout()
    save_htag_fig(fig, "htag_dihiggs_rhh_signal_region_scatter.png")
    plt.show()

    # --- Plot B: r_HH distributions ---
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111)
    bins_rhh = np.linspace(0, 200, 81)

    ax.hist(
        sig_rhh,
        bins=bins_rhh,
        histtype="stepfilled",
        alpha=0.25,
        color="teal",
        density=True,
        label="Signal",
    )
    ax.hist(
        sig_rhh, bins=bins_rhh, histtype="step", linewidth=2, color="teal", density=True
    )
    ax.hist(
        bkg_rhh,
        bins=bins_rhh,
        weights=qcd_w_h,
        histtype="step",
        linewidth=2,
        color="grey",
        linestyle="--",
        density=True,
        label="QCD (weighted)",
    )
    ax.axvline(
        r_hh_cut,
        color="crimson",
        linestyle="-",
        linewidth=2,
        label=f"$r_{{HH}}$ cut = {r_hh_cut:g}",
    )

    ax.set_xlabel(r"$r_{HH}$ [GeV]")
    ax.set_ylabel("Density")
    ax.set_title(f"{label_h} - $r_{{HH}}$ distribution (AK8)")
    ax.legend(fontsize=10)
    plt.tight_layout()
    save_htag_fig(fig, "htag_dihiggs_rhh_distribution.png")
    plt.show()

    # --- Print significance summary ---
    print("\n" + "=" * 105)
    print(f"{'Region':<30} {'S':>10} {'B (wtd)':>14} {'S/sqrt(S+B)':>16}")
    print("-" * 105)
    print(f"{'r_HH only':<30} {S_rhh:>10.0f} {B_rhh:>14.3e} {signif_rhh:>16.4f}")
    print(f"{'mHH window only':<30} {S_win:>10.0f} {B_win:>14.3e} {signif_win:>16.4f}")
    print(
        f"{'r_HH + mHH window':<30} {S_comb:>10.0f} {B_comb:>14.3e} {signif_comb:>16.4f}"
    )
    print("-" * 105)
    print(
        f"Settings: mH center={MH_CENTER:.1f} GeV, r_HH cut={r_hh_cut:g} GeV, mHH window={sig_window_hh}"
    )
    print("Note: B values are weighted QCD sums.")
    print("=" * 105)

else:
    print("Insufficient signal/background events for r_HH significance calculation.")

# QCD Weight Usage Log (Cell-by-Cell)

Scope: only QCD weight usage is logged here. Kinematic weights are intentionally excluded.

| Cell # | QCD weights used? | What they are used for |
|---|---|---|
| 1 | No | - |
| 2 | No | Markdown cell |
| 3 | No | - |
| 4 | Yes | Collects `all_qcd_weights_val` from validation loader (`qcd_weights_batch`) and builds `sigma_to_ngen` map for later luminosity scaling checks |
| 5 | No | - |
| 6 | No | - |
| 7 | No | - |
| 8 | No | - |
| 9 | Yes | Uses post-cut `all_qcd_weights_val` as `sample_weight` for trained ParT ROC; builds reference-tagger QCD background weights from raw bin cross-section and passes as `bkg_weights` to `roc_from_scores` |
| 10 | Yes | Uses post-cut QCD weights for PR curve, PR-AUC, `S/sqrt(B)` vs threshold, efficiency tables, and per-bin PR/SR metrics |
| 11 | No | Consumes prior outputs only |
| 12 | No | Markdown cell |
| 13 | No | - |
| 14 | Yes | During chunked QCD loading for di-Higgs: assigns each selected QCD event raw bin cross-section weight, concatenates into `qcd_weights`, builds `sigma_to_ngen`, stores in result dict |
| 15 | No | - |
| 16 | No | - |
| 17 | No | Markdown cell |
| 18 | No | - |
| 19 | No | - |
| 20 | No | - |
| 21 | No | - |
| 22 | No | - |
| 23 | No | - |
| 24 | No | - |
| 25 | No | - |
| 26 | No | Markdown cell |
| 27 | No | - |
| 28 | No | - |
| 29 | Yes | Uses `qcd_weights` in weighted QCD histograms and in luminosity significance (`compute_significance_at_luminosity` with raw weights + `sigma_to_ngen`; inclusive `B` via `scale_qcd_weights_raw`) |
| 30 | Yes | Uses `qcd_weights` for `R_HH` weighted histograms and significance (`compute_significance_at_luminosity` + `scale_qcd_weights_raw`) |
| 31 | No | - |
| 32 | No | - |
| 33 | Yes | Uses `all_qcd_weights_val` as `sample_weight` for permutation and ablation AUC-based feature-importance estimates |
| 34 | Yes | Uses `all_qcd_weights_val` for weighted AUC diagnostics (k-NN estimate, overall AUC, AUC vs pT bins) |
| 35 | No | - |
| 36 | No | - |
| 37 | No | Markdown cell |
| 38 | Yes | H-tag ROC/AUC section: uses `all_qcd_weights_val` as `htag_weights` for ROC and per-(pT, |eta|) AUC heatmap |
| 39 | No | Markdown cell |
| 40 | Yes | AK8 H-tag di-Higgs QCD loading: assigns raw per-bin cross-section weights to selected QCD events (`qcd_weights_htag`), builds `sigma_to_ngen_htag`, stores both |
| 41 | Yes | Uses `qcd_w_h` for significance (`compute_significance_at_luminosity`, `scale_qcd_weights_raw`); most shape overlays are unweighted density plots |
| 42 | Yes | Uses `qcd_w_h` for `R_HH` and `mHH` significance and combined-region weighted `B`; also weighted QCD `R_HH` histogram |

Notes:
- Raw QCD bin cross-sections are preserved in reconstruction result dicts and converted to luminosity-scaled expected counts only at significance/evaluation time.
- ROC/PR cells use the QCD weights directly as metric sample/background weights.